# 🌱 GreenLedger — Campus Resource Optimisation (IC26-05-01) · v12
**Team Shenzein** — Anup Batakurki · Het Balchandani · Kamaal · Dayananda Sagar University, Bengaluru
### Real reference data (Concordia University) + a clearly-labelled simulated Indian campus

**In one line:** GreenLedger reads every meter, waste bill and device on campus, **recommends** what to change, **switches** room fan/lights/AC
from a live camera, and **proves** with a ghost twin + placebo test whether a change really saved anything — counting only the low end.

**Two kinds of data — never mixed:**
| | What | Why |
|---|---|---|
| **Real** | Concordia University open data (CC BY 4.0): 6 meters, waste invoices | the trust test and the ground-truth proof (COVID closure) |
| **Campus model** | 16 departments on a Bengaluru campus (incl. Medical, Law, MBA, Pharmacy, Sciences, Architecture, Commerce, Sports) · weather, academic calendar, holidays · ~2,800-device e-waste register | department-wise analysis, and a test where the **true** saving is known |

| Page | What it shows |
|---|---|
| Dashboard | **Total energy saved** (verified) · **money saved per month** · savings by department · energy, water, waste, e-waste value |
| Departments | pick a department **on the campus photo** · department-wise energy savings · energy, cost, water, waste, CO₂, money saved, one suggestion |
| Campus Digital Twin | **interactive DSU campus dashboard**: switch between energy use / energy saved / money saved / water / waste and every block re-colours |
| Judge's Guide | the brief, ticked, with “Show me” buttons · how we know it works · pilot plan · targets |
| Support & Info | team · how to use · words explained · FAQ · troubleshooting · settings · safety & privacy · about the data |
| Live vs Ghost Twin | every action: verdict, saving range, placebo test, **money saved (₹) running total**, true vs estimated |

**Live twin vs ghost twin, in one line each:** the *live twin* is what really happened (the meter reading); the *ghost twin* is what the same
building would have used without the action (a model fed the real weather and calendar). **Saving = ghost − live**, counted only when it is bigger
than normal noise, and only its low end. Every page in the app starts with a short "How to read this page" box. Money is shown **per month, in ₹ thousands (₹K)**.
| Live Room Control | camera → people → fan / lights / AC switched live |
| E-waste | sell second-hand (and when), repair, reusable parts, recycler |
| Carbon · Recommendations · Reports | CO₂ by department, suggestions, exports |

**How to run (Google Colab):**
1. Click the 📁 folder icon and upload `DATA_OUTPUT.csv`, `WASTE_INVOICES.csv`, `WASTE_TYPE.csv`, `BUILDING_LIST.csv`
   (optional: your `ewaste_register.csv`). If you skip this, cell 2 tries to download them from Concordia's open-data site.
2. Run the cells **one at a time, top to bottom** (Shift + Enter). Each prints a short, plain result.
3. The last cell prints two links — open the **Colab link** (or the **public link** for phones). Allow camera access on the Live Room page.

Total run time ≈ 3 minutes.

## Cell 1 · Install
Installs the libraries: web app (FastAPI + Uvicorn), charts (Plotly), holiday calendar, statistics, gradient boosting (LightGBM),
people detection (Ultralytics YOLOv8) and thermal comfort (pythermalcomfort, ASHRAE 55).

In [ ]:
!pip -q install fastapi uvicorn plotly holidays scipy lightgbm ultralytics pythermalcomfort


## Cell 2 · Find the data files
Finds your uploaded CSV files (even if Colab renamed them) — or downloads them from Concordia's official open-data site.

🎤 **Tell the judges:** *"Everything comes from Concordia University's public open data — real meters and real waste invoices. We didn't simulate or fill in a single number."*

In [ ]:
import os, re, csv, glob, io, json, zipfile, datetime, warnings, urllib.request, base64
import numpy as np
import pandas as pd
from scipy import stats
import holidays
import lightgbm as lgb
import plotly.graph_objects as go
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

SOURCE_URL = "https://opendata.concordia.ca/datasets/Facilities/"
FILES = ["DATA_OUTPUT.csv", "WASTE_INVOICES.csv", "WASTE_TYPE.csv", "BUILDING_LIST.csv"]
OPTIONAL = ["POINT_LIST.csv"]          # Concordia's official sensor list, used if available
OUT_DIR = os.environ.get("GL_OUT_DIR", "/content/greenledger_out" if os.path.isdir("/content") else "./greenledger_out")
DATA_DIR = os.environ.get("GL_DATA_DIR", "/content" if os.path.isdir("/content") else ".")
os.makedirs(OUT_DIR, exist_ok=True)


def find(name):
    """Find a file even if it was renamed on upload (e.g. 'DATA_OUTPUT (1).csv' or 'abc-DATA_OUTPUT.csv')."""
    stem = name[:-4].upper()
    for root in [DATA_DIR, ".", "/content", "/content/drive/MyDrive"]:
        if not root or not os.path.isdir(root):
            continue
        cands = glob.glob(os.path.join(root, "*.csv")) + glob.glob(os.path.join(root, "*", "*.csv"))
        hits = []
        for p in cands:
            s_ = re.sub(r"\(\d+\)$", "", os.path.splitext(os.path.basename(p))[0].upper().replace(" ", ""))
            if (s_ == stem or s_.endswith("-" + stem)) and os.path.getsize(p) > 0:
                hits.append(p)
        if hits:
            return sorted(hits, key=len)[0]
    return None


PATHS = {f: find(f) for f in FILES + OPTIONAL}
missing = [f for f in FILES if not PATHS[f]]
for f in missing + [o for o in OPTIONAL if not PATHS[o]]:   # try Concordia's official open-data site
    if True:
        try:
            dst = os.path.join(OUT_DIR, f)
            urllib.request.urlretrieve(SOURCE_URL + f, dst)
            PATHS[f] = dst
            print(f"Downloaded {f} from {SOURCE_URL}")
        except Exception as e:
            print(f"Could not download {f}: {e}")
missing = [f for f in FILES if not PATHS[f]]
assert not missing, (f"Missing {missing}. In Colab: click the folder icon on the left, upload the 4 CSV files, "
                     f"then run this cell again.")
for f, p in PATHS.items():
    print(f"{f:20s} → {p or 'not found (optional)'}")
print("\nData: Concordia University open data (Facilities), licence CC BY 4.0 — "
      "https://www.concordia.ca/web/open-data.html")


## Cell 3 · Read the meters
Concordia's building-automation export is many reports stacked in one file. This cell stitches them into one hourly table and names each sensor from Concordia's own API documentation.

| Meter | What it measures |
|---|---|
| ⚡ Loyola campus | electricity demand of the whole Loyola campus (kW) |
| ⚡ EV building | electricity demand of the Engineering & Visual Arts building |
| ⚡ EV + GM + MB + TD | electricity of four downtown buildings together |
| 🔥 EV + GM + MB / GM | natural gas (running counters that reset each month) |
| 💧 Municipal water | the city water supply meter (running counter) |
| 🌡️ Weather | outdoor temperature from the EV rooftop station |

In [ ]:
# ---- What each sensor is: Concordia's own API documentation (opendataConcordiaU/documentation,
#      facilities/consumption and facilities/environmental) ----
METER_INFO = {
    # point:  (system name, resource, plain name, buildings it covers, reading type, unit)
    "Point_12": ("DEMANDE.ELEC.MOYEN60S.BS", "electricity", "Loyola campus",            "LOY campus", "power", "kW"),
    "Point_7":  ("TOTKWEV.IC",               "electricity", "EV building",              "EV",          "power", "kW"),
    "Point_6":  ("TOTKWEV-MB.IC",            "electricity", "EV + GM + MB + TD buildings", "EV,GM,MB,TD", "power", "kW"),
    "Point_8":  ("MDI.001-650.IC",           "gas",         "EV + GM + MB buildings",   "EV,GM,MB",    "counter", "m³ (as documented)"),
    "Point_9":  ("MDICOR.GM",                "gas",         "GM building",              "GM",          "counter", "m³ (as documented)"),
    "Point_10": ("MDE.EFDP.IC",              "water",       "Municipal water supply",   "",            "counter", "m³"),
}
# campus totals without double counting: EV is inside EV+GM+MB+TD, GM is inside EV+GM+MB
CAMPUS_TOTAL = {"electricity": ["Point_12", "Point_6"], "gas": ["Point_8"], "water": ["Point_10"]}
WEATHER_POINT = "Point_3"     # MTX1-017.IC — outdoor temperature, EV-building weather station (°C)
RESOURCE = {
    "electricity": {"label": "Electricity", "unit": "kWh", "color": "#10B981", "icon": "⚡"},
    "gas":         {"label": "Natural gas", "unit": "m³",  "color": "#F97316", "icon": "🔥"},
    "water":       {"label": "Water",       "unit": "m³",  "color": "#3B82F6", "icon": "💧"},
    "waste":       {"label": "Waste",       "unit": "t",   "color": "#F59E0B", "icon": "🗑️"},
}

# ---- read Concordia's building-automation export (it is many reports stacked in one file) ----
rows, header = [], None
with open(PATHS["DATA_OUTPUT.csv"], errors="replace") as fh:
    for r in csv.reader(fh):
        if r and r[0] == "<>Date":
            header = ["date", "time"] + r[2:]
            continue
        if header and len(r) >= len(header) and "/" in r[0]:
            rows.append(r[:len(header)])
raw = pd.DataFrame(rows, columns=header)
stamp = pd.to_datetime(raw["date"] + " " + raw["time"].str.strip(), format="%m/%d/%Y %H:%M:%S")
raw = raw.drop(columns=["date", "time"]).apply(lambda s: pd.to_numeric(s.str.strip(), errors="coerce"))  # "No Data" → missing
raw.index = stamp.dt.floor("h").values
raw = raw[~raw.index.duplicated(keep="last")].sort_index()           # overlapping reports: keep the latest
HOURLY = raw.reindex(pd.date_range(raw.index.min(), raw.index.max(), freq="h"))
DAYS = pd.date_range(HOURLY.index.min().normalize(), HOURLY.index.max().normalize(), freq="D")

# optional: Concordia's official point list (adds building / description if the file is available)
if PATHS.get("POINT_LIST.csv"):
    try:
        _pl = pd.read_csv(PATHS["POINT_LIST.csv"])
        print("Official point list loaded:", list(_pl.columns))
    except Exception as e:
        print("POINT_LIST.csv could not be read:", e)

METERS = pd.DataFrame([{"point": p, "system name": v[0], "resource": v[1], "meter": v[2], "buildings": v[3] or "not stated",
                        "reading": "average power (kW)" if v[4] == "power" else "running total (resets each month/day)",
                        "first reading": HOURLY[p].first_valid_index(), "last reading": HOURLY[p].last_valid_index(),
                        "hours with data": f"{HOURLY[p].notna().mean():.0%}"} for p, v in METER_INFO.items()])
print(f"Hourly readings: {HOURLY.index.min():%d %b %Y} → {HOURLY.index.max():%d %b %Y}  ({len(HOURLY):,} hours)")
print(METERS.drop(columns=["point"]).to_string(index=False))


## Cell 4 · Clean the data (3 simple rules)
1. **Incomplete day** — fewer than 22 of 24 hours recorded → not used.
2. **Impossible jump** — one hour can't use more than the meter ever used in a whole day → that hour is removed.
3. **Not measuring** — the counter rose by the same amount every hour all day → it was estimating, not measuring.

🎤 *"We found real faults — the GM gas meter jumped 306,266 units in one hour; the water meter flat-lined for 5 weeks. We show every excluded day and why. We never fill gaps."*

In [ ]:
# ---- Turn hourly readings into one number per meter per day, with three plain quality rules ----
COMPLETE = 0.90      # a day counts only if ≥90% of its hours have a reading (22 of 24)
FAULTS = []          # every excluded day, with the reason — shown in the app


def daily_from_power(p):
    s = HOURLY[p]
    n = s.resample("D").count()
    return (s.resample("D").mean() * 24).where(n >= COMPLETE * 24).reindex(DAYS)   # average kW × 24 h = kWh


def hourly_use_from_counter(p):
    s = HOURLY[p]
    use = s.diff()
    reset = use < 0
    use[reset] = s[reset]                       # counter restarted at zero → the reading is the use since restart
    use = use.where(s.notna() & s.shift(1).notna())
    return use, reset


def daily_from_counter(p):
    use, reset = hourly_use_from_counter(p)
    # Rule A — impossible jump: one hour cannot use more than the most this meter ever used in a whole day
    while True:
        tot = use.resample("D").sum(min_count=1).dropna().sort_values(ascending=False)
        limit = np.where(use.index.normalize() == tot.index[0], tot.iloc[1], tot.iloc[0])
        bad = use > limit
        if not bad.any():
            break
        for t in use[bad].index:
            FAULTS.append({"point": p, "day": t.normalize(), "rule": "impossible jump",
                           "detail": f"{use[t]:,.0f} units in one hour at {t:%H:%M} — more than any whole day"})
        use[bad] = np.nan
    # Rule B — not measuring: every hourly step identical (±1 unit) all day = a fixed estimate, not a reading
    u = use[~reset]
    g = u.groupby(u.index.normalize())
    stuck = ((g.max() - g.min()) <= 1) & (g.count() >= COMPLETE * 24 - 1)
    for d in stuck[stuck].index:
        FAULTS.append({"point": p, "day": d, "rule": "not measuring",
                       "detail": f"same amount every hour ({g.median()[d]:,.0f}/h) — meter was not measuring"})
    n = use.resample("D").count()
    day = (use.resample("D").mean() * 24).where(n >= COMPLETE * 24).reindex(DAYS)
    day[day.index.isin(stuck[stuck].index)] = np.nan
    return day


DAILY = pd.DataFrame({p: (daily_from_power(p) if v[4] == "power" else daily_from_counter(p))
                      for p, v in METER_INFO.items()})
for p in DAILY:
    miss = DAILY[p].isna() & HOURLY[p].resample("D").count().reindex(DAYS).gt(0)
    for d in DAILY.index[miss]:
        if not any(f["point"] == p and f["day"] == d for f in FAULTS):
            FAULTS.append({"point": p, "day": d, "rule": "incomplete day", "detail": "fewer than 22 of 24 hours recorded"})
FAULTS = pd.DataFrame(FAULTS, columns=["point", "day", "rule", "detail"]).sort_values(["point", "day"])
FAULTS["meter"] = FAULTS.point.map(lambda p: f"{RESOURCE[METER_INFO[p][1]]['label']} · {METER_INFO[p][2]}")


def fault_periods(F):
    """Consecutive days with the same meter and rule → one line (easier to read)."""
    out = []
    for (m, rule), g in F.sort_values("day").groupby(["meter", "rule"]):
        days = list(g.day); det = list(g.detail); i = 0
        while i < len(days):
            j = i
            while j + 1 < len(days) and days[j + 1] - days[j] == pd.Timedelta(days=1):
                j += 1
            out.append({"meter": m, "from": days[i].date(), "to": days[j].date(), "days": j - i + 1, "rule": rule,
                        "detail": det[i] if i == j else det[i].split(" — ")[-1]})
            i = j + 1
    return pd.DataFrame(out).sort_values("from")

# daily outdoor temperature (EV weather station) + Quebec public holidays (official calendar via `holidays`)
TEMP = HOURLY[WEATHER_POINT].resample("D").mean().where(HOURLY[WEATHER_POINT].resample("D").count() >= COMPLETE * 24).reindex(DAYS)
QC_HOLIDAYS = holidays.Canada(subdiv="QC", years=sorted(set(DAYS.year)))
OFF_DAY = pd.Series([(d.dayofweek >= 5) or (d.date() in QC_HOLIDAYS) for d in DAYS], index=DAYS)

print("Good days per meter (out of", len(DAYS), "):")
for p in DAILY:
    print(f"  {RESOURCE[METER_INFO[p][1]]['icon']} {METER_INFO[p][2]:30s} {DAILY[p].notna().sum():4d} good days")
print("\nExcluded days by reason:")
print(FAULTS.groupby(["meter", "rule"]).size().rename("days").to_string())
print("\nThe problems (not counting incomplete days):")
print(fault_periods(FAULTS[FAULTS.rule != "incomplete day"]).to_string(index=False))


## Cell 5 · Learn "normal" — the ghost twin's brain
Two models are trained on one full year (every season once), using the day's **temperature**, **weekday/holiday** and **season**:
* the **ASHRAE change-point model** (explainable: "+1,233 kWh per °C colder"), and
* **gradient boosting (LightGBM)** — machine learning.

Each baseline month is hidden in turn and predicted by both; **whichever misses less becomes that meter's ghost twin.**
The change-point model is always kept to explain the building and to grade accuracy (ASHRAE Guideline 14).

🎤 *"We don't pick AI or physics by taste — each meter gets the model that predicts months it has never seen better."*

In [ ]:
# ---- "Expected use": how much each meter normally uses for a given temperature and day type ----
# ASHRAE change-point model (ASHRAE Guideline 14 / Inverse Modeling Toolkit, RP-1050):
#   daily use = base + (off-day change) + heating slope × max(0, Th − temp) + cooling slope × max(0, temp − Tc)
# The balance temperatures Th, Tc are found from the data (every whole °C observed is tried); the simplest
# model that fits is chosen by BIC. Slopes are kept ≥ 0 (colder can't reduce heating; hotter can't reduce cooling).
_tm = TEMP.notna().groupby(TEMP.index.to_period("M")).mean()
BASE_START = _tm[_tm >= COMPLETE].index[0].to_timestamp()               # first month with weather data
BASE_END = BASE_START + pd.DateOffset(years=1) - pd.Timedelta(days=1)    # one full year = every season once
MON_START = BASE_END + pd.Timedelta(days=1)
print(f"Baseline year: {BASE_START:%d %b %Y} → {BASE_END:%d %b %Y}   ·   Monitoring: {MON_START:%d %b %Y} → {DAYS[-1]:%d %b %Y}")


def _X(idx, th, tc):
    t = TEMP.reindex(idx).values
    cols = [np.ones(len(idx)), OFF_DAY.reindex(idx).values.astype(float)]
    if th is not None: cols.append(np.maximum(0, th - t))
    if tc is not None: cols.append(np.maximum(0, t - tc))
    return np.column_stack(cols)


def _ml_features(idx):
    idx = pd.DatetimeIndex(idx)
    return pd.DataFrame({"temp": TEMP.reindex(idx).values, "off": OFF_DAY.reindex(idx).values.astype(int),
                         "dow": idx.dayofweek, "doy_sin": np.sin(2 * np.pi * idx.dayofyear / 365.25),
                         "doy_cos": np.cos(2 * np.pi * idx.dayofyear / 365.25)}, index=idx)


def fit_model(y, kind="cp"):
    """kind='cp': ASHRAE change-point model · kind='ml': gradient boosting (LightGBM, library defaults)."""
    if kind == "ml":
        y = y.dropna(); y = y[TEMP.reindex(y.index).notna()]
        mdl = lgb.LGBMRegressor(random_state=0, verbosity=-1).fit(_ml_features(y.index), y.values)
        return {"kind": "ml", "mdl": mdl, "idx": y.index, "n": len(y)}
    y = y.dropna()
    y = y[TEMP.reindex(y.index).notna()]
    t = TEMP.reindex(y.index).values
    o = OFF_DAY.reindex(y.index).values.astype(float)
    v = y.values
    n = len(v)
    grid = [None] + sorted(set(np.round(t).astype(int)))
    best = None
    for th in grid:
        h = None if th is None else np.maximum(0, th - t)
        for tc in grid:
            if th is not None and tc is not None and tc < th:
                continue
            cols = [np.ones(n), o] + ([h] if h is not None else []) + ([np.maximum(0, t - tc)] if tc is not None else [])
            X = np.column_stack(cols)
            b = np.linalg.lstsq(X, v, rcond=None)[0]
            if len(b) > 2 and b[2:].min() < 0:
                continue
            sse = float(((X @ b - v) ** 2).sum())
            k = X.shape[1] + (th is not None) + (tc is not None)
            bic = n * np.log(sse / n) + k * np.log(n)
            if best is None or bic < best["bic"]:
                best = {"kind": "cp", "bic": bic, "th": th, "tc": tc, "b": b, "k": k, "n": n, "idx": y.index}
    return best


def expected(m, idx):
    idx = pd.DatetimeIndex(idx)
    if m["kind"] == "ml":
        return pd.Series(m["mdl"].predict(_ml_features(idx)), index=idx).clip(lower=0)
    return pd.Series(_X(idx, m["th"], m["tc"]) @ m["b"], index=idx).clip(lower=0)


def accuracy(y, m):
    yb = y.reindex(m["idx"])
    e = yb - expected(m, yb.index)
    cv = float(np.sqrt((e ** 2).sum() / (len(e) - m["k"])) / yb.mean() * 100)
    nmbe = float(e.sum() / (len(e) - m["k"]) / yb.mean() * 100)
    rho = float(e.autocorr(1))
    return cv, nmbe, rho


def describe_model(p, m):
    res, unit = METER_INFO[p][1], RESOURCE[METER_INFO[p][1]]["unit"]
    b = list(m["b"]); base, offd = b[0], b[1]; i = 2
    s = [f"On a normal weekday in mild weather it uses about {base:,.0f} {unit}/day; on weekends and holidays "
         f"{abs(offd):,.0f} {'less' if offd < 0 else 'more'}."]
    if m["th"] is not None:
        s.append(f"For every 1 °C below {m['th']} °C it uses {b[i]:,.0f} {unit} more (heating)."); i += 1
    if m["tc"] is not None:
        s.append(f"For every 1 °C above {m['tc']} °C it uses {b[i]:,.0f} {unit} more (cooling).")
    return " ".join(s)


# ---- Which model predicts better? Test both on months they never saw (leave-one-month-out) ----
# For every meter, each baseline month is hidden in turn and predicted by (a) the change-point model and
# (b) gradient boosting (LightGBM, library defaults, same inputs + season). The model with the smaller
# typical miss on hidden months is used for predictions (alerts, verification). The change-point model is
# always kept too, because its slopes EXPLAIN the building (used for the accuracy grade and recommendations).
BASE_MONTHS = list(pd.period_range(BASE_START, BASE_END, freq="M"))
MODEL, CP_MODEL, MODEL_KIND, EXPECTED, OOS, MISS, ACC, MODEL_CHECK = {}, {}, {}, {}, {}, {}, [], []
for p in DAILY:
    yb = DAILY[p].loc[BASE_START:BASE_END].where(TEMP.notna())
    oos = {"cp": [], "ml": []}; miss = {"cp": {}, "ml": {}}
    for mo in BASE_MONTHS:
        a, b = mo.start_time, mo.end_time.normalize()
        hold = yb.loc[a:b].dropna()
        train = yb.copy(); train.loc[a:b] = np.nan
        days = DAYS[(DAYS >= a) & (DAYS <= b) & TEMP.notna().values]
        for kind in ("cp", "ml"):
            m_ = fit_model(train, kind)
            oos[kind].append(expected(m_, days))
            if len(hold) >= COMPLETE * len(pd.date_range(a, b)):
                e = expected(m_, hold.index).sum()
                miss[kind][a] = float((e - hold.sum()) / e * 100)
    rms = {k: float(np.sqrt(np.mean(np.square(list(miss[k].values()))))) for k in miss}
    MODEL_KIND[p] = "ml" if rms["ml"] < rms["cp"] else "cp"
    OOS[p] = pd.concat(oos[MODEL_KIND[p]])
    MISS[p] = miss[MODEL_KIND[p]]
    CP_MODEL[p] = fit_model(yb, "cp")
    MODEL[p] = CP_MODEL[p] if MODEL_KIND[p] == "cp" else fit_model(yb, "ml")
    EXPECTED[p] = expected(MODEL[p], DAYS[TEMP.notna().values])
    MODEL_CHECK.append({"point": p, "Meter": f"{RESOURCE[METER_INFO[p][1]]['icon']} {METER_INFO[p][2]}",
                        "Hidden months": len(miss["cp"]), "Change-point typical miss %": round(rms["cp"], 1),
                        "Gradient boosting typical miss %": round(rms["ml"], 1),
                        "Used for predictions": "Gradient boosting (ML)" if MODEL_KIND[p] == "ml" else "Change-point"})
    m = CP_MODEL[p]
    cv, nmbe, rho = accuracy(DAILY[p], m)
    grade = "Good" if cv <= 15 and abs(nmbe) <= 5 else ("OK" if cv <= 30 and abs(nmbe) <= 10 else "Not reliable")
    ACC.append({"point": p, "Meter": f"{RESOURCE[METER_INFO[p][1]]['icon']} {RESOURCE[METER_INFO[p][1]]['label']} · {METER_INFO[p][2]}",
                "Baseline days": m["n"], "Typical daily error %": round(cv, 1), "Bias %": round(nmbe, 2),
                "Accuracy": grade, "Predictions by": MODEL_CHECK[-1]["Used for predictions"],
                "What the model learned": describe_model(p, m)})
ACC = pd.DataFrame(ACC)
MODEL_CHECK = pd.DataFrame(MODEL_CHECK)
RELIABLE = set(ACC[ACC.Accuracy != "Not reliable"].point)
print("Accuracy grade uses ASHRAE Guideline 14 limits on the change-point fit: Good = CV ≤ 15% & bias ≤ ±5% "
      "(monthly-data limit); OK = CV ≤ 30% & bias ≤ ±10% (hourly-data limit); otherwise Not reliable (no savings claimed).\n")
for _, r in ACC.iterrows():
    print(f"{r.Meter}\n   accuracy: {r.Accuracy} (typical daily error {r['Typical daily error %']}%) · predictions by: "
          f"{r['Predictions by']}\n   {r['What the model learned']}")
print("\nWhich model predicts hidden months better (typical monthly miss, lower is better):")
print(MODEL_CHECK.drop(columns="point").to_string(index=False))


## Cell 6 · Alerts and discoveries
After the baseline year, a meter is flagged when it uses **more** than its ghost twin by more than it ever did in the whole baseline year (each meter sets its own bar). Sustained **drops** are *discoveries* — good news, not counted until someone explains them.

In [ ]:
# ---- Unusual-change alerts ----
# Each meter sets its own bar from last year: we re-predict every baseline month with a model that did NOT
# see that month (leave-one-month-out), so the baseline errors are honest "surprises".
# ALERT     = after the baseline, a day (or 7-day stretch) used MORE than expected by more than the
#             biggest surprise of the whole baseline year.
# DISCOVERY = a 7-day stretch used LESS than expected by more than any 7 days of the baseline year
#             (good news nobody has explained yet — not counted as a saving until an action is linked).
# OOS (each baseline month predicted by a model that never saw it) comes from cell 5.
SCALES = {"1 day": 1, "7 days": 7}
RECORD, ALERTS, DISCOVERIES = {}, [], []
_expected_false = 0.0
for p in DAILY:
    y = DAILY[p]
    res_label, unit = RESOURCE[METER_INFO[p][1]]["label"], RESOURCE[METER_INFO[p][1]]["unit"]
    base_gap = (y - OOS[p]).loc[BASE_START:BASE_END]
    mon_gap = (y - EXPECTED[p]).loc[MON_START:]
    for name, w in SCALES.items():
        bg = base_gap.rolling(w, min_periods=w).sum().dropna()
        mg = mon_gap.rolling(w, min_periods=w).sum().dropna()
        hi, lo = float(bg.max()), float(bg.min())
        RECORD[(p, name)] = (hi, lo, len(bg))
        _expected_false += len(mg) / (len(bg) + 1)
        hits = mg[mg > hi]
        for d, v in hits.items():
            ALERTS.append({"point": p, "scale": name, "end": d, "start": d - pd.Timedelta(days=w - 1), "extra": v, "record": hi})
        if name == "7 days":
            for d, v in mg[mg < lo].items():
                DISCOVERIES.append({"point": p, "end": d, "start": d - pd.Timedelta(days=6), "less": -v, "record": -lo})


def _merge(df):
    """Join overlapping flagged windows into one event per meter."""
    out = []
    for p, g in df.sort_values("start").groupby("point", sort=False):
        cur = None
        for _, r in g.iterrows():
            if cur is not None and r.start <= cur["end"] + pd.Timedelta(days=1):
                cur["end"] = max(cur["end"], r.end); cur["rows"].append(r)
            else:
                if cur: out.append(cur)
                cur = {"point": p, "start": r.start, "end": r.end, "rows": [r]}
        if cur: out.append(cur)
    return out


def _qty(v, p):
    return f"{v:,.0f} {RESOURCE[METER_INFO[p][1]]['unit']}"


ALERT_TABLE = []
for ev in _merge(pd.DataFrame(ALERTS, columns=["point", "scale", "end", "start", "extra", "record"])):
    p = ev["point"]; worst = max(ev["rows"], key=lambda r: r.extra - r.record)
    y, e = DAILY[p].loc[ev["start"]:ev["end"]], EXPECTED[p].loc[ev["start"]:ev["end"]]
    extra = float((y - e).clip(lower=0).sum())
    ALERT_TABLE.append({
        "id": "", "meter": f"{RESOURCE[METER_INFO[p][1]]['icon']} {METER_INFO[p][2]}", "point": p,
        "from": ev["start"].date(), "to": ev["end"].date(), "extra use": _qty(extra, p),
        "extra %": round(extra / max(float(e.sum()), 1e-9) * 100, 1),
        "what happened": (f"{RESOURCE[METER_INFO[p][1]]['label']} at {METER_INFO[p][2]} used {_qty(worst.extra, p)} more than "
                          f"expected over {worst.scale} (ending {worst.end:%a %d %b}). The most it ever went over in the whole "
                          f"baseline year was {_qty(worst.record, p)}. This is new — worth a check."),
    })
ALERT_TABLE = pd.DataFrame(ALERT_TABLE, columns=["id", "meter", "point", "from", "to", "extra use", "extra %", "what happened"])
ALERT_TABLE["id"] = [f"AL-{i + 1:02d}" for i in range(len(ALERT_TABLE))]

DISCOVERY_TABLE = []
for ev in _merge(pd.DataFrame(DISCOVERIES, columns=["point", "end", "start", "less", "record"])):
    p = ev["point"]
    y, e = DAILY[p].loc[ev["start"]:ev["end"]], EXPECTED[p].loc[ev["start"]:ev["end"]]
    less = float((e - y).sum())
    DISCOVERY_TABLE.append({"meter": f"{RESOURCE[METER_INFO[p][1]]['icon']} {METER_INFO[p][2]}", "point": p,
                            "first seen": ev["rows"][0].end.date(), "from": ev["start"].date(), "to": ev["end"].date(),
                            "less than expected": _qty(less, p), "%": round(less / max(float(e.sum()), 1e-9) * 100, 1)})
DISCOVERY_TABLE = pd.DataFrame(DISCOVERY_TABLE, columns=["meter", "point", "first seen", "from", "to", "less than expected", "%"])
EXPECTED_FALSE_ALERTS = _expected_false

print(f"Alerts after the baseline: {len(ALERT_TABLE)}  (if nothing had changed, ≈{EXPECTED_FALSE_ALERTS:.1f} "
      f"flagged windows would be expected by chance)")
print(ALERT_TABLE[["id", "meter", "from", "to", "extra use", "extra %"]].to_string(index=False))
print(f"\nDiscoveries (sustained drops nobody has explained): {len(DISCOVERY_TABLE)}")
print(DISCOVERY_TABLE.drop(columns="point").to_string(index=False))


## Cell 7 · Waste
666 real hauling invoices — tonnes and cost per stream and building — and the **diversion rate** (compost + recycling ÷ total sorted). Bulky waste is shown but left out of the rate (Concordia: split unknown). Pickups with no weight are counted, not guessed.

In [ ]:
# ---- Waste: Concordia's real hauling invoices (Zero Waste program) ----
# Mass is in metric tonnes and Qty is a count of bins (Concordia API documentation, waste/invoices).
# Bulky waste is a mix of recycling and landfill in unknown proportions (Concordia's own definition),
# so it is reported separately and left out of the diversion rate. Pickups with no recorded weight are
# counted and shown — never filled in.
WASTE_TYPES = pd.read_csv(PATHS["WASTE_TYPE.csv"])
BUILDINGS = pd.read_csv(PATHS["BUILDING_LIST.csv"])
_w = pd.read_csv(PATHS["WASTE_INVOICES.csv"])
_w = _w.loc[:, [c for c in _w.columns if not c.startswith("Unnamed")]]
_w["Date"] = pd.to_datetime(_w["Date"], errors="coerce")
_w["tonnes"] = pd.to_numeric(_w["Mass"], errors="coerce")
_w["cost"] = pd.to_numeric(_w["Total_Cost"].astype(str).str.replace(r"[$,\s]", "", regex=True), errors="coerce")
WASTE = _w.dropna(subset=["Date"]).copy()
WASTE["month"] = WASTE["Date"].dt.to_period("M").dt.to_timestamp()
DIVERTED = ["Organics", "Paper", "Mixed_recycling"]                # sent to composting / recycling
STREAM_COLOR = {"Landfill": "#94A3B8", "Organics": "#22C55E", "Paper": "#38BDF8", "Mixed_recycling": "#A78BFA",
                "Bulky_waste": "#F59E0B"}
STREAM_LABEL = {"Landfill": "Landfill", "Organics": "Compost", "Paper": "Paper & cardboard",
                "Mixed_recycling": "Mixed recycling", "Bulky_waste": "Bulky (mixed, unknown split)"}

WASTE_MONTHLY = WASTE.pivot_table(index="month", columns="Waste_Type", values="tonnes", aggfunc="sum")
_weighed = WASTE[WASTE.tonnes.notna()].pivot_table(index="month", columns="Waste_Type", values="tonnes", aggfunc="count")
_full = _weighed.reindex(columns=["Landfill"] + DIVERTED).notna().all(axis=1)     # month has every stream weighed
_div = WASTE_MONTHLY.reindex(columns=DIVERTED).sum(axis=1)
_land = WASTE_MONTHLY.reindex(columns=["Landfill"]).sum(axis=1)
DIVERSION = (_div / (_div + _land) * 100).where(_full)
WASTE_BY_BUILDING = (WASTE.pivot_table(index="Building", columns="Waste_Type", values="tonnes", aggfunc="sum")
                     .fillna(0).round(1))
WASTE_BY_BUILDING["Total t"] = WASTE_BY_BUILDING.sum(axis=1)
WASTE_BY_BUILDING = WASTE_BY_BUILDING.sort_values("Total t", ascending=False)
WASTE_COST_PER_T = (WASTE[WASTE.tonnes > 0].groupby("Waste_Type").apply(lambda g: g.cost.sum() / g.tonnes.sum())).round(0)
NO_WEIGHT = int(WASTE.tonnes.isna().sum())

# before vs during the COVID closure, same 12-month baseline year as the meters
_b = WASTE[(WASTE.Date >= BASE_START) & (WASTE.Date <= BASE_END)]
WASTE_BASE = _b.groupby("Waste_Type").tonnes.sum().round(1)
_d = DIVERSION.loc[BASE_START:BASE_END].dropna()

print(f"Invoices: {len(WASTE)} ({WASTE.Date.min():%b %Y} → {WASTE.Date.max():%b %Y}), "
      f"{WASTE.Building.nunique()} buildings, {NO_WEIGHT} pickups with no recorded weight (left out, not guessed)")
print("\nTonnes in the baseline year:", WASTE_BASE.to_dict())
print(f"Diversion rate (compost + recycling ÷ those + landfill), months with every stream weighed: "
      f"baseline-year median {_d.median():.0f}%  ({len(_d)} months)")
print("\nCost per tonne ($ as invoiced):", WASTE_COST_PER_T.to_dict())
print("\nTonnes by building (all years):")
print(WASTE_BY_BUILDING.to_string())

# 12-month periods aligned with the meter baseline year (Mar → Feb)
WASTE["period"] = WASTE.Date.apply(lambda d: f"{d.year if d.month >= BASE_START.month else d.year - 1}–"
                                             f"{(d.year if d.month >= BASE_START.month else d.year - 1) + 1}")
_y = WASTE.pivot_table(index="period", columns="Waste_Type", values="tonnes", aggfunc="sum").fillna(0)
_y = _y[[c for c in ["Landfill"] + DIVERTED + ["Bulky_waste"] if c in _y]]
_cnt = WASTE.groupby("period").Date.agg(["min", "max"])
WASTE_YEARLY = _y.round(1)
_wc = WASTE[WASTE.tonnes > 0].pivot_table(index="period", columns="Waste_Type", values="tonnes", aggfunc="count")
_allw = _wc.reindex(index=_y.index, columns=["Landfill"] + DIVERTED).notna().all(axis=1)
for c in _y.columns:                                  # a stream never weighed in a period shows "—", not 0
    WASTE_YEARLY[c] = WASTE_YEARLY[c].where(_wc.reindex(index=_y.index).get(c, pd.Series(index=_y.index)).notna())
WASTE_YEARLY["Diversion %"] = (_y[DIVERTED].sum(axis=1) / (_y[DIVERTED].sum(axis=1) + _y["Landfill"]) * 100).round(0).where(_allw)
WASTE_YEARLY["Invoices from"] = _cnt["min"].dt.strftime("%b %Y")
WASTE_YEARLY["to"] = _cnt["max"].dt.strftime("%b %Y")
print("\nTonnes per 12-month period (Mar→Feb; partial periods marked by their invoice dates):")
print(WASTE_YEARLY.to_string())
print("Diversion % is shown only for periods where landfill, compost, paper and mixed recycling were all weighed.")
print("Note: 'Building' is where the bin or compactor sits — one compactor can serve several buildings.")


## Cell 7e · E-waste
Reads **your real asset register** (template provided) and routes every device the way India's **E-Waste (Management) Rules, 2022** require: keep in use, a *registered* refurbisher, or a *registered* recycler. Expected life comes from the CPCB EPR portal for each EEE code. Concordia's open data has no e-waste records, so nothing is shown until you upload a register — nothing is invented.

In [ ]:
# ---- E-waste: asset register → keep · sell second-hand · repair · harvest parts · recycle ----
# Upload your real register as ewaste_register.csv (template written below). If none is uploaded, the app shows the clearly-labelled
# SIMULATED departmental register from cell 13c, so the page is never empty. Disposal follows India's E-Waste (Management)
# Rules, 2022: anything that leaves campus as waste goes only to a REGISTERED refurbisher or recycler.
EWASTE_COLS = ["device_id", "device_type", "department", "purchase_date", "expected_life_years", "condition",
               "price_inr", "weight_kg", "embodied_kgco2e"]
EWASTE_TEMPLATE = os.path.join(OUT_DIR, "ewaste_register_TEMPLATE.csv")
pd.DataFrame(columns=EWASTE_COLS).to_csv(EWASTE_TEMPLATE, index=False)
EWASTE_HELP = {"condition": "working / faulty / non-functional",
               "expected_life_years": "average life for the device's EEE code (CPCB EPR portal)",
               "price_inr": "purchase price (for the second-hand value)", "weight_kg": "from the spec sheet",
               "embodied_kgco2e": "manufacturer's product carbon footprint (PCF) sheet"}
EWASTE_EOL_T_CO2E_PER_T = 0.02 / 0.907185    # EPA GHG Emission Factors Hub (2024), mixed electronics, recycled
# Reusable parts by device type — what a technician pulls before anything is recycled
REUSABLE_PARTS = {
    "Laptop": "RAM, SSD, charger, battery (if health > 80%)", "Desktop PC": "RAM, SSD/HDD, power supply, graphics card",
    "Monitor": "whole screen for reuse, stand, cables", "Printer": "toner, paper trays, power board",
    "Projector": "lamp, lens, remote", "Air conditioner": "compressor, copper coil, fan motor",
    "UPS": "batteries (after a load test), enclosure", "Network switch": "SFP modules, power supply",
    "Server": "RAM, drives, power supplies, CPUs",
}
SELL_FROM, SELL_TO = 0.6, 1.0       # second-hand window: from 60% of expected life until the end of it (working units)
RESALE_MARKET = 0.5                 # used-market price ≈ half of straight-line remaining value (illustrative)
PATHS["ewaste_register.csv"] = find("ewaste_register.csv")


def ewaste_plan(df, today=None):
    """Route every device and say WHEN: keep, sell second-hand (with value and sell-by date), repair, harvest parts, recycle."""
    today = pd.Timestamp(today or datetime.date.today())
    d = df.copy()
    for c in EWASTE_COLS:
        if c not in d:
            d[c] = np.nan
    d["purchase_date"] = pd.to_datetime(d["purchase_date"], errors="coerce")
    for c in ["expected_life_years", "weight_kg", "embodied_kgco2e", "price_inr"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d["condition"] = d["condition"].astype(str).str.strip().str.lower()
    d["age_years"] = ((today - d["purchase_date"]).dt.days / 365.25).round(1)
    d["life_used"] = d["age_years"] / d["expected_life_years"]
    d["life_used_%"] = (d["life_used"] * 100).round(0)
    d["sell_from"] = d["purchase_date"] + pd.to_timedelta(d["expected_life_years"] * SELL_FROM * 365.25, unit="D")
    d["sell_by"] = d["purchase_date"] + pd.to_timedelta(d["expected_life_years"] * SELL_TO * 365.25, unit="D")
    d["resale_inr"] = (d["price_inr"] * (1 - d["life_used"]).clip(lower=0) * RESALE_MARKET).round(-2)
    d["parts"] = d["device_type"].map(REUSABLE_PARTS).fillna("—")

    def route(r):
        u, c = r["life_used"], r["condition"]
        if c == "working" and u < SELL_FROM:
            return "Keep in use", f"working, {r['life_used_%']:.0f}% of its life used — sell window opens {r['sell_from']:%b %Y}"
        if c == "working" and u < SELL_TO:
            return "Sell second-hand", f"working and still valuable — sell before {r['sell_by']:%b %Y}"
        if c == "working":
            return "Donate / refurbisher", "working but past its expected life — reuse before recycling"
        if c == "faulty" and u < SELL_FROM:
            return "Repair", "faulty but young — repair is cheaper than replacing"
        if c == "faulty":
            return "Harvest parts", f"pull reusable parts first: {r['parts']}"
        if c == "non-functional":
            return "Recycle (registered)", "hand to a CPCB-registered recycler; keep the receipt"
        return "Check record", "condition not recorded"
    rt = d.apply(route, axis=1, result_type="expand")
    d["route"], d["why"] = rt[0], rt[1]
    d["leaving campus"] = d["route"].isin(["Sell second-hand", "Donate / refurbisher", "Recycle (registered)"])
    d.loc[d.route != "Sell second-hand", "resale_inr"] = np.nan
    d["eol_kgco2e"] = (d["weight_kg"] / 1000 * EWASTE_EOL_T_CO2E_PER_T * 1000).where(d["route"] == "Recycle (registered)")
    # reuse keeps embodied carbon in use (no new device needed): counted for sell / donate / repair
    d["co2e_kept_kg"] = d["embodied_kgco2e"].where(d["route"].isin(["Sell second-hand", "Donate / refurbisher", "Repair"]))
    d["sell_soon"] = (d["condition"] == "working") & (d["sell_from"] > today) & (d["sell_from"] <= today + pd.DateOffset(months=12))
    return d


EWASTE, EWASTE_SOURCE = None, None
if PATHS["ewaste_register.csv"]:
    EWASTE, EWASTE_SOURCE = ewaste_plan(pd.read_csv(PATHS["ewaste_register.csv"])), "your register"
    print(f"E-waste register loaded: {len(EWASTE)} devices")
    print(EWASTE.groupby("route").size().to_string())
else:
    print("No e-waste register uploaded — the app uses the campus model's device register (cell 13c).")


## Cell 8 · Action register + "Did it work?" — contextual ghost twin + placebo-in-time
* **Ghost twin:** the building *without* the action, trained on the 12 months before and fed the **real** weather and calendar of every later day.
* **Placebo-in-time:** we pretend the action happened on every week of the year before (when nothing was done). Those fake "savings" show what noise looks like.
* **VERIFIED** only if the real change is outside the noise: 95% range excludes zero **and** p ≤ 5%. We count only the **low end**.

The register starts with one **ground-truth test** — the documented COVID-19 closure (13 March 2020) — to prove the engine on a real event. Approved recommendations and your own actions are added in the app.

🎤 *"A before/after dashboard says gas fell 40%. The ghost twin, fed the real spring weather, says that is normal — not proven, nothing counted."*

In [ ]:
# ---- Action register + "Did it work?" — contextual GHOST TWIN + PLACEBO-IN-TIME test ----
#   1. GHOST TWIN: a copy of the meter that never got the action. It is trained on the 12 months BEFORE the
#      action and then fed the REAL context of every later day (actual weather, weekday/holiday, season), so it
#      shows what the building would have used without the action. (IPMVP Option C / ASHRAE Guideline 14.)
#   2. saving = ghost twin − actual
#   3. PLACEBO-IN-TIME: we pretend the action happened on every week of the year before (when nothing was
#      done), hide that stretch, rebuild the ghost twin without it, and record the fake "saving". Those fakes
#      show what pure noise looks like for this meter.
#        • 95% range = ± t × (typical fake saving) × √(1 + 1/k), k = independent stretches in a year
#        • p-value  = share of fake savings at least as large as the real one (permutation test)
#   4. VERIFIED only if the whole 95% range excludes zero AND p ≤ 5%. We count the LOW end of the range.
ACTIONS_CSV = os.path.join(OUT_DIR, "actions.csv")
ACTION_COLS = ["action_id", "date", "point", "title", "type", "claimed_%", "source"]
if os.path.exists(ACTIONS_CSV):
    ACTIONS = pd.read_csv(ACTIONS_CSV, parse_dates=["date"])
else:
    # one real, documented event to start with (not a sustainability action — clearly labelled)
    ACTIONS = pd.DataFrame([{
        "action_id": "EV-01", "date": pd.Timestamp("2020-03-14"), "point": p,
        "title": "Ground-truth test — COVID-19: both campuses closed from midnight 13 March 2020",
        "type": "Ground-truth test (real, documented event) — proves the engine; not a sustainability saving",
        "claimed_%": np.nan,
        "source": "The Link, 13 Mar 2020: thelinknewspaper.ca/article/concordia-closed-until-march-29-due-to-covid-19-prevention"}
        for p in DAILY], columns=ACTION_COLS)
    ACTIONS["action_id"] = [f"EV-01{chr(97 + i)}" for i in range(len(ACTIONS))]
    ACTIONS.to_csv(ACTIONS_CSV, index=False)


def add_action(date, point, title, claimed_pct=None, source="logged in GreenLedger"):
    global ACTIONS
    aid = f"A-{len(ACTIONS) + 1:03d}"
    ACTIONS = pd.concat([ACTIONS, pd.DataFrame([{"action_id": aid, "date": pd.Timestamp(date), "point": point, "title": title,
                                                  "type": "Sustainability action", "claimed_%": claimed_pct,
                                                  "source": source}])], ignore_index=True)
    ACTIONS.to_csv(ACTIONS_CSV, index=False)
    return aid


WINDOWS = [28, 42, 56]          # 4, 6 or 8 whole weeks after the action (same weekday mix every time)
ELEC_KG_CO2_PER_KWH = 0.0005    # Hydro-Québec, 0.5 kg CO2/MWh for 2019 and 2020 ("CO2 emissions and Hydro-Québec electricity 1990-2020")
GAS_KG_CO2_PER_M3 = 1.926       # Québec MELCC GHG quantification guide, Table 5 (natural gas, CO2)
GAS_M3_PER_METER_UNIT = None    # Concordia's API documents m³. Sanity check (cell 10): at 1 unit = 1 m³ the EV+GM+MB meter
                                # alone would emit ~57,000 t CO₂/yr — far above everything else measured — so the unit is
                                # treated as unconfirmed. Set to 1.0 once Facilities confirms m³ (the app has a switch).


INDIA_GRID_KG_PER_KWH = 0.710   # CEA CO2 Baseline Database v21.0 (India) — used for the simulated Indian departments


def co2_kg(point, amount):
    if str(point).startswith("D_"):                  # simulated departments sit on India's grid
        return amount * INDIA_GRID_KG_PER_KWH
    res = METER_INFO[point][1]
    if res == "electricity":
        return amount * ELEC_KG_CO2_PER_KWH
    if res == "gas" and GAS_M3_PER_METER_UNIT:
        return amount * GAS_M3_PER_METER_UNIT * GAS_KG_CO2_PER_M3
    return None


ALPHA = 0.05                       # 95% confidence — statistical convention
PLACEBO_STEP = 7                   # fake action dates one week apart: every date has the same weekday mix


def placebo_in_time(y_base, window, kind="cp"):
    """Fake action on every week of the baseline year → the 'saving' the ghost twin finds when nothing was done."""
    start, end = y_base.index.min(), y_base.index.max()
    fakes, d = [], start
    while d + pd.Timedelta(days=window - 1) <= end:
        b = d + pd.Timedelta(days=window - 1)
        hold = y_base.loc[d:b].dropna()
        if len(hold) >= COMPLETE * window:
            train = y_base.copy(); train.loc[d:b] = np.nan
            e = expected(fit_model(train, kind), hold.index)
            fakes.append(float((e.sum() - hold.sum()) / e.sum() * 100))
        d += pd.Timedelta(days=PLACEBO_STEP)
    return np.array(fakes)


def range_from_misses(misses, k=None):
    """Prediction interval from real misses. k = number of INDEPENDENT stretches (overlapping fakes share data)."""
    k = k or len(misses)
    typical = float(np.sqrt(np.mean(np.square(misses))))       # typical miss (root-mean-square, includes any bias)
    return float(stats.t.ppf(1 - ALPHA / 2, k - 1) * typical * np.sqrt(1 + 1 / k)), typical


def verify(point, date, window=42, other_events=None):
    T0 = pd.Timestamp(date).normalize()
    res, unit = METER_INFO[point][1], RESOURCE[METER_INFO[point][1]]["unit"]
    y = DAILY[point].where(TEMP.notna())
    base = y.loc[T0 - pd.Timedelta(days=365): T0 - pd.Timedelta(days=1)]
    post = y.loc[T0: T0 + pd.Timedelta(days=window - 1)]
    out = {"point": point, "date": T0, "window": window, "unit": unit}
    if post.notna().sum() < COMPLETE * window:
        return {**out, "verdict": "WAITING FOR DATA",
                "why": f"Needs {window} days of good readings after the action; {int(post.notna().sum())} available."}
    if base.notna().sum() < COMPLETE * 365:
        return {**out, "verdict": "NOT ENOUGH HISTORY",
                "why": f"Needs 12 months of good readings (with weather) before the action; "
                       f"{int(base.notna().sum())} days available."}
    kind = MODEL_KIND[point]
    cp = fit_model(base, "cp")
    cv, nmbe, rho = accuracy(base, cp)                       # ASHRAE Guideline 14 grade on the change-point fit
    ghost = cp if kind == "cp" else fit_model(base, kind)   # the meter's better predictor (cell 5)
    fakes = placebo_in_time(base, window, kind)
    k_ind = 365 // window
    if len(fakes) < k_ind:
        return {**out, "verdict": "NOT ENOUGH HISTORY", "why": "Too few complete weeks in the baseline year for the placebo test."}
    half, typical = range_from_misses(fakes, k_ind)
    yp = post.dropna()
    exp_ = expected(ghost, yp.index)
    saving = float((exp_.sum() - yp.sum()) / exp_.sum() * 100)
    lo, hi = saving - half, saving + half
    p = float((1 + np.sum(np.abs(fakes) >= abs(saving))) / (len(fakes) + 1))
    pre = y.loc[T0 - pd.Timedelta(days=window): T0 - pd.Timedelta(days=1)].mean()
    naive = float((pre - yp.mean()) / pre * 100)
    reliable = cv <= 30 and abs(nmbe) <= 10
    if not reliable:
        verdict = "MODEL NOT ACCURATE ENOUGH"
    elif lo > 0 and p <= ALPHA:
        verdict = "VERIFIED SAVING"
    elif hi < 0 and p <= ALPHA:
        verdict = "VERIFIED INCREASE"
    else:
        verdict = "NOT PROVEN"
    warn = []
    if other_events is not None:                         # IPMVP: a non-routine event in the baseline or the window
        clash = other_events[(other_events.point == point) & (other_events.date >= T0 - pd.Timedelta(days=365)) &
                             (other_events.date <= T0 + pd.Timedelta(days=window - 1))]
        if len(clash):
            warn.append("Another logged event on this meter falls inside the 12-month baseline or the measurement window "
                        f"({', '.join(clash.title.str[:50])}), so the change cannot be attributed to this action alone.")
            if verdict in ("VERIFIED SAVING", "VERIFIED INCREASE"):
                verdict = "CONFOUNDED"
    counted = max(0.0, lo / 100 * float(exp_.sum())) if verdict == "VERIFIED SAVING" else 0.0
    return {**out, "verdict": verdict, "saving_%": saving, "low_%": lo, "high_%": hi, "p": p, "naive_%": naive,
            "expected": float(exp_.sum()), "actual": float(yp.sum()), "counted": counted,
            "counted_co2_kg": co2_kg(point, counted) if counted else 0.0, "cv": cv, "rho": rho,
            "fakes": fakes, "typical_miss": typical, "kind": kind, "k_ind": k_ind,
            "days": len(yp), "model": ghost, "series_exp": exp_, "series_act": yp, "warnings": warn}


def explain(r):
    """One plain-English paragraph a judge can read."""
    name = f"{RESOURCE[METER_INFO[r['point']][1]]['label']} at {METER_INFO[r['point']][2]}"
    if "saving_%" not in r:
        return f"{r['verdict']}: {r['why']}"
    u = r["unit"]
    txt = (f"Over the {r['days']} days after {r['date']:%d %b %Y}, {name} used {r['actual']:,.0f} {u}. "
           f"Based on the 12 months before — adjusted for the actual weather and weekdays/holidays — it would have used "
           f"about {r['expected']:,.0f} {u} — that is its ghost twin ({'gradient-boosting' if r['kind'] == 'ml' else 'change-point'} model, "
           f"fed the real weather and calendar). That is a {abs(r['saving_%']):.1f}% {'drop' if r['saving_%'] > 0 else 'rise'}. "
           f"Placebo test: we pretended the action happened on {len(r['fakes'])} other dates last year; those fake "
           f"'savings' were typically {r['typical_miss']:.1f}%, so the 95% range is {r['low_%']:.1f}% to {r['high_%']:.1f}% "
           f"and only {r['p'] * 100:.0f}% of fakes were this large (p = {r['p']:.3f}). "
           f"A simple before/after comparison would have said {r['naive_%']:.1f}%. ")
    if r["verdict"] == "VERIFIED SAVING":
        txt += f"Both tests pass, so we count the LOW end: {r['counted']:,.0f} {u}."
        if r["counted_co2_kg"]:
            grid = "India's grid (CEA)" if str(r["point"]).startswith("D_") else "Québec's hydro-powered grid"
            txt += f" That avoids about {r['counted_co2_kg']:,.0f} kg CO₂ on {grid}."
    elif r["verdict"] == "VERIFIED INCREASE":
        txt += "Use went UP beyond normal variation."
    elif r["verdict"] == "CONFOUNDED":
        txt += "The change is real, but another logged event overlaps it — nothing is counted for this action."
    elif r["verdict"] == "MODEL NOT ACCURATE ENOUGH":
        txt += (f"This meter's model error ({r['cv']:.0f}%) is above the ASHRAE Guideline 14 limit, so no saving is "
                f"claimed — only the raw numbers are shown.")
    else:
        txt += "At least one test fails, so the change could be normal variation. Nothing is counted."
    for w in r["warnings"]:
        txt += " ⚠ " + w
    return txt


VERIFIED = {}
for _, a in ACTIONS.iterrows():
    VERIFIED[a.action_id] = verify(a.point, a.date, 42, ACTIONS[ACTIONS.action_id != a.action_id])
    print(f"[{a.action_id}] {RESOURCE[METER_INFO[a.point][1]]['icon']} {METER_INFO[a.point][2]}: {VERIFIED[a.action_id]['verdict']}")
    print("    " + explain(VERIFIED[a.action_id]))


## Cell 9 · Can you trust it? (the fair test)
Normal months where we **know nothing was done**, each hidden from the model: how often does each method report a "saving"?

🎤 *"On months where nothing happened, a simple dashboard reported savings almost half the time. GreenLedger almost never does."*

In [ ]:
# ---- "Can you trust it?" — a fair test where we KNOW nothing was done ----
# For each of the 12 normal months in the baseline year, we pretend an action started on the 1st,
# hide that month from the model, and ask both methods whether it "saved" anything.
# A simple before/after dashboard (this month vs the 30 days before) vs. GreenLedger (cell 8 method).
TRUST = []
for p in RELIABLE:
    y = DAILY[p].where(TEMP.notna())
    miss = MISS[p]                                           # each month predicted by a model that never saw it (cell 5)
    for a, gl in miss.items():
        others = np.array([v for k, v in miss.items() if k != a])       # the range never uses the month being tested
        half, _ = range_from_misses(others)              # monthly stretches don't overlap → k = number of months
        post = y.loc[a:a + pd.offsets.MonthEnd(0)].dropna()
        prev = y.loc[a - pd.Timedelta(days=30): a - pd.Timedelta(days=1)].mean()
        TRUST.append({"point": p, "month": a, "naive_%": float((prev - post.mean()) / prev * 100),
                      "gl_%": gl, "gl_low": gl - half, "gl_high": gl + half})
TRUST = pd.DataFrame(TRUST)
TT = TRUST.dropna(subset=["gl_%", "naive_%"])
TRUST_SUMMARY = {
    "tests": len(TT),
    "naive_saving": int((TT["naive_%"] > 0).sum()),
    "naive_big": int((TT["naive_%"] >= 5).sum()),
    "gl_saving": int((TT.gl_low > 0).sum()),
    "gl_increase": int((TT.gl_high < 0).sum()),
}
TRUST_SUMMARY["naive_avg_abs"] = round(float(TT["naive_%"].abs().mean()), 1)
TRUST_SUMMARY["gl_avg_abs"] = round(float(TT["gl_%"].abs().mean()), 1)
print(f"{TRUST_SUMMARY['tests']} normal months on meters with an accurate model — nothing was done in any of them.")
print(f"  A simple before/after dashboard reported a 'saving' in {TRUST_SUMMARY['naive_saving']} of them "
      f"({TRUST_SUMMARY['naive_big']} of 5% or more); average apparent change {TRUST_SUMMARY['naive_avg_abs']}%.")
print(f"  GreenLedger verified a saving in {TRUST_SUMMARY['gl_saving']} and an increase in {TRUST_SUMMARY['gl_increase']}; "
      f"average apparent change {TRUST_SUMMARY['gl_avg_abs']}%.")


## Cell 9r · 💡 Recommendations — the optimisation layer
Turns each building's own data into actions, plus a **what-if simulator** (you pick: °C of setpoint change, % less on weekends, % less always-on load → kWh, m³, CO₂ from the building's own model and last year's real weather): weekend/holiday schedules (benchmarked against another Concordia meter that already does better), heating/cooling sensitivity per 1 °C (with last year's real weather), always-on base load, gas used regardless of weather, missing recycling/compost bins, cheaper waste streams. Every size is a what-if from real numbers — **potential, not verified** — and a person approves or rejects each one. Approved actions go into the register, so the ghost twin verifies them later.

🎤 *"Where's the optimisation? Here: recommendations sized from real data → a person approves → the ghost twin proves it. Recommend, act, verify."*

In [ ]:
# ---- 💡 Recommendations: what to do next — read straight from each building's own data ----
# No invented targets. Every size below is either
#   • an internal benchmark — "if this meter did what another Concordia meter ALREADY does", or
#   • a per-unit sensitivity — "each 1 °C / each 1 % is worth …", computed with last year's real weather.
# Each recommendation is marked "potential — not verified": once it is done, log it in the action register
# and GreenLedger verifies it after 4–8 weeks (cell 8). That closes the loop: recommend → act → verify.
LANDFILL_T_CO2E_PER_T = 0.58 / 0.907185   # EPA GHG Emission Factors Hub (2024), mixed MSW landfilled: 0.58 t CO2e per short ton
RECS = []


def _rec(group, area, where, finding, size, size_value, unit, action, point=None):
    RECS.append({"priority": group, "area": area, "where": where, "finding": finding, "potential (not verified)": size,
                 "_value": size_value, "unit": unit, "next step": action, "_point": point})


# 1) open alerts → fix now
for _, a in ALERT_TABLE.iterrows():
    _rec("1 · Fix now", RESOURCE[METER_INFO[a.point][1]]["label"], METER_INFO[a.point][2],
         a["what happened"], f"{a['extra use']} extra already used", np.nan, "", "Check what was running; log the fix as an action.", a.point)

BY_RES = {}
for p in RELIABLE:
    BY_RES.setdefault(METER_INFO[p][1], []).append(p)
BASE_DAYS = pd.date_range(BASE_START, BASE_END, freq="D")
N_OFF = int(OFF_DAY.loc[BASE_START:BASE_END].sum())

# 2) weekend / holiday schedules — internal benchmark
OFFDAY = {}
for p in RELIABLE:
    y = DAILY[p].loc[BASE_START:BASE_END]
    off = OFF_DAY.loc[BASE_START:BASE_END]
    W, O = float(y[~off].mean()), float(y[off].mean())
    OFFDAY[p] = {"weekday": W, "offday": O, "cut": (W - O) / W * 100}
for res, pts in BY_RES.items():
    best = max(pts, key=lambda q: OFFDAY[q]["cut"])
    for p in pts:
        d, bd = OFFDAY[p], OFFDAY[best]
        unit = RESOURCE[res]["unit"]
        if p == best:
            continue
        gap = (bd["cut"] - d["cut"]) / 100 * d["weekday"] * N_OFF
        _rec("2 · Quick win", RESOURCE[res]["label"], METER_INFO[p][2],
             f"On weekends and holidays it uses {abs(d['cut']):.0f}% {'less' if d['cut'] >= 0 else 'MORE'} than on weekdays. "
             f"{METER_INFO[best][2]} already cuts {bd['cut']:.0f}%.",
             f"≈ {gap:,.0f} {unit}/year if it matched {METER_INFO[best][2]}", gap, unit,
             "Review HVAC and lighting schedules for weekends and holidays.", p)

# 3) temperature sensitivity — each 1 °C of setpoint / balance temperature, with last year's real weather
for p in RELIABLE:
    m, res = CP_MODEL[p], METER_INFO[p][1]
    unit = RESOURCE[res]["unit"]
    t = TEMP.loc[BASE_START:BASE_END].dropna()
    exp_year = float(expected(m, t.index).sum())
    i = 2
    if m["th"] is not None:
        sh = m["b"][i]; i += 1
        per_deg = float(sh * np.clip(m["th"] - t, 0, 1).sum())
        share = float(sh * np.maximum(0, m["th"] - t).sum()) / exp_year * 100
        _rec("3 · Plan", RESOURCE[res]["label"], METER_INFO[p][2],
             f"{share:.0f}% of its use is driven by cold weather (starts below {m['th']} °C; +{sh:,.0f} {unit} per °C colder).",
             f"≈ {per_deg:,.0f} {unit}/year for each 1 °C lower heating balance point", per_deg, unit,
             "Lower heating setpoints / add night & weekend setbacks; then verify.", p)
    if m["tc"] is not None:
        sc = m["b"][i]
        per_deg = float(sc * np.clip(t - m["tc"], 0, 1).sum())
        share = float(sc * np.maximum(0, t - m["tc"]).sum()) / exp_year * 100
        _rec("3 · Plan", RESOURCE[res]["label"], METER_INFO[p][2],
             f"{share:.0f}% of its use is driven by warm weather (starts above {m['tc']} °C; +{sc:,.0f} {unit} per °C warmer).",
             f"≈ {per_deg:,.0f} {unit}/year for each 1 °C higher cooling balance point", per_deg, unit,
             "Raise cooling setpoints / widen the dead-band; then verify.", p)

# 4) base load (electricity, hourly data) — what runs 24/7
BASELOAD = {}
for p in [q for q in RELIABLE if METER_INFO[q][1] == "electricity"]:
    h = HOURLY[p].loc[BASE_START:BASE_END + pd.Timedelta(hours=23)]
    g = h.groupby(h.index.normalize())
    ok = g.count() >= COMPLETE * 24
    mn, mean = g.min()[ok], g.mean()[ok]
    kw = float(mn.median())
    BASELOAD[p] = {"kW": kw, "share": float((mn / mean).median() * 100), "kWh_year": kw * 8760}
    _rec("3 · Plan", "Electricity", METER_INFO[p][2],
         f"Never drops below about {kw:,.0f} kW, even at night — {BASELOAD[p]['share']:.0f}% of its average demand runs around the clock.",
         f"≈ {kw * 8760 / 100:,.0f} kWh/year for each 1% cut in this always-on load", kw * 8760 / 100, "kWh",
         "Night walk-through: ventilation, servers, lab equipment left on.", p)

# 5) gas used regardless of weather
for p in [q for q in RELIABLE if METER_INFO[q][1] == "gas"]:
    m = CP_MODEL[p]
    t = TEMP.loc[BASE_START:BASE_END].dropna()
    base_part = float((m["b"][0] + m["b"][1] * OFF_DAY.reindex(t.index).astype(float)).sum())
    share = base_part / float(expected(m, t.index).sum()) * 100
    _rec("3 · Plan", "Natural gas", METER_INFO[p][2],
         f"{share:.0f}% of its gas is used whatever the weather (≈ {m['b'][0]:,.0f} m³ on a mild weekday).",
         f"≈ {base_part / 100:,.0f} m³/year for each 1% cut in weather-independent gas", base_part / 100, "m³",
         "Check boiler lock-out in warm weather and hot-water schedules.", p)

# 6) waste — streams missing where landfill is invoiced, cost and CO2e per tonne moved
_wb = WASTE[WASTE.tonnes > 0].pivot_table(index="Building", columns="Waste_Type", values="tonnes", aggfunc="sum").fillna(0)
_wbase = (WASTE[(WASTE.Date >= BASE_START) & (WASTE.Date <= BASE_END) & (WASTE.Waste_Type == "Landfill")]
          .groupby("Building").tonnes.sum())
for bld, r in _wb.iterrows():
    if r.get("Landfill", 0) > 0:
        missing = [STREAM_LABEL[s_].lower() for s_ in DIVERTED if r.get(s_, 0) == 0]
        if missing:
            lt = float(_wbase.get(bld, 0))
            _rec("2 · Quick win", "Waste", f"{bld} bins",
                 f"{r['Landfill']:,.0f} t to landfill invoiced here (all years), but no {' / '.join(missing)} stream at this location.",
                 f"{lt:,.0f} t landfilled here in the baseline year ≈ {lt * LANDFILL_T_CO2E_PER_T:,.0f} t CO₂e; "
                 f"each tonne diverted avoids ≈ {LANDFILL_T_CO2E_PER_T:.2f} t CO₂e", lt, "t",
                 "Add the missing bins next to the landfill bins; track the diversion rate.")
_cheap = WASTE_COST_PER_T.drop(labels=["Landfill", "Bulky_waste"], errors="ignore")
_cheap = _cheap[_cheap < WASTE_COST_PER_T.get("Landfill", np.inf)]
if len(_cheap):
    s = _cheap.idxmin()
    diff = WASTE_COST_PER_T["Landfill"] - _cheap[s]
    _rec("2 · Quick win", "Waste", "campus", f"Landfill costs ${WASTE_COST_PER_T['Landfill']:,.0f}/t; {STREAM_LABEL[s].lower()} costs "
         f"${_cheap[s]:,.0f}/t (from the invoices).", f"${diff:,.0f} saved per tonne moved from landfill to {STREAM_LABEL[s].lower()}",
         diff, "$/t", "Better sorting signs / bin placement for recyclables.")
LANDFILL_BASE_T = float(WASTE_BASE.get("Landfill", 0))
LANDFILL_BASE_CO2E = LANDFILL_BASE_T * LANDFILL_T_CO2E_PER_T

RECS = pd.DataFrame(RECS).sort_values(["priority", "area", "_value"], ascending=[True, True, False]).reset_index(drop=True)
_wb_drop = RECS[(RECS.area == "Waste") & (RECS._value < 1)].index      # locations with no landfill in the baseline year
RECS = RECS.drop(_wb_drop).reset_index(drop=True)
RECS.insert(0, "id", [f"R-{i + 1:02d}" for i in range(len(RECS))])

# ---- human in the loop: every recommendation waits for a person to approve or reject it (with a reason) ----
DECISIONS_CSV = os.path.join(OUT_DIR, "recommendation_decisions.csv")
DECISIONS = pd.read_csv(DECISIONS_CSV) if os.path.exists(DECISIONS_CSV) else pd.DataFrame(
    columns=["time", "rec_id", "decision", "by", "reason", "action_id"])


def decide_rec(rec_id, decision, by="", reason="", action_date=None):
    """Approve → the action goes into the register (dated) so the ghost twin can verify it later. Reject → reason kept."""
    global DECISIONS
    r = RECS.set_index("id").loc[rec_id]
    aid = ""
    if decision == "approved" and isinstance(r["_point"], str) and action_date:
        aid = add_action(action_date, r["_point"], f"{r['next step']} ({rec_id})", None, f"recommendation {rec_id}")
    DECISIONS = pd.concat([DECISIONS, pd.DataFrame([{"time": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
                                                     "rec_id": rec_id, "decision": decision, "by": by, "reason": reason,
                                                     "action_id": aid}])], ignore_index=True)
    DECISIONS.to_csv(DECISIONS_CSV, index=False)
    return aid
TOP_RECS = pd.concat([RECS[RECS.priority.str.startswith("1")].head(1),
                      RECS[RECS.priority.str.startswith("2") & (RECS.area != "Waste")].head(1),
                      RECS[(RECS.priority.str.startswith("3")) & (RECS.area == "Electricity")].head(1)])
print(f"{len(RECS)} recommendations ({RECS.priority.value_counts().sort_index().to_dict()})")
print(f"Landfill in the baseline year: {LANDFILL_BASE_T:,.0f} t ≈ {LANDFILL_BASE_CO2E:,.0f} t CO₂e (EPA factor)\n")
for _, r in RECS.iterrows():
    print(f"{r.id} [{r.priority}] {r.area} · {r['where']}\n    {r.finding}\n    → {r['potential (not verified)']} · {r['next step']}")


# ---- 🎚️ What-if simulator: YOU choose the scenario; the numbers come from each building's own model ----
# Uses the change-point slopes and LAST YEAR'S REAL WEATHER, campus meters only (no double counting:
# EV sits inside EV+GM+MB+TD, GM inside EV+GM+MB). These are projections — not savings until verified.
_T_BASE = TEMP.loc[BASE_START:BASE_END].dropna()
_OFF_BASE = OFF_DAY.reindex(_T_BASE.index).astype(bool)


def _slopes(m):
    b = list(m["b"]); i = 2; sh = sc = 0.0
    if m["th"] is not None:
        sh = b[i]; i += 1
    if m["tc"] is not None:
        sc = b[i]
    return sh, sc


def what_if(heat_down=0.0, cool_up=0.0, offday_cut=0.0, base_cut=0.0):
    """heat_down: °C lower heating balance point · cool_up: °C higher cooling balance point ·
    offday_cut: % less use on weekends/holidays · base_cut: % less always-on electricity."""
    rows = []
    for res in ("electricity", "gas"):
        for p in CAMPUS_TOTAL[res]:
            if p not in RELIABLE:
                continue
            m = CP_MODEL[p]; sh, sc = _slopes(m); t = _T_BASE
            heat = sh * float((np.maximum(0, m["th"] - t) - np.maximum(0, m["th"] - heat_down - t)).sum()) if m["th"] is not None else 0.0
            cool = sc * float((np.maximum(0, t - m["tc"]) - np.maximum(0, t - m["tc"] - cool_up)).sum()) if m["tc"] is not None else 0.0
            offd = float(expected(m, t.index)[_OFF_BASE.values].sum()) * offday_cut / 100
            base = BASELOAD[p]["kWh_year"] * base_cut / 100 if p in BASELOAD else 0.0
            rows.append({"point": p, "meter": METER_INFO[p][2], "resource": res, "unit": RESOURCE[res]["unit"],
                         "heating": heat, "cooling": cool, "weekends": offd, "base load": base,
                         "total": heat + cool + offd + base})
    W = pd.DataFrame(rows)
    kwh = float(W[W.resource == "electricity"].total.sum()); m3 = float(W[W.resource == "gas"].total.sum())
    co2_e = kwh * ELEC_KG_CO2_PER_KWH / 1000
    co2_g = m3 * (GAS_M3_PER_METER_UNIT or 0) * GAS_KG_CO2_PER_M3 / 1000 if GAS_M3_PER_METER_UNIT else None
    return W, kwh, m3, co2_e, co2_g


_W, _k, _m, _ce, _cg = what_if(1, 1, 10, 1)
print(f"\nWhat-if example (1 °C lower heating point, 1 °C higher cooling point, 10% less on weekends, 1% less always-on load), "
      f"with last year's weather: ≈ {_k / 1000:,.0f} MWh electricity and ≈ {_m:,.0f} m³ (meter units) of gas per year "
      f"→ {_ce:,.1f} t CO₂ from electricity. Projection only — verify after doing it.")


## Cell 10 · 🌍 Carbon report
GHG Protocol scopes with sourced factors: Scope 2 electricity (Hydro-Québec), Scope 3 landfill and e-waste (US EPA Emission Factors Hub), Scope 1 gas withheld until the meter unit is confirmed. Data gaps are not filled.

🎤 *"In Québec, landfill waste out-emits all metered electricity — on India's grid the same electricity would be ~1,400× more CO₂. Where you are changes what to fix first."*

In [ ]:
# ---- 🌍 Carbon report (GHG Protocol scopes) — only what can be measured and sourced ----
# Scope 1  natural gas burned on campus  → needs the gas meter unit confirmed (cell 8); until then: not computed
# Scope 2  purchased electricity         → metered kWh × Hydro-Québec 0.5 kg CO2/MWh (2019, 2020)
# Scope 3  waste sent to landfill        → invoiced tonnes × EPA landfill factor (cell 9r)
#          e-waste handed over           → kg × EPA mixed-electronics factor (cell 7e), if a register is uploaded
# Days with no good reading are NOT filled in; each line shows how complete the data was.
def _period(d):
    y0 = d.year if d.month >= BASE_START.month else d.year - 1
    return f"{pd.Timestamp(y0, BASE_START.month, 1):%b %Y} → {pd.Timestamp(y0 + 1, BASE_START.month, 1) - pd.Timedelta(days=1):%b %Y}"


CARBON_ROWS = []
for p in CAMPUS_TOTAL["electricity"]:
    s = DAILY[p].dropna()
    for per, g in s.groupby(s.index.map(_period)):
        days_in = len(pd.date_range(*[pd.Timestamp(x) for x in (g.index.min(), g.index.max())]))
        CARBON_ROWS.append({"period": per, "scope": "Scope 2 · electricity", "source": METER_INFO[p][2],
                            "amount": float(g.sum()), "unit": "kWh", "t CO2e": float(g.sum()) * ELEC_KG_CO2_PER_KWH / 1000,
                            "data": f"{len(g)} good days"})
for p in CAMPUS_TOTAL["gas"]:
    s = DAILY[p].dropna()
    for per, g in s.groupby(s.index.map(_period)):
        co2 = co2_kg(p, float(g.sum()))
        CARBON_ROWS.append({"period": per, "scope": "Scope 1 · natural gas", "source": METER_INFO[p][2],
                            "amount": float(g.sum()), "unit": "meter units (documented m³)",
                            "t CO2e": co2 / 1000 if co2 is not None else np.nan,
                            "data": f"{len(g)} good days" + ("" if co2 is not None else " · unit not confirmed → CO₂ not computed")})
_lw = WASTE[(WASTE.Waste_Type == "Landfill") & WASTE.tonnes.notna()]
for per, g in _lw.groupby(_lw.Date.map(_period)):
    CARBON_ROWS.append({"period": per, "scope": "Scope 3 · waste to landfill", "source": "hauling invoices",
                        "amount": float(g.tonnes.sum()), "unit": "t", "t CO2e": float(g.tonnes.sum()) * LANDFILL_T_CO2E_PER_T,
                        "data": f"{len(g)} weighed pickups"})
if EWASTE is not None and EWASTE["leaving campus"].any():
    kg = float(EWASTE.loc[EWASTE["leaving campus"], "weight_kg"].sum())
    CARBON_ROWS.append({"period": "register", "scope": "Scope 3 · e-waste", "source": "asset register",
                        "amount": kg, "unit": "kg", "t CO2e": kg / 1000 * EWASTE_EOL_T_CO2E_PER_T, "data": "end-of-life only"})
CARBON = pd.DataFrame(CARBON_ROWS)
CARBON_BY_PERIOD = CARBON.pivot_table(index="period", columns="scope", values="t CO2e",
                                      aggfunc=lambda s: s.sum(min_count=1)).round(1)
INDIA_GRID_KG_PER_KWH = 0.710   # CEA CO2 Baseline Database v21.0 (India) — for comparison only

def avoided_table():
    ty = ACTIONS.set_index("action_id")["type"]
    return pd.DataFrame([{"action": k, "type": "ground-truth test" if "Ground-truth" in str(ty.get(k, "")) else "action",
                          "meter": METER_INFO[v["point"]][2], "counted": v.get("counted", 0), "unit": v["unit"],
                          "kg CO2e avoided": v.get("counted_co2_kg") or 0.0, "verdict": v["verdict"]} for k, v in VERIFIED.items()])


def refresh_ewaste_carbon():
    """Add / update the e-waste line after a register is loaded in the app."""
    global CARBON, CARBON_BY_PERIOD
    CARBON = CARBON[CARBON.scope != "Scope 3 · e-waste"]
    if EWASTE is not None and EWASTE["leaving campus"].any() and EWASTE.loc[EWASTE["leaving campus"], "weight_kg"].notna().any():
        kg = float(EWASTE.loc[EWASTE["leaving campus"], "weight_kg"].sum())
        CARBON = pd.concat([CARBON, pd.DataFrame([{"period": "e-waste register", "scope": "Scope 3 · e-waste", "source": "asset register",
                                                   "amount": kg, "unit": "kg", "t CO2e": kg / 1000 * EWASTE_EOL_T_CO2E_PER_T,
                                                   "data": "end-of-life only (EPA)"}])], ignore_index=True)
    CARBON_BY_PERIOD = CARBON.pivot_table(index="period", columns="scope", values="t CO2e",
                                          aggfunc=lambda s: s.sum(min_count=1)).round(1)


AVOIDED = avoided_table()
CARBON_FACTORS = pd.DataFrame([
    ("Electricity (Scope 2)", f"{ELEC_KG_CO2_PER_KWH * 1000:.1f} kg CO₂ / MWh", "Hydro-Québec — CO2 emissions and Hydro-Québec electricity 1990–2020"),
    ("India grid (comparison only)", f"{INDIA_GRID_KG_PER_KWH} kg CO₂ / kWh", "CEA CO2 Baseline Database v21.0"),
    ("Natural gas (Scope 1)", f"{GAS_KG_CO2_PER_M3} kg CO₂ / m³ (used once meter unit is confirmed)", "Québec MELCC GHG quantification guide, Table 5"),
    ("Landfilled mixed waste (Scope 3)", f"{LANDFILL_T_CO2E_PER_T:.3f} t CO₂e / t", "US EPA GHG Emission Factors Hub 2024 (0.58 t / short ton)"),
    ("E-waste, recycled (Scope 3)", f"{EWASTE_EOL_T_CO2E_PER_T:.3f} t CO₂e / t", "US EPA GHG Emission Factors Hub 2024 (0.02 t / short ton)"),
], columns=["What", "Factor", "Source"])
print("t CO₂e by 12-month period (blank = not measured / not computable):")
print(CARBON_BY_PERIOD.to_string())
_e = CARBON[(CARBON.scope.str.contains("electricity")) & (CARBON.period == _period(BASE_START))].amount.sum()
print(f"\nThe surprise for judges: Québec's grid is almost all hydro, so {_e / 1e6:,.1f} GWh of metered electricity in the "
      f"baseline year is only {_e * ELEC_KG_CO2_PER_KWH / 1000:,.0f} t CO₂ — landfill waste is the biggest measured source. "
      f"On India's grid (CEA, {INDIA_GRID_KG_PER_KWH} kg/kWh) the same electricity would be {_e * INDIA_GRID_KG_PER_KWH / 1000:,.0f} t CO₂.")

# ---- Gas sanity check: the documented unit vs everything else we measured ----
FT3_TO_M3 = 0.0283168                       # exact unit conversion (1 cubic foot in m³)
_gb = sum(float(DAILY[p].loc[BASE_START:BASE_END].sum()) for p in CAMPUS_TOTAL["gas"])
GAS_DOC_T = _gb * GAS_KG_CO2_PER_M3 / 1000             # if 1 meter unit = 1 m³ (as documented)
GAS_FT3_T = _gb * FT3_TO_M3 * GAS_KG_CO2_PER_M3 / 1000   # if the meter were actually in cubic feet
_land = float(_cb_land := CARBON[(CARBON.period == _period(BASE_START)) & CARBON.scope.str.contains("landfill")]["t CO2e"].sum())
print(f"\nGas, baseline year: {_gb:,.0f} meter units. As documented (m³) → {GAS_DOC_T:,.0f} t CO₂; "
      f"even if the unit were cubic feet → {GAS_FT3_T:,.0f} t CO₂. Either way gas is the biggest source "
      f"(landfill {_land:,.0f} t, electricity {_e * ELEC_KG_CO2_PER_KWH / 1000:,.0f} t) — so gas is the lever, and its unit "
      f"must be confirmed before a tonne figure is published.")


def carbon_by_period(include_gas_documented=False):
    C = CARBON.copy()
    if include_gas_documented and not GAS_M3_PER_METER_UNIT:
        g = C.scope.str.contains("gas")
        C.loc[g, "t CO2e"] = C.loc[g, "amount"] * GAS_KG_CO2_PER_M3 / 1000
    return C.pivot_table(index="period", columns="scope", values="t CO2e", aggfunc=lambda s: s.sum(min_count=1)).round(1)


## Cell 11 · The six sentences for your pitch
All computed above from real data. Read these out on stage.

In [ ]:
# ---- The numbers to say to the judges (all computed above from real data) ----
PARAMETERS = pd.DataFrame([
    ("Data", "Concordia University open data (Facilities), CC BY 4.0", "Real meters, real invoices — nothing simulated"),
    ("Day counts if", "≥ 90% of its hours recorded", "Common data-completeness rule; days below it are listed, not filled in"),
    ("Baseline", "12 months before (every season once)", "IPMVP / ASHRAE practice: a full annual cycle"),
    ("Expected-use model", "ASHRAE change-point model (temperature + weekday/holiday)", "ASHRAE Guideline 14 / Inverse Modeling Toolkit"),
    ("Balance temperatures", "found from the data", "every whole °C observed is tried; best by BIC"),
    ("Model accuracy grade", "Good ≤ 15% / OK ≤ 30% error; bias ≤ ±5% / ±10%", "ASHRAE Guideline 14 calibration limits (monthly / hourly data)"),
    ("Measurement window", "4, 6 or 8 whole weeks", "whole weeks = same weekday mix before and after"),
    ("Ghost twin", "the meter's better model (change-point or gradient boosting)", "picked per meter by its typical miss on 12 hidden months"),
    ("Gradient boosting", "LightGBM, library defaults", "no tuning — nothing to cherry-pick"),
    ("Placebo-in-time", "fake action every week of the year before", "same weekday mix for every fake date"),
    ("95% range", "t × typical fake saving × √(1+1/k)", "standard prediction interval; k = independent stretches in a year"),
    ("Significance", "p ≤ 5% (permutation test)", "statistical convention; VERIFIED needs range AND p"),
    ("Counted saving", "low end of the 95% range", "never overstate — the brief's requirement"),
    ("Alert bar", "biggest out-of-sample surprise of the baseline year", "each meter sets its own bar; no tuned thresholds"),
    ("Holidays", "Québec public holidays", "official calendar (python `holidays`)"),
    ("Electricity CO₂", "0.5 kg CO₂ / MWh", "Hydro-Québec, CO2 emissions and Hydro-Québec electricity 1990–2020 (2019 & 2020)"),
    ("Natural gas CO₂", "1.926 kg CO₂ / m³ — used only once the meter unit is confirmed",
     "Québec MELCC GHG quantification guide, Table 5"),
    ("Landfill CO₂e", "0.639 t CO₂e per tonne", "US EPA GHG Emission Factors Hub 2024: 0.58 t per short ton, mixed MSW landfilled"),
    ("E-waste CO₂e", "0.022 t CO₂e per tonne (end of life)", "US EPA GHG Emission Factors Hub 2024: 0.02 t per short ton, mixed electronics recycled"),
    ("E-waste routes", "keep / registered refurbisher / registered recycler", "India E-Waste (Management) Rules, 2022 (bulk consumer duties)"),
    ("E-waste expected life", "from the CPCB EPR portal, per EEE code", "entered with each device — never assumed"),
    ("Room vacancy switch-off", "20 minutes", "ASHRAE 90.1 occupancy-sensor rule (changeable in the app)"),
    ("Room comfort band", "80% acceptability, 7-day mean outdoor temperature", "ASHRAE 55 adaptive model via pythermalcomfort"),
    ("Fan credit", "0.6 m/s → +1.2 °C", "ASHRAE 55 elevated-air-speed step"),
    ("People counting", "YOLOv8n (COCO), default confidence", "Ultralytics; class 'person' only"),
    ("Waste diversion", "(compost + paper + mixed recycling) ÷ (those + landfill)", "bulky waste excluded: Concordia says its split is unknown"),
], columns=["What", "Value", "Why / source"])

_ok = {k: v for k, v in VERIFIED.items() if "saving_%" in v}
_ver = [v for v in _ok.values() if v["verdict"] == "VERIFIED SAVING"]
_gas = [v for v in _ok.values() if METER_INFO[v["point"]][1] == "gas"]
_top_e = RECS[(RECS.area == "Electricity") & RECS.priority.str.startswith("2")].head(1)
HEADLINE = []
HEADLINE.append(f"1. One real campus, every resource: {DAILY.shape[1]} meters (electricity, gas, water) + {len(WASTE)} waste invoices, "
                f"{HOURLY.index.min():%b %Y}–{HOURLY.index.max():%b %Y} — plus a live camera for room-level control.")
HEADLINE.append(f"2. It optimises: {len(RECS)} data-driven recommendations, each sized from the building's own data"
                + (f" — e.g. {_top_e.iloc[0]['where']}: {_top_e.iloc[0]['finding']} ({_top_e.iloc[0]['potential (not verified)']})." if len(_top_e) else "."))
HEADLINE.append(f"3. Trust test on {TRUST_SUMMARY['tests']} normal months (nothing done): a before/after dashboard showed a 'saving' "
                f"{TRUST_SUMMARY['naive_saving']} times; GreenLedger certified {TRUST_SUMMARY['gl_saving']}.")
if _gas:
    g = max(_gas, key=lambda v: v["naive_%"])
    HEADLINE.append(f"4. Not overstated: a before/after dashboard says gas fell {g['naive_%']:.0f}% after March 2020; the ghost twin "
                    f"(real weather, real calendar) says {g['saving_%']:.0f}% and the placebo test says that is normal noise "
                    f"(p = {g['p']:.2f}). Nothing counted.")
if len(DISCOVERY_TABLE) and _ver:
    first = DISCOVERY_TABLE.sort_values("first seen").iloc[0]
    best = max(_ver, key=lambda v: v["saving_%"])
    HEADLINE.append(f"5. Ground-truth test: nobody told GreenLedger about the COVID-19 closure — it flagged the drop on "
                    f"{pd.Timestamp(first['first seen']):%d %b %Y}, and the ghost twin measured {METER_INFO[best['point']][2]} "
                    f"electricity {best['saving_%']:.0f}% below normal (p = {best['p']:.3f}). The same engine verifies real actions.")
_cb = CARBON[CARBON.period == _period(BASE_START)]
HEADLINE.append(f"6. Carbon, honestly: in the baseline year landfill waste ≈ {_cb[_cb.scope.str.contains('landfill')]['t CO2e'].sum():,.0f} t CO₂e "
                f"vs metered electricity ≈ {_cb[_cb.scope.str.contains('electricity')]['t CO2e'].sum():,.0f} t (Québec hydro). Gas CO₂ is "
                f"withheld until the meter unit is confirmed; {len(FAULTS)} bad meter-days were excluded with reasons.")
print("\n".join(HEADLINE))


## Cell 12 · Export the report
HTML report + ZIP "labelled dataset" (daily readings with ghost-twin value, fault reason and alert id; alerts; actions; verification; recommendations & decisions; carbon; waste; e-waste; room log; sources). Files are in the 📁 panel under `greenledger_out`.

In [ ]:
# ---- Exportable report + labelled dataset (for the judges / facilities team) ----
def labelled_dataset():
    rows = []
    for p in DAILY:
        f = FAULTS[FAULTS.point == p].set_index("day").rule
        df = pd.DataFrame({"date": DAYS, "meter": METER_INFO[p][2], "resource": METER_INFO[p][1],
                           "system_name": METER_INFO[p][0], "actual": DAILY[p].values,
                           "expected": EXPECTED[p].reindex(DAYS).values, "temp_C": TEMP.values,
                           "off_day": OFF_DAY.values, "period": np.where(DAYS <= BASE_END, "baseline", "monitoring")})
        df["data_fault"] = df.date.map(f).fillna("")
        df["alert"] = ""
        for _, a in ALERT_TABLE[ALERT_TABLE.point == p].iterrows():
            df.loc[(df.date >= pd.Timestamp(a["from"])) & (df.date <= pd.Timestamp(a["to"])), "alert"] = a["id"]
        rows.append(df)
    return pd.concat(rows, ignore_index=True)


def verification_table():
    out = []
    for _, a in ACTIONS.iterrows():
        r = VERIFIED.get(a.action_id) or verify(a.point, a.date, 42, ACTIONS[ACTIONS.action_id != a.action_id])
        out.append({"action": a.action_id, "date": a.date.date(), "meter": METER_INFO[a.point][2], "title": a.title,
                    "verdict": r["verdict"], "change %": round(r.get("saving_%", np.nan), 1), "p": round(r.get("p", np.nan), 3),
                    "95% low": round(r.get("low_%", np.nan), 1), "95% high": round(r.get("high_%", np.nan), 1),
                    "naive %": round(r.get("naive_%", np.nan), 1), "counted": round(r.get("counted", 0), 0),
                    "unit": r["unit"], "explanation": explain(r)})
    return pd.DataFrame(out)


def build_report():
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    zpath = os.path.join(OUT_DIR, f"greenledger_concordia_{stamp}.zip")
    V = verification_table()
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("readings_daily_labelled.csv", labelled_dataset().to_csv(index=False))
        z.writestr("alerts.csv", ALERT_TABLE.drop(columns="point").to_csv(index=False))
        z.writestr("discoveries.csv", DISCOVERY_TABLE.drop(columns="point").to_csv(index=False))
        z.writestr("data_faults.csv", FAULTS.to_csv(index=False))
        z.writestr("actions.csv", ACTIONS.to_csv(index=False))
        z.writestr("verification.csv", V.to_csv(index=False))
        z.writestr("model_accuracy.csv", ACC.drop(columns="point").to_csv(index=False))
        z.writestr("trust_test.csv", TRUST.to_csv(index=False))
        z.writestr("waste_monthly_tonnes.csv", WASTE_MONTHLY.to_csv())
        z.writestr("waste_yearly.csv", WASTE_YEARLY.to_csv())
        z.writestr("parameters_and_sources.csv", PARAMETERS.to_csv(index=False))
        z.writestr("recommendations.csv", RECS.drop(columns=["_value", "_point"]).to_csv(index=False))
        z.writestr("recommendation_decisions.csv", DECISIONS.to_csv(index=False))
        z.writestr("carbon_inventory.csv", CARBON.to_csv(index=False))
        z.writestr("carbon_factors.csv", CARBON_FACTORS.to_csv(index=False))
        z.writestr("model_choice.csv", MODEL_CHECK.drop(columns="point").to_csv(index=False))
        z.writestr("room_control_log.csv", pd.DataFrame(ROOM["log"] if "ROOM" in globals() else []).to_csv(index=False))
        if EWASTE is not None:
            z.writestr("ewaste_plan.csv", EWASTE.to_csv(index=False))
        z.writestr("README.txt", "GreenLedger — reference campus: Concordia University (Montréal).\n"
                   "Source: Concordia University open data (Facilities), CC BY 4.0. Nothing simulated or filled in.\n"
                   "readings_daily_labelled.csv: one row per meter per day — actual, expected, data_fault reason, alert id.\n")
    tbl = lambda d: d.to_html(index=False, border=0, classes="t")
    html = f"""<html><head><meta charset="utf-8"><title>GreenLedger report</title><style>
body{{font-family:Inter,Arial,sans-serif;max-width:1000px;margin:30px auto;color:#1f2937;line-height:1.5}}
h1{{color:#047857}} .t{{border-collapse:collapse;width:100%;font-size:13px}} .t td,.t th{{border-bottom:1px solid #e5e7eb;padding:6px;text-align:left}}
.t th{{background:#f3f4f6}} li{{margin:6px 0}}</style></head><body>
<h1>GreenLedger — campus resource report</h1>
<p>Reference campus: <b>Concordia University, Montréal</b> · Concordia open data (CC BY 4.0) · generated {datetime.datetime.now():%d %b %Y %H:%M}</p>
<h2>Headlines</h2><ul>{''.join(f'<li>{h}</li>' for h in HEADLINE)}</ul>
<h2>Recommendations (potential — verified only after they are done)</h2>{tbl(RECS[['id', 'priority', 'area', 'where', 'finding', 'potential (not verified)', 'next step']])}
<h2>Carbon inventory (t CO₂e; blank = not measured / not computable)</h2>{CARBON_BY_PERIOD.reset_index().to_html(index=False, border=0, classes='t', na_rep='—')}
{tbl(CARBON_FACTORS)}
<h2>Did it work? (action register — ghost twin + placebo-in-time)</h2>{tbl(V[['action', 'date', 'meter', 'title', 'verdict', 'change %', '95% low', '95% high', 'p', 'naive %', 'counted', 'unit']])}
<h2>Alerts</h2>{tbl(ALERT_TABLE[['id', 'meter', 'from', 'to', 'extra use', 'what happened']])}
<h2>Discoveries (unexplained sustained drops)</h2>{tbl(DISCOVERY_TABLE.drop(columns='point'))}
<h2>Waste per 12-month period (tonnes)</h2>{WASTE_YEARLY.to_html(border=0, classes='t')}
<h2>Model accuracy</h2>{tbl(ACC.drop(columns='point'))}
<h2>Which model is the ghost twin?</h2>{tbl(MODEL_CHECK.drop(columns='point'))}
{'<h2>E-waste plan</h2>' + tbl(EWASTE[['device_id', 'device_type', 'department', 'age_years', 'expected_life_years', 'condition', 'route', 'why']]) if EWASTE is not None else ''}
{'<h2>Room control log</h2>' + tbl(pd.DataFrame(ROOM['log'])) if 'ROOM' in globals() and ROOM['log'] else ''}
<h2>Parameters and sources</h2>{tbl(PARAMETERS)}
</body></html>"""
    hpath = os.path.join(OUT_DIR, f"greenledger_report_{stamp}.html")
    open(hpath, "w").write(html)
    return hpath, zpath


REPORT_HTML, REPORT_ZIP = build_report()
print("Report:", REPORT_HTML)
print("Labelled dataset (ZIP):", REPORT_ZIP)


## Cell 13 · 🎥 Live room — camera → people → fan / lights / AC, set by the live weather
* **People** counted on every camera frame with **YOLOv8n** (median of the last 3 frames, so one bad frame can't flip a switch). A test photo counts as one steady scene.
* **Automatic settings from the live weather** (Open-Meteo: temperature, feels-like, humidity, 7-day mean — refreshed every 10 min) through the **ASHRAE 55** adaptive comfort model:
  * above the comfort-neutral temperature → **fan ON**
  * too hot even with a fan → **AC ON**, set to the comfort-neutral temperature (≥ the BEE default of 24 °C), plus fan
  * **many people** (default 10, editable) → **fan ON** even when it's mild — each seated person gives off ≈ 70 W of heat (ASHRAE Handbook—Fundamentals)
  * room empty for 20 min (ASHRAE 90.1; 30 s in demo mode) → **everything OFF**
* **Manual option:** switch the rules to *Manual* and type your own fan / AC temperatures, AC set-point and crowd size — or tap any device to switch it by hand (it stays manual until you press *back to auto*).
* The temperature used is the room temperature if you type one (thermometer / AC display), otherwise the live outdoor temperature — the screen always says which.
* **Live twin** = the room as GreenLedger runs it · **Ghost twin** = the same room where people switch things on but nobody switches them off.
  Saving = ghost − live over time × (how many × watts each) — an estimate for one room; campus savings are verified on meters.
* **Switching:** smart plug / Home Assistant webhook (`GL_DEVICE_WEBHOOK`) if set; otherwise a **virtual switch** that shows exactly what would switch.

In [ ]:
# ---- 🎥 Live room: camera → people → fan / AC / lights, with a LIVE TWIN vs GHOST TWIN of the room ----
# • People: YOLOv8n on every camera frame; the count is the median of the last 3 frames (removes flicker).
# • Settings come AUTOMATICALLY from the live weather (Open-Meteo, refreshed every 10 min) through the
#   ASHRAE 55 adaptive comfort model (pythermalcomfort) — or the user types their own (MANUAL mode):
#       temperature above the neutral point ...... FAN ON
#       too warm even with a fan (0.6 m/s) ....... AC ON, set to the comfort-neutral temperature, + fan
#       colder than the comfort band ............. fan and AC OFF
#   Crowd: every seated person gives off ~70 W of heat (ASHRAE Handbook—Fundamentals, ch. 18), so when the
#   camera counts "many people" (default 10 ≈ a 700 W heater, editable) the FAN goes ON even at a mild temperature.
#   Temperature used: the room temperature if typed (thermometer / AC display), otherwise the LIVE OUTDOOR one —
#   the screen always says which.
# • Any device can be switched by hand; it then stays on MANUAL until "back to auto" is pressed.
# • Empty for the vacancy time (ASHRAE 90.1: 20 min; shorter in demo mode) → everything off.
# • LIVE TWIN  = the room as GreenLedger runs it.
#   GHOST TWIN = the same room without GreenLedger: devices go on when people need them, but nobody
#                switches them off when the room empties (the case occupancy controls exist for).
#   Saving = ghost − live, integrated over time, using each device's nameplate kW (from its label).
#   This is an ESTIMATE for one room; campus savings are verified on real meters in "Act & Prove".
# • Switching: a smart plug / Home Assistant webhook (GL_DEVICE_WEBHOOK) if configured; otherwise a VIRTUAL
#   switch — the dashboard shows exactly what would be switched, and says so.
import threading as _th, urllib.parse, collections, shutil
try:
    from ultralytics import YOLO
    YOLO_PATH = os.path.join(OUT_DIR, "yolov8n.pt")           # saved now → no download needed during the demo
    if not os.path.exists(YOLO_PATH):
        _m = YOLO("yolov8n.pt")
        shutil.copy(getattr(_m, "ckpt_path", None) or "yolov8n.pt", YOLO_PATH)
    YOLO_MODEL = YOLO(YOLO_PATH)
except Exception as e:
    YOLO_MODEL = None
    print("Camera detection unavailable:", e)
try:
    from pythermalcomfort.models import adaptive_ashrae
except Exception as e:
    adaptive_ashrae = None
    print("pythermalcomfort unavailable:", e)

VACANCY_MIN_DEFAULT = 20      # ASHRAE 90.1 occupancy-sensor rule (minutes)
DEMO_VACANCY_MIN = 0.5        # demo mode: 30 s, so judges can watch it switch off
CONF_DEFAULT = 0.5            # detection confidence shown on screen and adjustable (Ultralytics' own default is 0.25)
FAN_AIR_SPEED = 0.6           # m/s — ASHRAE 55 first elevated-air-speed step (+1.2 °C)
STILL_AIR = 0.1               # m/s — still air in ASHRAE 55
BEE_DEFAULT_AC_C = 24         # India BEE default AC setting, for comparison
DEVICES = ("Fan", "Lights", "AC")
DEVICE_WEBHOOK = os.environ.get("GL_DEVICE_WEBHOOK", "")
ROOM_LOG_CSV = os.path.join(OUT_DIR, "room_control_log.csv")
WEATHER = {"city": None, "query": None, "now": None, "feels": None, "humidity": None, "trm": None, "fetched": None, "error": None}
WEATHER_REFRESH_MIN = 10       # live weather is re-read this often while the room page is running
CROWD_DEFAULT = 10            # "many people" → fan on (≈ 10 × 70 W = 700 W of body heat); editable in the app
PERSON_HEAT_W = 70            # ASHRAE Handbook—Fundamentals ch. 18, seated very light work, sensible heat
DEVICE_DEFAULTS = {"Fan": (4, 75), "Lights": (8, 20), "AC": (1, 1500)}   # (how many, watts each) — ceiling fan, LED tube, 1.5 t split AC


def _new_room():
    return {"lock": _th.Lock(), "last_person": None, "people": 0, "raw": 0, "counts": collections.deque(maxlen=3),
            "live": {d: False for d in DEVICES}, "ghost": {d: False for d in DEVICES}, "setpoint": None,
            "history": [], "log": [], "temp_source": "", "manual": {d: False for d in DEVICES}, "thr": None}


ROOM = _new_room()


def fetch_weather(city):
    """Current outdoor temperature + 7-day mean (ASHRAE 55 prevailing mean outdoor temperature), Open-Meteo."""
    try:
        g = json.loads(urllib.request.urlopen(
            "https://geocoding-api.open-meteo.com/v1/search?count=1&name=" + urllib.parse.quote(city), timeout=8).read())
        r0 = g["results"][0]
        w = json.loads(urllib.request.urlopen(
            f"https://api.open-meteo.com/v1/forecast?latitude={r0['latitude']}&longitude={r0['longitude']}"
            "&current=temperature_2m,relative_humidity_2m,apparent_temperature&daily=temperature_2m_mean"
            "&past_days=7&forecast_days=1&timezone=auto", timeout=8).read())
        past = [t for t in w["daily"]["temperature_2m_mean"][:7] if t is not None]
        cur = w["current"]
        WEATHER.update(city=f"{r0['name']}, {r0.get('country', '')}", query=city, now=float(cur["temperature_2m"]),
                       feels=cur.get("apparent_temperature"), humidity=cur.get("relative_humidity_2m"),
                       trm=float(np.mean(past)), fetched=datetime.datetime.now(), error=None)
    except Exception as e:
        WEATHER.update(error=str(e), query=city, fetched=datetime.datetime.now())
    return WEATHER


def weather_if_stale(city):
    """Re-read the live weather in the background when it is older than WEATHER_REFRESH_MIN or the city changed."""
    city = (city or "").strip()
    if not city:
        return
    old = WEATHER["fetched"] is None or (datetime.datetime.now() - WEATHER["fetched"]).total_seconds() > WEATHER_REFRESH_MIN * 60
    if old or city.lower() != str(WEATHER["query"] or "").lower():
        WEATHER["fetched"] = datetime.datetime.now()                 # stops repeated calls while this one runs
        WEATHER["query"] = city
        _th.Thread(target=fetch_weather, args=(city,), daemon=True).start()


def comfort(trm):
    """ASHRAE 55 adaptive: (lower, neutral, upper at still air, upper with a fan) for 80% acceptability."""
    r = adaptive_ashrae(tdb=25, tr=25, t_running_mean=trm, v=STILL_AIR)
    rf = adaptive_ashrae(tdb=25, tr=25, t_running_mean=trm, v=FAN_AIR_SPEED)
    g = (lambda o, k: getattr(o, k)) if not isinstance(r, dict) else (lambda o, k: o[k])
    return float(g(r, "tmp_cmf_80_low")), float(g(r, "tmp_cmf")), float(g(r, "tmp_cmf_80_up")), float(g(rf, "tmp_cmf_80_up"))


def thresholds(manual=None, outdoor_manual=None):
    """Switching temperatures. AUTO: from this week's live weather via ASHRAE 55. MANUAL: the user's own numbers."""
    trm = WEATHER["trm"] if WEATHER["trm"] is not None else outdoor_manual
    auto = None
    if trm is not None and adaptive_ashrae is not None:
        lo, neutral, hi, hi_fan = comfort(trm)
        auto = {"cold": lo, "fan_on": neutral, "ac_on": hi_fan, "ac_set": float(max(BEE_DEFAULT_AC_C, round(neutral))), "trm": trm}
    m = manual or {}
    if m.get("mode") == "manual":
        base = auto or {"cold": None, "fan_on": None, "ac_on": None, "ac_set": None, "trm": trm}
        t = {k: (m.get(k) if m.get(k) is not None else base[k]) for k in ("fan_on", "ac_on", "ac_set")}
        t.update(cold=base["cold"], trm=trm, source="manual (your settings)")
    elif auto:
        t = {**auto, "source": f"automatic — live weather, ASHRAE 55 (7-day mean {trm:.1f} °C)"}
    else:
        t = {"cold": None, "fan_on": None, "ac_on": None, "ac_set": None, "trm": None,
             "source": "no weather yet — type a city, or switch to manual and set the temperatures"}
    t["crowd"] = int(m.get("crowd") or CROWD_DEFAULT)
    return t


def decide(people, vacant_min, room_temp, thr, vacancy_limit):
    if people == 0:
        if vacant_min is not None and vacant_min >= vacancy_limit:
            return ({d: False for d in DEVICES}, None, f"Room empty for {vacant_min:.1f} min (≥ {vacancy_limit:g} min) → everything OFF.")
        left = max(0.0, vacancy_limit - (vacant_min or 0))
        return (None, None, f"No one in view — switching off in {left * 60:.0f} s unless someone comes back.")
    crowd = people >= thr["crowd"]
    heat = f" ({people} × {PERSON_HEAT_W} W ≈ {people * PERSON_HEAT_W / 1000:.1f} kW of body heat)" if crowd else ""
    if room_temp is None or thr["fan_on"] is None or thr["ac_on"] is None:
        return ({"Lights": True, "Fan": True if crowd else None, "AC": None}, None,
                f"{people} people → lights ON" + (f", crowded{heat} → FAN ON" if crowd else "") +
                ". No temperature yet for the AC decision — load the weather or type the room temperature.")
    if thr["cold"] is not None and room_temp < thr["cold"]:
        return ({"Lights": True, "Fan": False, "AC": False}, None,
                f"{people} present · {room_temp:.1f} °C is cooler than comfortable (below {thr['cold']:.1f} °C) → fan and AC OFF.")
    if room_temp > thr["ac_on"]:
        sp = thr["ac_set"]
        return ({"Lights": True, "Fan": True, "AC": True}, sp,
                f"{people} present · {room_temp:.1f} °C is above {thr['ac_on']:.1f} °C — too hot even with a fan → "
                f"AC ON at {sp:.0f} °C (the comfort-neutral point for this weather, never below the BEE default of {BEE_DEFAULT_AC_C} °C) + fan.")
    if room_temp > thr["fan_on"]:
        return ({"Lights": True, "Fan": True, "AC": False}, None,
                f"{people} present · {room_temp:.1f} °C is above {thr['fan_on']:.1f} °C; a fan keeps it comfortable up to "
                f"{thr['ac_on']:.1f} °C → FAN ON, AC OFF.")
    if crowd:
        return ({"Lights": True, "Fan": True, "AC": False}, None,
                f"{people} people — crowded{heat}, so FAN ON even though {room_temp:.1f} °C is mild; AC OFF.")
    return ({"Lights": True, "Fan": False, "AC": False}, None,
            f"{people} present · {room_temp:.1f} °C is comfortable (fan from {thr['fan_on']:.1f} °C, "
            f"or with {thr['crowd']}+ people) → no fan, no AC.")


def send_command(device, on, setpoint=None):
    if not DEVICE_WEBHOOK:
        return "virtual switch (no smart plug connected)"
    try:
        req = urllib.request.Request(DEVICE_WEBHOOK, method="POST", headers={"Content-Type": "application/json"},
                                     data=json.dumps({"device": device, "on": bool(on), "setpoint": setpoint}).encode())
        urllib.request.urlopen(req, timeout=5)
        return "sent to smart switch"
    except Exception as e:
        return f"switch error: {e}"


def _log(now, dev, action, reason, how):
    ROOM["log"].append({"time": now.strftime("%H:%M:%S"), "device": dev, "action": action,
                        "people": ROOM["people"], "reason": reason, "how": how})
    pd.DataFrame(ROOM["log"]).to_csv(ROOM_LOG_CSV, index=False)


def apply(want, setpoint, reason, auto, now):
    for dev in DEVICES:
        w = want.get(dev)
        if w is None:
            continue
        if w:
            ROOM["ghost"][dev] = True                  # without GreenLedger, people still switch things ON
        if ROOM["manual"][dev]:                        # switched by hand → GreenLedger leaves it alone
            continue
        changed = ROOM["live"][dev] != w or (dev == "AC" and w and setpoint != ROOM["setpoint"])
        if not changed:
            continue
        if auto:
            how = send_command(dev, w, setpoint if dev == "AC" else None)
            ROOM["live"][dev] = w
            if dev == "AC":
                ROOM["setpoint"] = setpoint if w else None
            _log(now, dev, ("ON" + (f" @ {setpoint:.0f} °C" if dev == "AC" and setpoint else "")) if w else "OFF", reason, how)
        elif not ROOM["log"] or ROOM["log"][-1]["reason"] != reason:
            _log(now, dev, "suggest " + ("ON" if w else "OFF"), reason, "auto-control is off")


def room_kw(states, kw):
    return sum(float(kw.get(d) if kw.get(d) is not None else DEVICE_DEFAULTS[d][0] * DEVICE_DEFAULTS[d][1] / 1000)
               for d in DEVICES if states[d])


def _pixelate(img, boxes, block=12):
    """Privacy: pixelate every detected person before the preview leaves the server (nobody is identifiable on screen)."""
    out = img.copy()
    h, w = out.shape[:2]
    for x1, y1, x2, y2 in boxes:
        x1, y1, x2, y2 = max(0, int(x1)), max(0, int(y1)), min(w, int(x2)), min(h, int(y2))
        if x2 - x1 < 2 or y2 - y1 < 2:
            continue
        import cv2
        roi = out[y1:y2, x1:x2]
        small = cv2.resize(roi, (max(1, (x2 - x1) // block), max(1, (y2 - y1) // block)), interpolation=cv2.INTER_AREA)
        out[y1:y2, x1:x2] = cv2.resize(small, (x2 - x1, y2 - y1), interpolation=cv2.INTER_NEAREST)
    return out


def process_frame(frame, room_temp=None, auto=True, vacancy_limit=VACANCY_MIN_DEFAULT, outdoor_manual=None,
                  conf=CONF_DEFAULT, kw=None, manual=None, still=False, privacy=True):
    """Frames are processed in memory only: nothing is written to disk — only the people COUNT is kept."""
    """One camera frame → annotated frame + decision; updates the live and ghost twins of the room."""
    now = datetime.datetime.now()
    kw = kw or {}
    annotated, raw = frame, 0
    if frame is not None and YOLO_MODEL is not None:
        bgr = np.ascontiguousarray(np.asarray(frame)[:, :, :3][:, :, ::-1])     # camera gives RGB; YOLO expects BGR
        r = YOLO_MODEL(bgr, classes=[0], conf=float(conf), verbose=False)[0]   # class 0 = person
        raw = int(len(r.boxes))
        base = _pixelate(bgr, r.boxes.xyxy.cpu().numpy()) if privacy and raw else bgr
        annotated = np.ascontiguousarray(r.plot(img=base)[:, :, ::-1])
    with ROOM["lock"]:
        if still:                                       # a test photo is one steady scene → no smoothing against old frames
            ROOM["counts"].clear()
        ROOM["counts"].append(raw)
        people = int(np.median(ROOM["counts"]))
        ROOM["people"], ROOM["raw"] = people, raw
        if people > 0:
            ROOM["last_person"] = now
        vacant = vacancy_limit if ROOM["last_person"] is None else (now - ROOM["last_person"]).total_seconds() / 60
        if room_temp is not None:
            rt, ROOM["temp_source"] = room_temp, "room thermometer"
        elif WEATHER["now"] is not None:
            rt, ROOM["temp_source"] = WEATHER["now"], "live OUTDOOR temperature (no room temperature typed)"
        else:
            rt, ROOM["temp_source"] = None, "none"
        thr = thresholds(manual, outdoor_manual)
        ROOM["thr"] = thr
        want, sp, reason = decide(people, vacant, rt, thr, float(vacancy_limit))
        if want and people == 0 and not any(want.values()):
            ROOM["manual"] = {d: False for d in DEVICES}   # empty room: switch off everything, hand-set devices too
        if want:
            apply(want, sp, reason, auto, now)
        ROOM["history"].append({"time": now, "people": people, "live_kw": room_kw(ROOM["live"], kw),
                                "ghost_kw": room_kw(ROOM["ghost"], kw),
                                "live_on": sum(ROOM["live"].values()), "ghost_on": sum(ROOM["ghost"].values())})
        ROOM["history"] = ROOM["history"][-1800:]          # last hour at one frame every 2 s
    return annotated, people, raw, reason


def room_savings():
    """Integrate ghost − live over the session: kWh (if nameplate kW given) and device-minutes switched off."""
    H = pd.DataFrame(ROOM["history"])
    if len(H) < 2:
        return 0.0, 0.0, H
    dt = H.time.diff().dt.total_seconds().fillna(0).clip(upper=60) / 3600       # hours; gaps > 1 min not counted
    kwh = float(((H.ghost_kw - H.live_kw).shift(1).fillna(0) * dt).sum())
    dev_min = float(((H.ghost_on - H.live_on).shift(1).fillna(0) * dt * 60).sum())
    return kwh, dev_min, H


print("Room control ready:", f"camera detection ON (YOLOv8n, weights saved at {YOLO_PATH} — no download needed on stage)"
      if YOLO_MODEL and os.path.exists(YOLO_PATH) else "camera detection OFF",
      "·", "comfort model ON (ASHRAE 55 adaptive)" if adaptive_ashrae else "comfort model OFF",
      "·", "smart switch via webhook" if DEVICE_WEBHOOK else "virtual switch (no smart plug connected)")
fetch_weather("Bengaluru")
print("Live weather:", f"{WEATHER['city']} {WEATHER['now']} °C now, 7-day mean {WEATHER['trm']:.1f} °C"
      if WEATHER["trm"] is not None else f"not reachable ({WEATHER['error']}) — type the outdoor 7-day mean in the app")
# self-test on a real photo that ships with Ultralytics, then reset the room
try:
    from ultralytics.utils import ASSETS
    import cv2
    _img = cv2.imread(str(ASSETS / "bus.jpg"))[:, :, ::-1]
    _a, _n, _raw, _why = process_frame(_img, room_temp=31, auto=True, outdoor_manual=24, still=True)
    print(f"Self-test on Ultralytics sample photo (31 °C): {_raw} people detected → {_why}")
    for _t in (24, 27):
        print(f"  same photo at {_t} °C →", decide(_n, 0, _t, thresholds(None, 24), 20)[2])
    print("  20 people at 24 °C →", decide(20, 0, 24, thresholds(None, 24), 20)[2])
except Exception as e:
    print("Self-test skipped:", e)
ROOM = _new_room()


## Cell 13b · 🏫 Departments — the campus model
Sixteen departments (CSE, ECE, Mechanical, Civil, Biotech, MBA, Medical Sciences, Law, Pharmacy, Basic Sciences, Architecture & Design,
Commerce, Sports Complex, Library, Admin, Hostels) on a Bengaluru campus, **day by day for 2+ years**:
real Bengaluru monthly temperatures, the academic calendar (semesters, exams, breaks) and Karnataka holidays drive occupancy;
each department has its own always-on load, people load and cooling load. Water and waste per person too.

Nine changes are planted with a **known true effect**. GreenLedger then verifies them exactly as it would real ones
(ghost twin + placebo test). Because we know the truth, you can see that it **never over-counts** — the counted saving is the low end.
The **Total energy saved** on the dashboard comes from here.

In [ ]:
# ---- 🏫 Departments — SIMULATED data that behaves like a real Indian campus (clearly labelled in the app) ----
# Why simulated: no university publishes department-level meters. This generator mimics how real buildings behave:
#   • Bengaluru climate (monthly normals) with day-to-day weather noise → cooling load above 24 °C
#   • academic calendar: odd semester Aug–Nov, even Jan–Apr, exams, winter & summer breaks, Sundays, Karnataka holidays
#   • always-on base load (servers, freezers, corridors) + occupancy load + cooling load + random noise
#   • water and waste per person present; hostels stay occupied at weekends
#   • sustainability ACTIONS on known dates with a KNOWN true effect (one with zero effect) — so we can check whether the
#     ghost twin + placebo test recovers the truth. That is the point of a simulation: it tests the method.
SIM_SEED = 2026
_rng = np.random.default_rng(SIM_SEED)
SIM_DAYS = pd.date_range("2024-06-01", "2026-09-30", freq="D")
TARIFF_INR_PER_KWH = 8.5           # assumption for the simulation (typical Indian HT educational tariff band); editable
INDIA_KG_PER_KWH = 0.710           # CEA CO2 Baseline Database v21.0

_normals = [21.6, 23.6, 26.0, 27.6, 26.9, 24.2, 23.3, 23.2, 23.4, 23.3, 22.0, 21.0]   # Bengaluru monthly mean °C (approx. normals)
_clim = pd.Series([_normals[d.month - 1] for d in SIM_DAYS], index=SIM_DAYS).rolling(31, center=True, min_periods=1).mean()
_noise = np.zeros(len(SIM_DAYS))
for i in range(1, len(SIM_DAYS)):
    _noise[i] = 0.7 * _noise[i - 1] + _rng.normal(0, 0.8)
SIM_TEMP = (_clim + _noise).round(1)

try:
    _KA = holidays.country_holidays("IN", subdiv="KA", years=sorted(set(SIM_DAYS.year)))
except Exception:
    _KA = holidays.country_holidays("IN", years=sorted(set(SIM_DAYS.year)))


def _period_of(d):
    m, day = d.month, d.day
    if (m == 12 and day >= 16) or (m == 1 and day <= 5):
        return "break"
    if m in (6, 7) or (m == 5 and day >= 21):
        return "break"
    if (m == 12 and day < 16) or (m == 5 and day < 21):
        return "exams"
    return "semester"


SIM_CAL = pd.DataFrame({"period": [_period_of(d) for d in SIM_DAYS],
                        "dow": SIM_DAYS.dayofweek, "holiday": [d.date() in _KA for d in SIM_DAYS]}, index=SIM_DAYS)


def _occupancy(kind):
    """Share of people present each day (0–1)."""
    occ = []
    for d, r in SIM_CAL.iterrows():
        base = {"semester": 1.0, "exams": 0.8, "break": 0.15 if kind != "hostel" else 0.25}[r.period]
        if kind == "hostel":
            o = base * (0.9 if r.dow == 6 else 1.0)
        elif r.holiday or r.dow == 6:
            o = 0.08
        elif r.dow == 5:
            o = base * 0.45
        else:
            o = base
        occ.append(o)
    return pd.Series(occ, index=SIM_DAYS)


DEPTS = pd.DataFrame([
    # code, name, building, kind, area m², students, staff, base kW, occupied kW (10 h), cooling kWh/°C·day, water L/person/day, waste kg/person/day, diversion
    ("CSE",  "Computer Science & Engg.",   "Block A",      "academic", 4200, 1200, 85, 38, 120, 26, 18, 0.11, 0.42),
    ("ECE",  "Electronics & Comm. Engg.",  "Block A",      "academic", 3800,  900, 70, 30, 105, 22, 18, 0.10, 0.38),
    ("MECH", "Mechanical Engg.",           "Block C",      "academic", 5200,  800, 75, 34, 140, 18, 22, 0.14, 0.31),
    ("CIVIL","Civil Engg.",                "Block C",      "academic", 3600,  600, 55, 16,  70, 14, 20, 0.12, 0.35),
    ("BT",   "Biotechnology",              "Block D",      "academic", 2600,  400, 45, 42,  60, 16, 30, 0.13, 0.33),
    ("MBA",  "MBA (Management Studies)",   "Block B",      "academic", 2400,  500, 40, 12,  45, 15, 16, 0.09, 0.47),
    ("MED",  "Medical Sciences",           "Medical Block","hostel",   6500,  900, 300, 110, 140, 45, 60, 0.25, 0.30),
    ("LAW",  "School of Law",              "Law School",   "academic", 2200,  600, 60, 14,  40, 10, 14, 0.08, 0.45),
    ("PHAR", "Pharmacy",                   "Block G",      "academic", 2400,  500, 70, 22,  60, 12, 20, 0.11, 0.36),
    ("SCI",  "Basic Sciences",             "Block S",      "academic", 2800,  700, 90, 25,  70, 14, 17, 0.10, 0.38),
    ("ARCH", "Architecture & Design",      "Block H",      "academic", 2000,  450, 50, 12,  45, 10, 15, 0.12, 0.40),
    ("COM",  "Commerce",                   "Commerce Block","academic", 1800, 650, 50, 10,  35,  9, 13, 0.08, 0.46),
    ("SPORTS","Sports Complex",            "Sports Complex","hostel",  8000,    0, 40, 15,  80,  4, 25, 0.06, 0.35),
    ("LIB",  "Central Library",            "Library",      "academic", 3000,    0, 30, 14,  55, 20, 10, 0.05, 0.55),
    ("ADM",  "Administration",             "Admin Block",  "academic", 1800,    0, 90, 10,  40, 35, 18, 0.08, 0.50),
    ("HOS",  "Hostels",                    "Hostel Zone",  "hostel",   9000, 1400, 30, 45, 160, 10, 135, 0.30, 0.22),
], columns=["code", "name", "building", "kind", "area_m2", "students", "staff", "base_kw", "occ_kw", "cool_kwh_per_c",
            "water_l_pp", "waste_kg_pp", "diversion"]).set_index("code")
DEPTS.loc["LIB", "students"] = 0
DEPT_PEOPLE = DEPTS.students + DEPTS.staff
DEPT_PEOPLE.loc["LIB"] = 2400          # daily library visitors (simulated)
DEPT_PEOPLE.loc["SPORTS"] = 1500       # daily users of the grounds, gym and indoor stadium (simulated)

# sustainability actions with a KNOWN true effect (what the ghost twin must recover)
SIM_ACTIONS = pd.DataFrame([
    ("D-01", "CSE",  "2025-08-04", "Lab PCs auto-shutdown after 8 pm",            "occ",  -0.12),
    ("D-02", "MECH", "2025-09-01", "Compressed-air leak repair in workshops",     "base", -0.10),
    ("D-03", "LIB",  "2025-10-06", "LED lighting + occupancy sensors",            "occ",  -0.25),
    ("D-04", "HOS",  "2025-11-03", "Solar water heaters (replace electric geysers)", "occ", -0.18),
    ("D-05", "BT",   "2026-01-12", "Ultra-low freezer consolidation",             "base", -0.15),
    ("D-06", "ADM",  "2026-03-02", "AC setpoint fixed at 24 °C (BEE default)",    "cool", -0.30),
    ("D-07", "CIVIL","2026-02-16", "Awareness posters only",                      "occ",   0.00),
    ("D-08", "MED",  "2025-12-01", "Night setback for lab and ward air-handling",  "base", -0.08),
    ("D-09", "SPORTS","2026-01-05", "LED floodlights with timers",                  "occ",  -0.35),
], columns=["id", "dept", "date", "title", "part", "true_effect"])
SIM_ACTIONS["date"] = pd.to_datetime(SIM_ACTIONS["date"])

SIM_E, SIM_W, SIM_WASTE, SIM_DIV, SIM_TRUE_SAVED = {}, {}, {}, {}, {}
for c, d in DEPTS.iterrows():
    occ = _occupancy(d.kind)
    cdd = np.maximum(0, SIM_TEMP - 24.0)
    base = pd.Series(d.base_kw * 24.0, index=SIM_DAYS)
    use = pd.Series(d.occ_kw * 10.0, index=SIM_DAYS) * occ
    cool = d.cool_kwh_per_c * cdd * np.maximum(occ, 0.3)
    parts = {"base": base.copy(), "occ": use.copy(), "cool": cool.copy()}
    for _, a in SIM_ACTIONS[SIM_ACTIONS.dept == c].iterrows():
        after = SIM_DAYS >= a.date
        parts[a.part][after] = parts[a.part][after] * (1 + a.true_effect)
    clean = parts["base"] + parts["occ"] + parts["cool"]
    no_action = base + use + cool
    noise = _rng.normal(1, 0.05, len(SIM_DAYS))
    SIM_E[c] = (clean * noise).round(1)
    SIM_TRUE_SAVED[c] = ((no_action - clean) * noise)                     # what the actions really saved (kWh/day)
    people = DEPT_PEOPLE[c] * occ
    SIM_W[c] = (people * d.water_l_pp * _rng.normal(1, 0.07, len(SIM_DAYS)) + d.area_m2 * 0.5).round(0)
    SIM_WASTE[c] = (people * d.waste_kg_pp * _rng.normal(1, 0.10, len(SIM_DAYS))).round(1)
SIM_E, SIM_W, SIM_WASTE = pd.DataFrame(SIM_E), pd.DataFrame(SIM_W), pd.DataFrame(SIM_WASTE)


# ---- department ghost twin: same idea as the real engine (cells 5 and 8) — learn 12 months before, predict after,
#      placebo-in-time on every week of the year before, VERIFIED only if 95% range excludes 0 and p ≤ 5%.
def _dept_X(idx):
    cal = SIM_CAL.reindex(idx)
    weekday_class = ((cal.dow < 5) & ~cal.holiday).astype(float)
    sat = ((cal.dow == 5) & ~cal.holiday).astype(float)
    off = (~(weekday_class.astype(bool) | sat.astype(bool))).astype(float)
    cols = [np.ones(len(idx))]
    for p in ("semester", "exams"):
        inp = (cal.period == p).astype(float)
        cols += [(inp * weekday_class).values, (inp * sat).values, (inp * off).values]
    cols.append(np.maximum(0, SIM_TEMP.reindex(idx).values - 24.0))
    return np.column_stack(cols)


def _dept_fit(y):
    y = y.dropna()
    return np.linalg.lstsq(_dept_X(y.index), y.values, rcond=None)[0], y.index


def _dept_pred(b, idx):
    return pd.Series(_dept_X(idx) @ b, index=idx)


def dept_verify(c, date, window=56):
    T0 = pd.Timestamp(date)
    y = SIM_E[c]
    base = y.loc[T0 - pd.Timedelta(days=365): T0 - pd.Timedelta(days=1)]
    post = y.loc[T0: T0 + pd.Timedelta(days=window - 1)]
    b, _ = _dept_fit(base)
    fakes, d = [], base.index.min()
    while d + pd.Timedelta(days=window - 1) <= base.index.max():
        tr = base.copy(); tr.loc[d: d + pd.Timedelta(days=window - 1)] = np.nan
        bb, _ = _dept_fit(tr)
        hold = base.loc[d: d + pd.Timedelta(days=window - 1)]
        e = _dept_pred(bb, hold.index)
        fakes.append(float((e.sum() - hold.sum()) / e.sum() * 100))
        d += pd.Timedelta(days=7)
    fakes = np.array(fakes)
    half, typical = range_from_misses(fakes, 365 // window)
    ghost = _dept_pred(b, post.index)
    saving = float((ghost.sum() - post.sum()) / ghost.sum() * 100)
    lo, hi = saving - half, saving + half
    p = float((1 + np.sum(np.abs(fakes) >= abs(saving))) / (len(fakes) + 1))
    verdict = "VERIFIED SAVING" if (lo > 0 and p <= ALPHA) else ("VERIFIED INCREASE" if (hi < 0 and p <= ALPHA) else "NOT PROVEN")
    after = y.loc[T0:]                                    # saved to date: ghost − actual since the action, at the LOW end
    g_after = _dept_pred(b, after.index)
    measured_to_date = float((g_after - after).sum())
    counted = max(0.0, measured_to_date * lo / saving) if verdict == "VERIFIED SAVING" and saving > 0 else 0.0
    true_to_date = float(SIM_TRUE_SAVED[c].loc[T0:].sum())
    true_pct = float(SIM_TRUE_SAVED[c].loc[post.index].sum() / (post.sum() + SIM_TRUE_SAVED[c].loc[post.index].sum()) * 100)
    return {"dept": c, "date": T0, "saving_%": saving, "low_%": lo, "high_%": hi, "p": p, "verdict": verdict,
            "measured_kwh": measured_to_date, "counted_kwh": counted, "true_kwh": true_to_date, "true_%": true_pct,
            "ghost": g_after, "actual": after, "pre": base.iloc[-56:], "pre_ghost": _dept_pred(b, base.index[-56:]),
            "fakes": fakes, "typical": typical}


DEPT_RESULTS = {a.id: {**dept_verify(a.dept, a.date), "title": a.title, "id": a.id} for _, a in SIM_ACTIONS.iterrows()}
DEPT_SAVED_KWH = sum(r["counted_kwh"] for r in DEPT_RESULTS.values())
DEPT_TRUE_KWH = sum(r["true_kwh"] for r in DEPT_RESULTS.values())

# ---- department analysis: last 12 months ----
_L = SIM_DAYS[-365:]
_occ_mean = {c: float(_occupancy(DEPTS.loc[c, "kind"]).loc[_L].mean()) for c in DEPTS.index}
DEPT_TABLE = pd.DataFrame({
    "Department": DEPTS.name, "Building": DEPTS.building, "Area m²": DEPTS.area_m2, "People": DEPT_PEOPLE,
    "Energy MWh/yr": SIM_E.loc[_L].sum() / 1000,
    "kWh per m²": SIM_E.loc[_L].sum() / DEPTS.area_m2,
    "kWh per person": SIM_E.loc[_L].sum() / DEPT_PEOPLE,
    "Water L/person/day": SIM_W.loc[_L].sum() / DEPT_PEOPLE / 365 / pd.Series(_occ_mean),
    "Waste kg/person/day": SIM_WASTE.loc[_L].sum() / DEPT_PEOPLE / 365 / pd.Series(_occ_mean),
    "Diversion %": DEPTS.diversion * 100,
    "Always-on %": (DEPTS.base_kw * 24 * 365) / SIM_E.loc[_L].sum() * 100,
    "Weekend vs weekday %": pd.Series({c: float(SIM_E.loc[_L][c][SIM_CAL.loc[_L].dow == 6].mean() /
                                               SIM_E.loc[_L][c][(SIM_CAL.loc[_L].dow < 5) & (SIM_CAL.loc[_L].period == "semester")].mean() * 100)
                                       for c in DEPTS.index}),
})
DEPT_TABLE["CO₂ t/yr"] = DEPT_TABLE["Energy MWh/yr"] * INDIA_KG_PER_KWH
DEPT_TABLE["Cost ₹/month"] = DEPT_TABLE["Energy MWh/yr"] * 1000 * TARIFF_INR_PER_KWH / 12
_med = DEPT_TABLE["kWh per m²"].median()
DEPT_TABLE["Efficiency"] = np.where(DEPT_TABLE["kWh per m²"] > 1.15 * _med, "High use",
                                    np.where(DEPT_TABLE["kWh per m²"] < 0.85 * _med, "Efficient", "Average"))
DEPT_TABLE["Saved MWh (verified)"] = pd.Series({c: sum(r["counted_kwh"] for r in DEPT_RESULTS.values() if r["dept"] == c) / 1000
                                                for c in DEPTS.index})
LAST_MONTH = (SIM_DAYS[-1] + pd.Timedelta(days=1) - pd.offsets.MonthBegin(1)).normalize()   # last full month in the data


def counted_daily(r):
    """kWh saved per day by one verified action, scaled to the LOW end of its range (same rule as the headline)."""
    m = (r["ghost"] - r["actual"]).dropna()
    return m * (r["low_%"] / r["saving_%"]) if r["verdict"] == "VERIFIED SAVING" and r["saving_%"] > 0 else m * 0


DEPT_TABLE["Saved kWh last month"] = pd.Series({c: float(sum(counted_daily(r).loc[LAST_MONTH:].sum() for r in DEPT_RESULTS.values()
                                                         if r["dept"] == c)) for c in DEPTS.index})
DEPT_TABLE["Saved ₹ last month"] = DEPT_TABLE["Saved kWh last month"] * TARIFF_INR_PER_KWH


def dept_tip(c):
    """One plain suggestion per department: the measure where it is furthest above the campus median (relative gap)."""
    r, T = DEPT_TABLE.loc[c], DEPT_TABLE
    m = lambda k: T[k].median()
    tips = []
    if r["Always-on %"] > m("Always-on %") * 1.1:
        tips.append((r["Always-on %"] / m("Always-on %"), f"{r['Always-on %']:.0f}% of its energy is always-on (campus median "
                     f"{m('Always-on %'):.0f}%) — night/weekend shutdown of labs, servers and freezers."))
    if r["Weekend vs weekday %"] > m("Weekend vs weekday %") * 1.1:
        tips.append((r["Weekend vs weekday %"] / m("Weekend vs weekday %"), f"Sundays still use {r['Weekend vs weekday %']:.0f}% of a weekday "
                     f"(median {m('Weekend vs weekday %'):.0f}%) — tighten weekend schedules."))
    if r["kWh per m²"] > m("kWh per m²") * 1.1:
        tips.append((r["kWh per m²"] / m("kWh per m²"), f"Uses {r['kWh per m²']:.0f} kWh per m² a year (campus median {m('kWh per m²'):.0f}) — "
                     f"audit the biggest loads: lab equipment, AC hours and set-points."))
    if r["Water L/person/day"] > m("Water L/person/day") * 1.2:
        tips.append((r["Water L/person/day"] / m("Water L/person/day"), f"{r['Water L/person/day']:.0f} L per person per day "
                     f"(median {m('Water L/person/day'):.0f}) — low-flow taps and leak checks."))
    if r["Diversion %"] < m("Diversion %") * 0.9:
        tips.append((m("Diversion %") / max(r["Diversion %"], 1), f"Only {r['Diversion %']:.0f}% of waste is diverted (median "
                     f"{m('Diversion %'):.0f}%) — add segregation bins."))
    return max(tips)[1] if tips else "Performing at or better than the campus median on every measure — keep it up."


DEPT_TABLE["Top suggestion"] = [dept_tip(c) for c in DEPT_TABLE.index]
print(f"Campus model: {len(DEPTS)} departments · {len(SIM_DAYS)} days · {len(SIM_ACTIONS)} actions with known true effects")
print(f"Total energy saved (verified, counted at the low end): {DEPT_SAVED_KWH / 1000:,.1f} MWh "
      f"— true simulated saving {DEPT_TRUE_KWH / 1000:,.1f} MWh (GreenLedger never over-counts)")
for r in DEPT_RESULTS.values():
    print(f"  {r['id']} {r['dept']:5s} {r['title'][:42]:42s} → {r['verdict']:16s} est {r['saving_%']:5.1f}% "
          f"[{r['low_%']:5.1f}, {r['high_%']:5.1f}] p={r['p']:.3f} · true {r['true_%']:5.1f}%")
print(DEPT_TABLE[["Department", "Energy MWh/yr", "kWh per m²", "kWh per person", "Efficiency", "Saved MWh (verified)"]].round(1).to_string())


## Cell 13c · 💻 E-waste — SIMULATED asset register (clearly labelled)
About 2,800 devices across 16 departments (PCs, laptops, monitors, projectors, ACs, UPS, oscilloscopes, fans…) with purchase dates, condition and market prices.
For each device GreenLedger says what to do and **when**:
* **Keep in use** — and the month it reaches its best resale moment (a year before end of life)
* **Sell second-hand now** — and how much is lost by waiting a year
* **Repair** — keep using it, or repair and resell
* **Harvest parts** — RAM, SSD, power supply, compressor, copper coil… before a **registered recycler** takes the rest (E-Waste Rules 2022)

In [ ]:
# ---- 💻 E-waste — SIMULATED asset register for the departments (clearly labelled in the app) ----
# Mimics a real campus IT/appliance inventory: device mix per department, purchase years, condition worsening with age,
# market prices and yearly depreciation. For every device GreenLedger answers: keep it, SELL it second-hand (and when),
# repair it, or HARVEST reusable parts before a registered recycler takes the rest (E-Waste Rules 2022).
_er = np.random.default_rng(SIM_SEED + 1)
TODAY = pd.Timestamp(datetime.date.today())
DEVICE_TYPES = {
    # type: (expected life yrs, new price ₹, weight kg, depreciation/yr, [(reusable part, second-hand ₹, usable only if age ≤ yrs)])
    "Desktop PC":       (6, 55000, 8.0, 0.25, [("RAM 8 GB", 1200, 12), ("SSD 256 GB", 1000, 8), ("Power supply", 800, 10), ("CPU", 2500, 10)]),
    "Laptop":           (5, 65000, 2.0, 0.25, [("RAM 8 GB", 1500, 12), ("SSD 512 GB", 1800, 8), ("Display panel", 2500, 8), ("Charger", 500, 10), ("Battery", 900, 3)]),
    "Monitor 22\"":     (7,  9000, 4.0, 0.20, [("LCD panel", 1500, 9), ("Stand & cables", 250, 15)]),
    "Projector":        (7, 45000, 3.0, 0.20, [("Lamp", 2500, 4), ("Lens assembly", 3000, 10)]),
    "Laser printer":    (5, 25000, 9.0, 0.25, [("Toner unit", 1500, 6), ("Fuser", 2000, 6)]),
    "Split AC 1.5 T":   (10, 42000, 45.0, 0.12, [("Compressor", 6000, 12), ("Copper coil", 2500, 30), ("Control board", 1500, 10)]),
    "UPS 1 kVA":        (7,  8000, 12.0, 0.15, [("Battery (buy-back)", 600, 5), ("Inverter board", 700, 10)]),
    "Network switch":   (5, 18000, 3.0, 0.25, [("SFP modules", 1000, 10), ("Power supply", 600, 10)]),
    "Oscilloscope":     (10, 80000, 6.0, 0.12, [("Probes", 1500, 15), ("Display module", 4000, 12)]),
    "Ceiling fan":      (10, 2800, 4.0, 0.10, [("Motor (rewind)", 300, 20)]),
}
MIX = {  # devices per department (simulated)
    "CSE": {"Desktop PC": 140, "Laptop": 25, "Monitor 22\"": 150, "Projector": 10, "Laser printer": 4, "Network switch": 12, "UPS 1 kVA": 20, "Split AC 1.5 T": 18, "Ceiling fan": 40},
    "ECE": {"Desktop PC": 70, "Laptop": 15, "Monitor 22\"": 70, "Oscilloscope": 30, "Projector": 8, "Laser printer": 3, "UPS 1 kVA": 12, "Split AC 1.5 T": 12, "Ceiling fan": 35},
    "MECH": {"Desktop PC": 50, "Monitor 22\"": 50, "Projector": 8, "Laser printer": 4, "UPS 1 kVA": 8, "Split AC 1.5 T": 6, "Ceiling fan": 45},
    "CIVIL": {"Desktop PC": 35, "Monitor 22\"": 35, "Projector": 6, "Laser printer": 3, "Split AC 1.5 T": 4, "Ceiling fan": 30},
    "BT": {"Desktop PC": 25, "Laptop": 8, "Monitor 22\"": 25, "Projector": 4, "Laser printer": 2, "UPS 1 kVA": 10, "Split AC 1.5 T": 14, "Ceiling fan": 20},
    "MBA": {"Desktop PC": 20, "Laptop": 30, "Monitor 22\"": 20, "Projector": 8, "Laser printer": 3, "Split AC 1.5 T": 10, "Ceiling fan": 15},
    "LIB": {"Desktop PC": 40, "Monitor 22\"": 40, "Laser printer": 4, "Network switch": 4, "UPS 1 kVA": 6, "Split AC 1.5 T": 16, "Ceiling fan": 30},
    "ADM": {"Desktop PC": 45, "Laptop": 20, "Monitor 22\"": 45, "Laser printer": 10, "Network switch": 6, "UPS 1 kVA": 10, "Split AC 1.5 T": 12, "Ceiling fan": 20},
    "HOS": {"Desktop PC": 10, "Monitor 22\"": 10, "Network switch": 8, "Split AC 1.5 T": 4, "Ceiling fan": 400},
    "MED": {"Desktop PC": 60, "Laptop": 20, "Monitor 22\"": 60, "Projector": 6, "Laser printer": 8, "Network switch": 6, "UPS 1 kVA": 20, "Split AC 1.5 T": 30, "Ceiling fan": 60},
    "LAW": {"Desktop PC": 25, "Laptop": 15, "Monitor 22\"": 25, "Projector": 6, "Laser printer": 4, "Split AC 1.5 T": 6, "Ceiling fan": 25},
    "PHAR": {"Desktop PC": 30, "Monitor 22\"": 30, "Projector": 5, "Laser printer": 3, "UPS 1 kVA": 6, "Split AC 1.5 T": 8, "Ceiling fan": 25},
    "SCI": {"Desktop PC": 35, "Monitor 22\"": 35, "Projector": 8, "Laser printer": 3, "UPS 1 kVA": 6, "Oscilloscope": 10, "Split AC 1.5 T": 6, "Ceiling fan": 35},
    "ARCH": {"Desktop PC": 40, "Laptop": 10, "Monitor 22\"": 45, "Projector": 5, "Laser printer": 6, "Split AC 1.5 T": 5, "Ceiling fan": 20},
    "COM": {"Desktop PC": 25, "Laptop": 10, "Monitor 22\"": 25, "Projector": 6, "Laser printer": 3, "Split AC 1.5 T": 4, "Ceiling fan": 20},
    "SPORTS": {"Desktop PC": 4, "Monitor 22\"": 4, "Network switch": 2, "UPS 1 kVA": 2, "Split AC 1.5 T": 4, "Ceiling fan": 30},
}
_rows, _n = [], 0
for dept, mix in MIX.items():
    for t, n in mix.items():
        life, price, kg, dep, parts = DEVICE_TYPES[t]
        for _ in range(n):
            _n += 1
            age = float(_er.uniform(0.3, life * 1.6))
            buy = TODAY - pd.Timedelta(days=int(age * 365.25))
            pf = min(0.6, 0.04 + 0.35 * (age / life) ** 2)                 # older → more likely faulty / dead
            pn = min(0.5, 0.02 + 0.25 * max(0, age / life - 0.7) ** 1.5)
            cond = _er.choice(["working", "faulty", "non-functional"], p=[1 - pf - pn, pf, pn])
            _rows.append((f"{dept}-{_n:04d}", t, dept, buy.normalize(), round(age, 1), life, cond, kg, price))
EW_SIM = pd.DataFrame(_rows, columns=["device_id", "type", "dept", "purchase_date", "age_years", "life_years", "condition", "weight_kg", "price_new"])


def ew_decide(r):
    life, price, kg, dep, parts = DEVICE_TYPES[r.type]
    value_now = price * (1 - dep) ** r.age_years
    usable = [(n, v) for n, v, max_age in parts if r.age_years <= max_age]
    parts_value = sum(v for _, v in usable)
    sell_at = r.purchase_date + pd.DateOffset(years=life - 1)               # best moment: a year before end of life
    if r.condition == "working" and r.age_years < life - 1:
        return "Keep in use", 0.0, "", sell_at, value_now * (1 - dep) ** (life - 1 - r.age_years)
    if r.condition == "working":
        return "Sell second-hand now", value_now, "", TODAY, value_now
    if r.condition == "faulty" and r.age_years < life - 1:                  # young: fixing it beats buying a new one
        repair = 0.15 * price
        if value_now > repair:
            return "Repair & keep using", value_now - repair, "", TODAY, value_now - repair
    if r.condition == "faulty":
        repair = 0.15 * price
        if value_now - repair > parts_value:
            return "Repair & resell", value_now - repair, "", TODAY, value_now - repair
        return "Harvest parts", parts_value, ", ".join(n for n, _ in usable), TODAY, parts_value
    return "Harvest parts + recycle", parts_value, ", ".join(n for n, _ in usable), TODAY, parts_value


_d = EW_SIM.apply(ew_decide, axis=1, result_type="expand")
EW_SIM["action"], EW_SIM["value_now"], EW_SIM["reusable_parts"], EW_SIM["when"], EW_SIM["value_then"] = _d[0], _d[1], _d[2], _d[3], _d[4]
EW_SIM["lost_per_year_waiting"] = np.where(EW_SIM.action == "Sell second-hand now",
                                           EW_SIM.value_now * EW_SIM.type.map(lambda t: DEVICE_TYPES[t][3]), 0.0)
EW_SIM["to_recycler_kg"] = np.where(EW_SIM.action.str.contains("recycle|Harvest"), EW_SIM.weight_kg, 0.0)
EW_SUMMARY = {
    "devices": len(EW_SIM),
    "sell_now": int((EW_SIM.action == "Sell second-hand now").sum()),
    "sell_value": float(EW_SIM.loc[EW_SIM.action == "Sell second-hand now", "value_now"].sum()),
    "repair": int(EW_SIM.action.str.startswith("Repair").sum()),
    "repair_value": float(EW_SIM.loc[EW_SIM.action.str.startswith("Repair"), "value_now"].sum()),
    "harvest": int(EW_SIM.action.str.startswith("Harvest").sum()),
    "parts_value": float(EW_SIM.loc[EW_SIM.action.str.startswith("Harvest"), "value_now"].sum()),
    "recycler_kg": float(EW_SIM.to_recycler_kg.sum()),
    "keep": int((EW_SIM.action == "Keep in use").sum()),
    "lost_per_year": float(EW_SIM.lost_per_year_waiting.sum()),
}
EW_SUMMARY["recoverable_value"] = EW_SUMMARY["sell_value"] + EW_SUMMARY["repair_value"] + EW_SUMMARY["parts_value"]
_upc = EW_SIM[(EW_SIM.action == "Keep in use") & (EW_SIM["when"] <= TODAY + pd.DateOffset(months=12))]
EW_UPCOMING = _upc.groupby(_upc["when"].dt.to_period("M")).agg(devices=("device_id", "count"), value=("value_then", "sum"))
EW_PARTS = (EW_SIM[EW_SIM.reusable_parts != ""].assign(part=EW_SIM.reusable_parts.str.split(", ")).explode("part")
            .groupby("part").size().sort_values(ascending=False))
EW_BY_DEPT = EW_SIM.pivot_table(index="dept", columns="action", values="device_id", aggfunc="count", fill_value=0)
print(f"E-waste register: {EW_SUMMARY['devices']} devices across {EW_SIM.dept.nunique()} departments")
print(f"  Sell second-hand now: {EW_SUMMARY['sell_now']} devices ≈ ₹{EW_SUMMARY['sell_value']:,.0f} "
      f"(waiting a year loses ≈ ₹{EW_SUMMARY['lost_per_year']:,.0f})")
print(f"  Repair: {EW_SUMMARY['repair']} · Harvest parts: {EW_SUMMARY['harvest']} devices ≈ ₹{EW_SUMMARY['parts_value']:,.0f} "
      f"· to registered recycler: {EW_SUMMARY['recycler_kg']:,.0f} kg · keep in use: {EW_SUMMARY['keep']}")
print("  Most reusable parts:", EW_PARTS.head(6).to_dict())


## Cell 13d · 🗺️ Your campus image (DSU)
The aerial campus picture behind the **interactive campus dashboard** and the **Departments** page. Each department sits in a block on the photo
(Block A · CSE, Block B · ECE, … Admin Block, Central Library, Hostels). To use another photo, upload `university-campus.jpg` and move the pins in this cell.

In [ ]:
# ---- Your campus: the DSU aerial image — the backdrop of the interactive campus dashboard and the Departments page ----
# To use a different photo, upload university-campus.jpg (or .png) to Colab; it replaces this one (then move the pins below).
CAMPUS_IMG_B64 = '...'
_own = next((p for root in [DATA_DIR, '.', '/content'] for ext in ('jpg', 'jpeg', 'png')
             for p in glob.glob(os.path.join(root, f'university-campus*.{ext}'))), None)
CAMPUS_IMG_BYTES = open(_own, 'rb').read() if _own else base64.b64decode(CAMPUS_IMG_B64)
CAMPUS_IMG_TYPE = "image/png" if CAMPUS_IMG_BYTES[:4] == b"\x89PNG" else ("image/webp" if CAMPUS_IMG_BYTES[8:12] == b"WEBP" else "image/jpeg")
CAMPUS_PINS = {  # block on the photo → (x %, y %) of the image, and the department in it
    "Block A · CSE": (44, 26, ["CSE"]), "Block B · ECE": (30, 35, ["ECE"]), "Block C · Mechanical": (25, 49, ["MECH"]),
    "Block D · Civil": (47, 61, ["CIVIL"]), "Block E · Biotech": (66, 54, ["BT"]), "Block F · MBA": (77, 40, ["MBA"]),
    "Admin Block": (61, 27, ["ADM"]), "Central Library": (92, 52, ["LIB"]), "Hostels": (8, 29, ["HOS"]),
    "Medical Block": (80, 62, ["MED"]), "School of Law": (93, 71, ["LAW"]), "Block G · Pharmacy": (32, 16, ["PHAR"]),
    "Block S · Sciences": (91, 34, ["SCI"]), "Block H · Architecture": (79, 23, ["ARCH"]), "Commerce Block": (21, 23, ["COM"]),
    "Sports Complex": (17, 6, ["SPORTS"]), "Indoor Stadium": (55, 9, []), "Car Park": (74, 84, []),
}
for _b, (_x, _y, _d) in CAMPUS_PINS.items():          # keep the department table's building names in step with the photo
    for _c in _d:
        DEPTS.loc[_c, "building"] = _b
        DEPT_TABLE.loc[_c, "Building"] = _b
print(f"Campus image ready ({len(CAMPUS_IMG_BYTES) // 1024} KB) · {len(CAMPUS_PINS)} places, "
      f"{sum(bool(v[2]) for v in CAMPUS_PINS.values())} with department data")


## Cell 13e · 🏅 Judge's guide, full report and labelled dataset
The brief asks for an **exportable report** and a **labelled sample dataset**, and asks every team to state **assumptions, limitations,
safety and how they would validate** the prototype. This cell writes all of it once, from the results above (nothing typed in by hand):
* **Validation numbers** — truth inside the 95% range on the simulation, never over-counted, false savings on real placebo months, COVID closure verified.
* **Judge's Guide** page in the app — every line of the brief mapped to where it is, plus assumptions, limitations, safety & privacy, pilot plan and measurable success criteria.
* **Report (HTML)** and **dataset (ZIP)** with `real/` and `simulated/` folders, a label on every row and a `DATA_DICTIONARY.csv`.

In [ ]:
# ---- 🏅 Judge's guide, Support & Info, the full exportable report and the labelled dataset ----
# The brief asks for: a resource dashboard, baseline/trend comparisons, an action register, an exportable report and a
# labelled sample dataset — and asks every team to state assumptions, limitations, safety considerations and how they
# would validate the prototype. This cell writes all of that ONCE; the app's "Judge's guide" page and the report use it.

# ---------- 1. validation evidence, computed from the results above (nothing typed in by hand) ----------
SIM_CHECK = pd.DataFrame([{
    "action": r["id"], "department": DEPTS.loc[r["dept"], "name"], "what was done": r["title"], "date": r["date"].date(),
    "true effect %": round(r["true_%"], 1), "GreenLedger estimate %": round(r["saving_%"], 1),
    "95% low": round(r["low_%"], 1), "95% high": round(r["high_%"], 1), "p": round(r["p"], 3), "verdict": r["verdict"],
    "truth inside range": bool(r["low_%"] <= r["true_%"] <= r["high_%"]),
    "counted MWh": round(r["counted_kwh"] / 1000, 1), "true MWh": round(r["true_kwh"] / 1000, 1),
    "never over-counted": bool(r["counted_kwh"] <= r["true_kwh"] + 1e-6)} for r in DEPT_RESULTS.values()])
_A = ACTIONS.set_index("action_id")
_gt = {k: v for k, v in VERIFIED.items() if "Ground-truth" in str(_A["type"].get(k, ""))
       and k in _A.index and METER_INFO[_A.loc[k, "point"]][1] == "electricity"}          # closure = lights, PCs, plug loads
VALIDATION = {
    "sim_inside": int(SIM_CHECK["truth inside range"].sum()), "sim_n": len(SIM_CHECK),
    "sim_no_overcount": int(SIM_CHECK["never over-counted"].sum()),
    "zero_effect_caught": bool((SIM_CHECK.loc[SIM_CHECK["true effect %"].abs() < 0.5, "verdict"] != "VERIFIED SAVING").all()),
    "trust_tests": TRUST_SUMMARY["tests"], "trust_naive": TRUST_SUMMARY["naive_saving"], "trust_gl": TRUST_SUMMARY["gl_saving"],
    "covid_verified": sum(v["verdict"] == "VERIFIED SAVING" for v in _gt.values()), "covid_n": len(_gt),
}

# ---------- 2. the judge's guide ----------
def _k(v):
    """Money in thousands of rupees, e.g. ₹226K."""
    return f"₹{v / 1000:,.0f}K"


TEAM = {"name": "Shenzein", "members": ["Anup Batakurki", "Het Balchandani", "Kamaal"],
        "college": "Dayananda Sagar University, Bengaluru", "problem": "IC26-05-01 · GreenLedger: Campus Resource Optimisation"}
_m = LAST_MONTH.strftime("%B %Y")
JUDGE_GUIDE = {
    "brief": [
        {"asks": "One place for electricity, water and waste", "how": "Every department’s electricity, water and waste side by side.", "where": "departments"},
        {"asks": "A resource dashboard", "how": "One screen: energy saved last month, money saved, water, waste and e-waste.", "where": "dashboard"},
        {"asks": "Compare with a baseline", "how": "A ‘ghost twin’ shows what each building would have used without the change, on the same weather and calendar.", "where": "prove"},
        {"asks": "Spot unusual changes", "how": "Alerts when use jumps, and ‘fix now’ items with the extra energy already wasted.", "where": "recs"},
        {"asks": "An action register", "how": "Every change made on campus, with a clear verdict: saved, not proven, or increased.", "where": "prove"},
        {"asks": "Never overstate the benefit", "how": "A saving counts only if it is bigger than normal ups and downs — and only the lower, safe amount.", "where": "prove"},
        {"asks": "An exportable report", "how": "One-click report and spreadsheets.", "where": "reports"},
        {"asks": "A labelled sample dataset", "how": "A ZIP with every reading labelled, plus a dictionary of every column.", "where": "reports"},
    ],
    "validation_done": [
        f"Found the true saving: on our test campus, where the real effect of every change was known, GreenLedger’s answer contained the truth {VALIDATION['sim_inside']} times out of {VALIDATION['sim_n']}.",
        f"Never claimed too much: the saving it counted was never more than the real saving ({VALIDATION['sim_no_overcount']} of {VALIDATION['sim_n']}). A change that did nothing (awareness posters) was correctly rejected.",
        f"No false alarms on real meters: in {VALIDATION['trust_tests']} real months where nothing was done, a normal dashboard claimed a saving {VALIDATION['trust_naive']} times. GreenLedger: {VALIDATION['trust_gl']}.",
        f"Catches real events: the COVID-19 closure of a real university is detected on {VALIDATION['covid_verified']} of {VALIDATION['covid_n']} electricity meters.",
    ],
    "validation_plan": [
        "Weeks 1–2: load our campus electricity and water bills.",
        "Week 3: GreenLedger learns each building’s normal use.",
        "Month 2: Facilities makes one change (e.g., lab PCs switch off at night).",
        "Months 3–4: GreenLedger checks it and the bill confirms it.",
    ],
    "assumptions": [
        {"what": "Electricity price", "value": f"₹{TARIFF_INR_PER_KWH} per kWh", "source": "typical Indian college tariff — change it to the rate on your bill"},
        {"what": "CO₂ per kWh (India)", "value": f"{INDIA_GRID_KG_PER_KWH} kg", "source": "Central Electricity Authority (CEA)"},
        {"what": "History needed", "value": "12 months before a change", "source": "IPMVP / ASHRAE Guideline 14"},
        {"what": "What is counted", "value": "the lower end of the saving range", "source": "so a saving is never over-claimed"},
        {"what": "Room comfort", "value": f"ASHRAE 55; AC never below {BEE_DEFAULT_AC_C} °C", "source": "ASHRAE 55; Bureau of Energy Efficiency"},
        {"what": "Empty room → off after", "value": f"{VACANCY_MIN_DEFAULT} min (30 s in demo)", "source": "ASHRAE 90.1"},
        {"what": "Heat from one person", "value": f"{PERSON_HEAT_W} W", "source": "ASHRAE Handbook — Fundamentals"},
    ],
    "safety": [
        "Camera: frames are checked in memory and never saved; only the number of people is kept. People are blurred on screen. No face recognition.",
        "Switching: virtual by default; real devices only through a certified smart plug, and every device can be switched by hand.",
        f"Comfort: lights stay on while anyone is in the room; AC never below {BEE_DEFAULT_AC_C} °C; nothing switches off until the room has been empty for the full delay.",
        "E-waste: anything leaving campus goes only to a CPCB-registered recycler (E-Waste Rules, 2022); data drives are wiped first.",
        "Data: runs on the university’s own machine; no personal data is collected.",
    ],
    "success_criteria": [
        {"metric": "False savings in months where nothing was done", "target": "5% or less", "today": f"{VALIDATION['trust_gl']} of {VALIDATION['trust_tests']}"},
        {"metric": "True saving inside GreenLedger’s range", "target": "95% or more", "today": f"{VALIDATION['sim_inside']} of {VALIDATION['sim_n']}"},
        {"metric": "Energy saved last month", "target": "grows every month", "today": f"{DEPT_TABLE['Saved kWh last month'].sum():,.0f} kWh · {_k(DEPT_TABLE['Saved ₹ last month'].sum())} ({_m})"},
        {"metric": "Devices with an e-waste decision", "target": "100%", "today": f"{len(EW_SIM):,} of {len(EW_SIM):,}"},
    ],
    "stakeholders": [
        {"who": "Facilities office", "role": "sees alerts, makes changes, logs them"},
        {"who": "Green-campus committee · IQAC", "role": "uses the verified report for green and energy audits (e.g., NAAC Criterion 7)"},
        {"who": "MoEFCC · BEE · CPCB", "role": "honest climate claims · energy norms · registered e-waste recyclers"},
    ],
    "about_data": "Real university meters: Concordia University open data (Montréal, CC BY 4.0) — 6 meters and 666 waste invoices, used for the trust test and the COVID check. "
                  "Departments, their changes and the device register come from GreenLedger’s campus model, built on Bengaluru weather and our academic calendar, "
                  "where the true effect of every change is known — that is how GreenLedger’s accuracy is measured. Connect your own bills or meters and it runs on them unchanged.",
}

# ---------- 3. labelled simulated dataset + data dictionary ----------
def sim_dataset():
    rows = []
    for c in DEPTS.index:
        rows.append(pd.DataFrame({
            "date": SIM_DAYS, "data_type": "SIMULATED", "department_code": c, "department": DEPTS.loc[c, "name"],
            "building": DEPTS.loc[c, "building"], "energy_kwh": SIM_E[c].values,
            "energy_without_actions_kwh": (SIM_E[c] + SIM_TRUE_SAVED[c]).round(1).values,
            "true_saving_kwh": SIM_TRUE_SAVED[c].round(1).values, "water_litres": SIM_W[c].values, "waste_kg": SIM_WASTE[c].values,
            "outdoor_temp_C": SIM_TEMP.values, "academic_period": SIM_CAL.period.values,
            "weekday": SIM_DAYS.day_name(), "holiday": SIM_CAL.holiday.values}))
    return pd.concat(rows, ignore_index=True)


DATA_DICTIONARY = pd.DataFrame([
    ("real/readings_daily_labelled.csv", "date / meter / resource", "one row per real Concordia meter per day", "REAL"),
    ("real/readings_daily_labelled.csv", "actual", "metered use that day (kWh, m³ or meter units)", "REAL"),
    ("real/readings_daily_labelled.csv", "expected", "ghost-twin prediction (weather + calendar adjusted baseline)", "MODEL"),
    ("real/readings_daily_labelled.csv", "data_fault", "why the reading was removed: impossible jump / not measuring / incomplete day", "LABEL"),
    ("real/readings_daily_labelled.csv", "alert", "id of the alert covering that day (blank = none)", "LABEL"),
    ("real/readings_daily_labelled.csv", "period", "baseline (Mar 2019–Feb 2020) or monitoring", "LABEL"),
    ("real/verification.csv", "verdict / 95% low / 95% high / p", "result of the ghost twin + placebo test for each logged action", "MODEL"),
    ("real/trust_test.csv", "naive_% / gl_% / gl_low / gl_high", "placebo months: simple before/after vs GreenLedger", "MODEL"),
    ("simulated/departments_daily.csv", "energy_kwh / water_litres / waste_kg", "simulated department readings per day", "SIMULATED"),
    ("simulated/departments_daily.csv", "energy_without_actions_kwh / true_saving_kwh", "the TRUE counterfactual — known only because it is a simulation", "SIMULATED (ground truth)"),
    ("simulated/departments_daily.csv", "academic_period / holiday / outdoor_temp_C", "calendar and weather that drive the simulation", "SIMULATED"),
    ("simulated/actions_truth_check.csv", "true effect % vs 95% low/high", "did the method recover the known truth?", "SIMULATED + MODEL"),
    ("simulated/ewaste_register.csv", "action / value_now / reusable_parts / when", "keep · sell · repair · harvest · recycle, with value and timing", "SIMULATED + MODEL"),
    ("simulated/money_saved_monthly.csv", "₹ by department", "verified energy saved (low end) × tariff", "MODEL"),
], columns=["file", "columns", "meaning", "type"])


def _money_monthly():
    cols = {}
    for r in DEPT_RESULTS.values():
        if r["verdict"] == "VERIFIED SAVING" and r["saving_%"] > 0:
            cols[r["id"]] = (r["ghost"] - r["actual"]).dropna() * (r["low_%"] / r["saving_%"])
    if not cols:
        return pd.DataFrame()
    d = pd.DataFrame(cols).fillna(0)
    by = d.T.groupby({r["id"]: DEPTS.loc[r["dept"], "name"] for r in DEPT_RESULTS.values()}).sum().T.resample("MS").sum()
    return (by * TARIFF_INR_PER_KWH).round(0)


# ---------- 4. the report (HTML, same look as the app) + the ZIP ----------


def build_report():
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    V = verification_table()
    zpath = os.path.join(OUT_DIR, f"greenledger_dataset_{stamp}.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for name, df in [("readings_daily_labelled.csv", labelled_dataset()), ("alerts.csv", ALERT_TABLE.drop(columns="point")),
                         ("discoveries.csv", DISCOVERY_TABLE.drop(columns="point")), ("data_faults.csv", FAULTS), ("actions.csv", ACTIONS),
                         ("verification.csv", V), ("model_accuracy.csv", ACC.drop(columns="point")), ("trust_test.csv", TRUST),
                         ("carbon_inventory.csv", CARBON), ("carbon_factors.csv", CARBON_FACTORS), ("parameters_and_sources.csv", PARAMETERS),
                         ("recommendations.csv", RECS.drop(columns=["_value", "_point"])), ("recommendation_decisions.csv", DECISIONS)]:
            z.writestr("real/" + name, df.to_csv(index=False))
        z.writestr("real/waste_monthly_tonnes.csv", WASTE_MONTHLY.to_csv())
        z.writestr("simulated/departments_daily.csv", sim_dataset().to_csv(index=False))
        z.writestr("simulated/department_summary.csv", DEPT_TABLE.to_csv())
        z.writestr("simulated/actions_truth_check.csv", SIM_CHECK.to_csv(index=False))
        z.writestr("simulated/ewaste_register.csv", EW_SIM.assign(data_type="SIMULATED").to_csv(index=False))
        z.writestr("simulated/money_saved_monthly.csv", _money_monthly().to_csv())
        if EWASTE is not None:
            z.writestr("your_data/ewaste_plan.csv", EWASTE.to_csv(index=False))
        if ROOM["log"]:
            z.writestr("live/room_control_log.csv", pd.DataFrame(ROOM["log"]).to_csv(index=False))
        z.writestr("DATA_DICTIONARY.csv", DATA_DICTIONARY.to_csv(index=False))
        z.writestr("README.txt",
                   "GreenLedger — labelled dataset\n\n"
                   "real/       Concordia University open data (Facilities), CC BY 4.0. Nothing simulated or filled in.\n"
                   "simulated/  A simulated 9-department Bengaluru campus. Every row says data_type=SIMULATED.\n"
                   "            It includes the TRUE effect of each action so anyone can check the method.\n"
                   "live/       The room-control log from this session (people counts only, no images).\n"
                   "DATA_DICTIONARY.csv explains every file and column.\n")
    tbl = lambda d: d.to_html(index=False, border=0, classes="t", na_rep="—")
    G, M = JUDGE_GUIDE, _money_monthly()
    ver = SIM_CHECK[SIM_CHECK.verdict == "VERIFIED SAVING"]
    html = f"""<!doctype html><html><head><meta charset="utf-8"><title>GreenLedger report</title><style>
body{{font-family:'DM Sans',Arial,sans-serif;max-width:1040px;margin:30px auto;padding:0 18px;color:#1f2a24;background:#f7f5ee;line-height:1.55}}
h1{{color:#24503f;margin-bottom:4px}} h2{{color:#24503f;border-bottom:2px solid #b7cc99;padding-bottom:4px;margin-top:34px}}
.t{{border-collapse:collapse;width:100%;font-size:12.5px;background:#fff}} .t td,.t th{{border-bottom:1px solid #e3e6da;padding:6px 8px;text-align:left;vertical-align:top}}
.t th{{background:#eef3e4}} .kpis{{display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin:16px 0}}
.k{{background:#fff;border:1px solid #e3e6da;border-radius:10px;padding:12px}} .k b{{display:block;font-size:22px;color:#24503f}}
.k.d{{background:#24503f;color:#fff}} .k.d b{{color:#cde46a}} .tag{{font-size:11px;font-weight:700;padding:2px 8px;border-radius:99px;background:#f6e7c8;color:#7a5414}}
.tag.r{{background:#e3eed5;color:#24503f}} .t{{font-variant-numeric:tabular-nums}} @media print{{body{{background:#fff}}}} .box{{background:#e9f0de;border:1px solid #b7cc99;border-radius:10px;padding:12px 16px}}</style></head><body>
<h1>GreenLedger — campus resource report</h1>
<div>Team {TEAM['name']} ({', '.join(TEAM['members'])}) · {TEAM['college']} · generated {datetime.datetime.now():%d %b %Y %H:%M}</div>
<div class="kpis"><div class="k d">Energy saved · {_m}<b>{DEPT_TABLE['Saved kWh last month'].sum():,.0f} kWh</b>verified, low end</div>
<div class="k">Money saved · {_m}<b>{_k(DEPT_TABLE['Saved ₹ last month'].sum())}</b>at ₹{TARIFF_INR_PER_KWH}/kWh</div>
<div class="k">CO₂ avoided · {_m}<b>{DEPT_TABLE['Saved kWh last month'].sum() * INDIA_GRID_KG_PER_KWH / 1000:,.1f} t</b>India grid (CEA)</div>
<div class="k">E-waste value<b>{_k(EW_SUMMARY['recoverable_value'])}</b>recoverable now</div></div>
<p>Since the first change: {DEPT_SAVED_KWH / 1000:,.1f} MWh saved · {_k(DEPT_SAVED_KWH * TARIFF_INR_PER_KWH)}. <i>{JUDGE_GUIDE['about_data']}</i></p>
<div class="box"><b>How to read it.</b> <b>Live twin</b> = what really happened (the meter). <b>Ghost twin</b> = what the same building would have used
without the action — a model fed the real weather and calendar. <b>Saving = ghost − live</b>, counted only when it is bigger than normal noise
(placebo test) and only at the low end of its 95% range.</div>
<h2>1 · Did the actions work?</h2>
<p>Each action had a known true effect, so the method can be checked: truth inside the range for <b>{VALIDATION['sim_inside']}/{VALIDATION['sim_n']}</b>,
never over-counted <b>{VALIDATION['sim_no_overcount']}/{VALIDATION['sim_n']}</b>.</p>{tbl(SIM_CHECK)}
<h2>2 · Departments — last 12 months</h2><div style="overflow-x:auto">{DEPT_TABLE.drop(columns=["Top suggestion", "Building"]).round(1).to_html(border=0, classes="t")}</div>
<h3>One suggestion per department</h3>{tbl(DEPT_TABLE.reset_index()[["Department", "Efficiency", "Top suggestion"]])}
<h2>3 · Money saved by month (₹)</h2>{M.reset_index().assign(index=lambda x: x["index"].dt.strftime("%b %Y")).rename(columns={"index": "month"}).to_html(index=False, border=0, classes="t") if len(M) else "<p>No verified saving yet.</p>"}
<h2>4 · E-waste</h2>
<p>{EW_SUMMARY['devices']:,} devices · sell second-hand now: {EW_SUMMARY['sell_now']} (≈ {_k(EW_SUMMARY['sell_value'])}; waiting a year loses ≈ {_k(EW_SUMMARY['lost_per_year'])}) ·
repair: {EW_SUMMARY['repair']} · harvest parts: {EW_SUMMARY['harvest']} (≈ {_k(EW_SUMMARY['parts_value'])}) · to a registered recycler: {EW_SUMMARY['recycler_kg']:,.0f} kg.</p>
{EW_BY_DEPT.reset_index().to_html(index=False, border=0, classes="t")}
<h2>5 · Can you trust it? (real university meters)</h2>
<p>On {VALIDATION['trust_tests']} real meter-months where nothing was done, a simple before/after dashboard reported a “saving” {VALIDATION['trust_naive']} times; GreenLedger certified {VALIDATION['trust_gl']}.
The COVID-19 closure is verified on {VALIDATION['covid_verified']}/{VALIDATION['covid_n']} electricity meters (a test of the engine — not counted as a saving).</p>
<h3>Action register — real meters</h3>{tbl(V[['action', 'date', 'meter', 'title', 'verdict', 'change %', '95% low', '95% high', 'p', 'naive %', 'counted', 'unit']])}
<h3>Alerts</h3>{tbl(ALERT_TABLE[['id', 'meter', 'from', 'to', 'extra use', 'what happened']])}
<h3>Recommendations (potential — verified only after they are done)</h3>{tbl(RECS[['id', 'priority', 'area', 'where', 'finding', 'potential (not verified)', 'next step']])}
<h3>Waste per 12-month period (tonnes)</h3>{WASTE_YEARLY.to_html(border=0, classes="t")}
<h3>Carbon inventory (t CO₂e)</h3>{CARBON_BY_PERIOD.reset_index().to_html(index=False, border=0, classes="t", na_rep="—")}{tbl(CARBON_FACTORS)}
<h3>Model accuracy</h3>{tbl(ACC.drop(columns="point"))}
<h2>6 · Settings and sources</h2>{tbl(pd.DataFrame(G['assumptions']))}
<h2>7 · Safety and privacy</h2><ul>{''.join(f'<li>{x}</li>' for x in G['safety'])}</ul>
<h2>8 · How we know it works — and the next step</h2><ul>{''.join(f'<li>{x}</li>' for x in G['validation_done'])}</ul><ol>{''.join(f'<li>{x}</li>' for x in G['validation_plan'])}</ol>
<h3>Measurable success criteria</h3>{tbl(pd.DataFrame(G['success_criteria']))}
<h2>9 · Every parameter</h2>{tbl(PARAMETERS)}{tbl(DATA_DICTIONARY)}
</body></html>"""
    hpath = os.path.join(OUT_DIR, f"greenledger_report_{stamp}.html")
    open(hpath, "w", encoding="utf-8").write(html)
    return hpath, zpath


REPORT_HTML, REPORT_ZIP = build_report()
print(f"Validation: truth inside range {VALIDATION['sim_inside']}/{VALIDATION['sim_n']} · never over-counted "
      f"{VALIDATION['sim_no_overcount']}/{VALIDATION['sim_n']} · false savings on real placebo months {VALIDATION['trust_gl']}/{VALIDATION['trust_tests']} "
      f"(simple dashboard {VALIDATION['trust_naive']}) · COVID verified on {VALIDATION['covid_verified']}/{VALIDATION['covid_n']} meters")
print("Full report:", REPORT_HTML)
print("Labelled dataset (real/ + simulated/ + data dictionary):", REPORT_ZIP)


## Cell 14 · The web app page
One HTML page in the style of your Lovable design. Chart, map and icon libraries are **saved by the notebook and served by GreenLedger itself**, so charts show even when the browser blocks CDNs. — cream background, dark-forest sidebar, DM Sans / Manrope, KPI cards with sparklines, an aerial campus twin and a dark *Campus insights* panel. It contains **no data**: every number is fetched from the API in the next cell.

In [ ]:
# ---- The web app page (one HTML file: Lovable-style layout, Chart.js charts, Leaflet aerial map) ----
# Fonts: DM Sans + Manrope · charts: Chart.js 4.4.1 · map: Leaflet 1.9.4 + Esri World Imagery · icons: Lucide.
# All numbers are fetched from the API in the next cell — nothing in this page is hard-coded data.
WEB_HTML = r'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>GreenLedger — Campus Resource Optimisation</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Manrope:wght@400;500;600;700;800&display=swap" rel="stylesheet">
<link rel="stylesheet" href="static/leaflet.css">
<script src="static/chart.umd.js"></script>
<script src="static/leaflet.js"></script>
<script src="static/lucide.min.js"></script>
<style>
:root{--radius:10px;--bg:oklch(96.2% .017 99);--ivory:oklch(98.4% .007 99);--fg:oklch(29% .029 159);--forest:oklch(33.4% .051 173);
--forest-dark:oklch(29.1% .044 174);--sage:oklch(86.5% .047 127);--pale-sage:oklch(92.8% .026 123);--lime:oklch(91.9% .177 117);
--muted:oklch(55.8% .027 148);--border:oklch(89.3% .022 126);--amber:oklch(51% .095 77);--coral:oklch(50% .13 30);
--sb-active:oklch(41.4% .05 171);--shadow:0 3px 10px oklch(29% .029 159/.04)}
*{box-sizing:border-box}html,body{margin:0;height:100%}
body{background:var(--bg);color:var(--fg);font-family:'DM Sans',system-ui,sans-serif;font-size:14px}
h1,h2,h3,.disp{font-family:Manrope,'DM Sans',sans-serif;letter-spacing:-.02em;margin:0}
a{color:inherit}
.app{display:grid;grid-template-columns:250px 1fr;min-height:100vh}
/* sidebar */
.sb{background:var(--forest);color:#fff;display:flex;flex-direction:column;position:sticky;top:0;height:100vh;padding:18px 14px}
.brand{display:flex;align-items:center;gap:10px;padding:6px 8px 22px;font:800 22px Manrope,system-ui,sans-serif;letter-spacing:-.02em}
.brand .dot{width:6px;height:6px;border-radius:50%;background:var(--lime);margin:-14px 0 0 -6px}
.sb .lbl{font:600 10.5px 'DM Sans',system-ui,sans-serif;letter-spacing:.14em;opacity:.7;padding:8px 10px}
.nav a{display:flex;align-items:center;gap:11px;padding:9px 11px;border-radius:8px;text-decoration:none;font-weight:500;font-size:13.5px;color:#e9efe9;margin:1px 0;cursor:pointer}
.nav a:hover{background:oklch(38% .05 172)}
.nav a.on{background:var(--sb-active);color:var(--lime);box-shadow:inset 0 0 0 1px oklch(48% .05 171)}
.nav a svg{width:16px;height:16px}
.nav .grp{font:600 10px 'DM Sans',system-ui,sans-serif;letter-spacing:.14em;opacity:.55;padding:14px 11px 4px}
.sbfoot{margin-top:auto;border-top:1px solid oklch(42% .045 172);padding:14px 8px 4px;font-size:12px;line-height:1.5}
.sbfoot b{display:flex;align-items:center;gap:8px;font-size:13px}.sbfoot b:before{content:"";width:8px;height:8px;border-radius:50%;background:var(--lime)}
.sbfoot .s{opacity:.8}.sbfoot .t{opacity:.55;margin-top:10px;font-size:11px}
/* top bar */
.main{min-width:0}
.top{display:flex;align-items:center;gap:14px;padding:12px 28px;border-bottom:1px solid var(--border);background:var(--bg);position:sticky;top:0;z-index:500}
.top .ttl{font:600 15px Manrope,system-ui,sans-serif;margin-right:auto}
.chip{display:inline-flex;align-items:center;gap:6px;border:1px solid var(--border);background:var(--ivory);border-radius:8px;padding:6px 10px;font-size:12.5px;font-weight:500}
.search{display:flex;align-items:center;gap:8px;border-left:1px solid var(--border);padding-left:14px;color:var(--muted)}
.search input{border:0;background:transparent;font:inherit;color:var(--fg);outline:none;width:150px}
.seg{display:inline-flex;border:1px solid var(--border);border-radius:9px;padding:3px;background:var(--ivory)}
.seg button{border:0;background:transparent;padding:6px 12px;border-radius:7px;font:500 12.5px 'DM Sans',system-ui,sans-serif;color:var(--fg);cursor:pointer}
.seg button.on{background:var(--forest);color:#fff}
.bell{position:relative}.bell i{position:absolute;top:-2px;right:-2px;width:7px;height:7px;border-radius:50%;background:var(--coral)}
.av{width:30px;height:30px;border-radius:50%;background:var(--sage);display:flex;align-items:center;justify-content:center;font:700 11px 'DM Sans',system-ui,sans-serif}
.page{padding:22px 28px 40px;max-width:1500px}
.ph{display:flex;align-items:center;gap:12px;margin-bottom:16px}
.ph .back{display:inline-flex;align-items:center;gap:5px;border:1px solid var(--border);border-radius:7px;padding:4px 9px;font-size:12px;background:var(--ivory);cursor:pointer;color:var(--muted)}
.ph h1{font-size:26px;font-weight:600}.ph .tag{border:1px solid var(--sage);color:var(--forest);background:var(--pale-sage);border-radius:999px;padding:2px 9px;font-size:11px;font-weight:600}
.eyebrow{font:600 10.5px 'DM Sans',system-ui,sans-serif;letter-spacing:.14em;text-transform:uppercase;color:var(--muted);display:flex;align-items:center;gap:6px}
.sub{color:var(--muted);font-size:13px;margin:6px 0 18px}
.hero{display:flex;justify-content:space-between;align-items:flex-end;margin-bottom:18px}
.hero h1{font-size:34px;font-weight:600;margin:6px 0 4px}
.btn{display:inline-flex;align-items:center;gap:7px;border:1px solid var(--border);background:var(--ivory);border-radius:8px;padding:8px 13px;font:600 13px 'DM Sans',system-ui,sans-serif;cursor:pointer;color:var(--fg);text-decoration:none}
.btn.pri{background:var(--forest);color:#fff;border-color:var(--forest)}.btn.pri:hover{background:var(--forest-dark)}
.btn.sm{padding:5px 10px;font-size:12px}.btn svg{width:15px;height:15px}
.btn.warn{color:var(--coral)}
/* cards */
.grid{display:grid;gap:14px}.g4{grid-template-columns:repeat(4,minmax(0,1fr))}.g3{grid-template-columns:repeat(3,minmax(0,1fr))}.g2{grid-template-columns:repeat(2,minmax(0,1fr))}
.g5{grid-template-columns:repeat(5,minmax(0,1fr))}
.split{display:grid;grid-template-columns:minmax(0,1fr) 340px;gap:16px;align-items:start}
.card{background:var(--ivory);border:1px solid var(--border);border-radius:var(--radius);box-shadow:var(--shadow);padding:16px 18px}
.card.dark{background:var(--forest);color:#fff;border-color:var(--forest)}
.card.sage{background:var(--pale-sage);border-color:var(--sage)}
.card h3{font-size:15px;font-weight:600}
.kpi .row{display:flex;justify-content:space-between;align-items:center}.kpi .lab{font-weight:600;font-size:13px}
.kpi .v{font:700 30px Manrope,system-ui,sans-serif;letter-spacing:-.03em;margin:14px 0 2px}.kpi .v small{font:500 13px 'DM Sans',system-ui,sans-serif;margin-left:6px;opacity:.7}
.kpi .d{font-size:11.5px;display:flex;align-items:center;gap:5px;opacity:.85}.kpi .d b{font-weight:700}
.kpi canvas{height:46px!important;margin-top:10px}.kpi .foot{font-size:10.5px;opacity:.6;margin-top:4px}
.ct{font:600 10.5px 'DM Sans',system-ui,sans-serif;letter-spacing:.14em;text-transform:uppercase;color:var(--muted);margin-bottom:10px}
.dark .ct{color:var(--sage)}
.pill{display:inline-flex;align-items:center;gap:5px;border:1px solid var(--border);background:var(--bg);border-radius:7px;padding:4px 9px;font-size:12px;font-weight:500;cursor:pointer}
.pill.on{background:var(--pale-sage);border-color:var(--sage)}.pill svg{width:13px;height:13px}
.status{font-size:10.5px;font-weight:600;border-radius:999px;padding:2px 9px;border:1px solid}
.st-Optimal{color:var(--forest);background:var(--pale-sage);border-color:var(--sage)}
.st-Warning{color:var(--amber);background:oklch(95% .04 85);border-color:oklch(86% .07 85)}
.st-Critical{color:var(--coral);background:oklch(95% .03 30);border-color:oklch(85% .07 30)}
.st-Unrated,.st-No{color:var(--muted);background:var(--bg);border-color:var(--border)}
.kv{display:grid;grid-template-columns:auto 1fr;gap:5px 14px;font-size:12.5px;margin-top:10px}.kv span{color:var(--muted)}.kv b{font-weight:600;text-align:right}
/* insights dark panel */
.ins{background:var(--forest);color:#fff;border-radius:var(--radius);padding:18px}
.ins .ring{display:flex;align-items:center;gap:14px;margin:8px 0 14px}
.ring svg{width:78px;height:78px}.ins .item{border-top:1px solid oklch(42% .045 172);padding:11px 0}
.ins .k{font:600 10px 'DM Sans',system-ui,sans-serif;letter-spacing:.12em;color:var(--sage);text-transform:uppercase}.ins .n{font:700 15px Manrope,system-ui,sans-serif;margin-top:3px}
.ins .s{font-size:11.5px;opacity:.75}.ins .opp{display:flex;gap:9px;margin:8px 0;font-size:12.5px;font-weight:600}
.ins .opp:before{content:"";flex:0 0 6px;height:6px;border-radius:50%;background:var(--coral);margin-top:6px}
.ins a.more{display:block;text-align:center;border-top:1px solid oklch(42% .045 172);padding-top:10px;margin-top:6px;font-size:12px;text-decoration:none;opacity:.85}
/* map */
.mapwrap{position:relative;border-radius:8px;overflow:hidden;border:1px solid var(--border)}
#map,.map{height:520px;background:#1d2a24}
.mapfoot{display:flex;justify-content:space-between;font-size:11px;color:var(--muted);padding:8px 2px 0}
.lbl-pin{background:#fff;color:var(--fg);border-radius:6px;padding:3px 7px;font:600 11px 'DM Sans',system-ui,sans-serif;box-shadow:0 2px 6px #0003;white-space:nowrap}
.pin{width:30px;height:30px;border-radius:50%;background:var(--forest);border:2px solid #fff;display:flex;align-items:center;justify-content:center;color:#fff;box-shadow:0 3px 8px #0005}
.pin svg{width:15px;height:15px}.pin.Warning{background:var(--amber)}.pin.Critical{background:var(--coral)}.pin.No,.pin.Unrated{background:#56645c}
.leaflet-popup-content{font-family:'DM Sans',system-ui,sans-serif;font-size:12.5px}
.leaflet-tooltip{background:transparent;border:0;box-shadow:none;padding:0}.leaflet-tooltip:before{display:none}
.far .leaflet-tooltip{display:none}
/* tables */
table.t{width:100%;border-collapse:collapse;font-size:12.5px}table.t th{text-align:left;font:600 10.5px 'DM Sans',system-ui,sans-serif;letter-spacing:.1em;text-transform:uppercase;color:var(--muted);border-bottom:1px solid var(--border);padding:8px}
table.t td{border-bottom:1px solid var(--border);padding:8px;vertical-align:top}
.scroll{max-height:420px;overflow:auto}
.verdict{border-radius:var(--radius);padding:16px 18px;border:1px solid}
.v-ok{background:var(--pale-sage);border-color:var(--sage)}.v-no{background:oklch(96% .03 85);border-color:oklch(86% .07 85)}
.v-bad{background:oklch(95% .03 30);border-color:oklch(85% .07 30)}.v-wait{background:var(--ivory);border-color:var(--border)}
.verdict .big{font:700 22px Manrope,system-ui,sans-serif}.verdict p{margin:8px 0 0;line-height:1.6;font-size:13px}
.stat{font:700 24px Manrope,system-ui,sans-serif}.stat+div{font-size:11.5px;color:var(--muted)}
.rec{background:var(--ivory);border:1px solid var(--border);border-radius:var(--radius);margin-bottom:10px}
.rec .h{display:flex;align-items:center;gap:14px;padding:14px 16px;cursor:pointer}
.rec .ic{width:36px;height:36px;border-radius:8px;background:var(--pale-sage);display:flex;align-items:center;justify-content:center;flex:0 0 36px}
.rec .ic svg{width:17px;height:17px}.rec .t1{font:600 10px 'DM Sans',system-ui,sans-serif;letter-spacing:.12em;color:var(--muted);text-transform:uppercase}
.rec .t2{font:600 15px Manrope,system-ui,sans-serif;margin:2px 0}.rec .t3{font-size:12px;color:var(--muted)}.rec .body{padding:0 16px 16px 66px;display:none;line-height:1.6}
.rec.open .body{display:block}
.field{display:flex;flex-direction:column;gap:4px;font-size:11.5px;color:var(--muted);font-weight:600}
.field input,.field select,textarea{border:1px solid var(--border);background:#fff;border-radius:7px;padding:7px 9px;font:500 13px 'DM Sans',system-ui,sans-serif;color:var(--fg)}
.row{display:flex;gap:10px;align-items:flex-end;flex-wrap:wrap}
.note{font-size:12px;color:var(--muted);line-height:1.55}
.msg{font-size:12.5px;font-weight:600;color:var(--forest);margin-top:8px}.msg.err{color:var(--coral)}
.dev{display:flex;align-items:center;gap:12px}.dev .sw{width:42px;height:24px;border-radius:999px;background:#cfd6cf;position:relative;transition:.2s}
.dev .sw:after{content:"";position:absolute;top:3px;left:3px;width:18px;height:18px;border-radius:50%;background:#fff;transition:.2s}
.dev .sw.on{background:var(--forest)}.dev .sw.on:after{left:21px}
.cam{background:#111;border-radius:8px;width:100%;aspect-ratio:4/3;object-fit:cover}
input[type=range]{width:100%;accent-color:oklch(33.4% .051 173)}
.badge{font-size:10.5px;font-weight:700;border-radius:999px;padding:2px 8px}
.b-open{background:oklch(95% .04 85);color:var(--amber)}.b-approved{background:var(--pale-sage);color:var(--forest)}.b-rejected{background:oklch(95% .03 30);color:var(--coral)}
.hide{display:none}
@media(max-width:1100px){.g4,.g5{grid-template-columns:repeat(2,minmax(0,1fr))}.split{grid-template-columns:1fr}.app{grid-template-columns:1fr}.sb{position:relative;height:auto}}

.tag.sim{border-color:oklch(86% .07 85);color:var(--amber);background:oklch(96% .04 85)}
.hero2{display:grid;grid-template-columns:minmax(0,1.6fr) minmax(0,1fr);gap:14px}
.big1{font:800 52px Manrope,system-ui,sans-serif;letter-spacing:-.04em;line-height:1;margin:10px 0 4px;color:var(--lime)}
.big1 small{font:600 18px 'DM Sans',system-ui,sans-serif;color:#cfe0c9;margin-left:6px}
.trio{display:flex;gap:26px;margin-top:14px;flex-wrap:wrap}.trio div{font:700 20px Manrope,system-ui,sans-serif}.trio span{display:block;font:500 11.5px 'DM Sans',system-ui,sans-serif;color:#cfe0c9}
.twin-img{position:relative;border-radius:10px;overflow:hidden;border:1px solid var(--border)}
.twin-img img{width:100%;display:block}
.tpin{position:absolute;transform:translate(-50%,-50%);display:flex;flex-direction:column;align-items:center;gap:4px;cursor:pointer}
.tpin .dot{width:34px;height:34px;border-radius:50%;border:2px solid #fff;display:flex;align-items:center;justify-content:center;color:#fff;box-shadow:0 3px 10px #0006;background:var(--forest)}
.tpin .dot svg{width:16px;height:16px}.tpin .lb{background:#fff;border-radius:6px;padding:3px 8px;font:600 11px 'DM Sans',system-ui,sans-serif;box-shadow:0 2px 6px #0004;white-space:nowrap}
.e-Efficient{background:var(--forest)!important}.e-Average{background:var(--amber)!important}.e-High{background:var(--coral)!important}.e-No{background:#56645c!important}
.chips{display:flex;gap:6px;flex-wrap:wrap;margin-bottom:14px}
.vbadge{font-size:11px;font-weight:700;border-radius:999px;padding:3px 9px}
.vb-ok{background:var(--pale-sage);color:var(--forest)}.vb-no{background:oklch(96% .04 85);color:var(--amber)}
.acts{display:grid;gap:8px}.acts .a{display:flex;justify-content:space-between;align-items:center;gap:10px;padding:10px 12px;border:1px solid var(--border);border-radius:8px;background:#fff;cursor:pointer}
.acts .a.on{border-color:var(--forest);box-shadow:inset 3px 0 0 var(--forest)}
.legend-note{font-size:11px;color:var(--muted);margin-top:6px}
.btn{white-space:nowrap}
.mt{display:inline-block;font:700 9.5px 'DM Sans',system-ui,sans-serif;letter-spacing:.06em;text-transform:uppercase;border-radius:999px;padding:2px 7px;margin-left:6px;vertical-align:middle}
.mt.s{background:#f6e7c8;color:#7a5414;border:1px solid #e3c98f}.mt.r{background:var(--pale-sage);color:var(--forest);border:1px solid var(--sage)}
.card.dark .mt.s{background:#3b4a2a;color:#e8f0b0;border-color:#5d6d3c}
.explain{background:var(--pale-sage);border:1px solid var(--sage);border-radius:var(--radius);padding:12px 16px;margin:-6px 0 16px;font-size:13px;line-height:1.55}
.explain b.h{display:block;font:700 11px 'DM Sans',system-ui,sans-serif;letter-spacing:.08em;text-transform:uppercase;color:var(--forest);margin-bottom:4px}
.explain ul{margin:0;padding-left:18px}.explain li{margin:2px 0}
.twins{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:10px;margin:0 0 16px}
.twins>div{background:var(--card);border:1px solid var(--border);border-radius:var(--radius);padding:10px 14px;font-size:12.5px;color:var(--muted);line-height:1.45}
.twins b{display:flex;align-items:center;gap:8px;color:var(--fg);font-size:13.5px;margin-bottom:3px}
.twins i.ln{display:inline-block;width:26px;height:0;border-top:3px solid var(--forest)}.twins i.ln.g{border-top:3px dashed #97a59c}.twins i.ln.s{border:0;height:12px;background:rgba(205,228,106,.6);border-radius:3px}
@media(max-width:800px){.twins{grid-template-columns:1fr}}
.tpin.lm .dot{width:22px;height:22px;background:#56645c;opacity:.85}.tpin.lm .dot svg{width:11px;height:11px}.tpin.lm .lb{opacity:.8;font-weight:500}
.tpin .lb small{display:block;font-weight:500;color:var(--muted);text-align:center}
.tpin:hover .dot{transform:scale(1.15)}.tpin .dot{transition:transform .15s}.tpin.sel .dot{outline:3px solid var(--lime);outline-offset:2px}.tpin.sel .lb{background:var(--forest);color:#fff}.tpin.sel .lb small{color:#cfe0c9}
.twin-img.compact .lb{font-size:10px;padding:2px 6px}.twin-img.compact .tpin .dot{width:26px;height:26px}.twin-img.compact .tpin .dot svg{width:12px;height:12px}
.mleg{display:flex;gap:12px;flex-wrap:wrap;font-size:11.5px;color:var(--muted);margin-top:10px}.mleg span{display:inline-flex;align-items:center;gap:5px}.mleg i{width:11px;height:11px;border-radius:50%;display:inline-block}
details summary{cursor:pointer;font-weight:600;color:var(--forest);margin:6px 0}
</style>
</head>
<body>
<div class="app">
<aside class="sb">
  <div class="brand"><i data-lucide="leaf" style="color:var(--lime)"></i>GreenLedger<span class="dot"></span></div>
  <div class="lbl">WORKSPACE</div>
  <nav class="nav" id="nav"></nav>
  <div class="sbfoot"><b>Campus resource optimisation</b><div class="s">Team Shenzein · Anup Batakurki, Het Balchandani, Kamaal</div>
  <div class="t">IC26-05-01 · Climate Action</div></div>
</aside>
<div class="main">
  <div class="top"><div class="ttl" id="ttl">Dashboard</div>
    <span class="chip"><i data-lucide="building-2" style="width:14px"></i>University campus</span>
    <div class="av">GL</div>
  </div>
  <div class="page" id="page">Loading…</div>
</div>
</div>
<script>
const NAV=[["dashboard","Dashboard","layout-dashboard"],["departments","Departments","building-2"],["twin","Campus Digital Twin","map"],
["prove","Live vs Ghost Twin","git-compare-arrows"],["room","Live Room Control","video"],["ewaste","E-waste","cpu"],
["carbon","Carbon Footprint","cloud"],["recs","Recommendations","lightbulb"],["reports","Reports & Data","file-text"],["guide","Judge's Guide","badge-check"],["help","Support & Info","life-buoy"]];
const $=s=>document.querySelector(s), page=()=>$("#page");
const api=(u,o)=>fetch(u.replace(/^\//,""),o).then(r=>{if(!r.ok)throw new Error(r.status+" "+u);return r.json()});
const post=(u,b)=>api(u,{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify(b||{})});
const ok=v=>v!=null&&!isNaN(v);
const f0=v=>ok(v)?Math.round(v).toLocaleString("en-IN"):"—";
const f1=v=>ok(v)?(+v).toLocaleString("en-IN",{maximumFractionDigits:1,minimumFractionDigits:1}):"—";
const inr=v=>!ok(v)?"—":Math.abs(v)>=1000?"₹"+Math.round(v/1000).toLocaleString("en-IN")+"K":"₹"+f0(v);
const esc=s=>String(s??"").replace(/[&<>"]/g,c=>({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));
const C={forest:"#24503f",sage:"#b7cc99",lime:"#cde46a",amber:"#9a6b1f",coral:"#ad4530",muted:"#7f8f86",grid:"#e3e6da",water:"#3a7a8a",ghost:"#97a59c"};
const DEPC=["#24503f","#3a7a8a","#9a6b1f","#ad4530","#6b8f3a","#5b6fa8","#8a5a9a","#b08a2e","#4f7f6a","#c0763a","#2f6f8f","#7a8f2a","#a05a7a","#5a7a4a","#8f6f4f","#3f5f9f"];
const HAS_CHART=typeof Chart!="undefined";if(HAS_CHART){Chart.defaults.animation=false;Chart.defaults.font.family="DM Sans, system-ui, sans-serif"}
if(HAS_CHART){Chart.defaults.font.family="'DM Sans',system-ui,sans-serif";Chart.defaults.color="#6d7c73";Chart.defaults.font.size=11;Chart.defaults.maintainAspectRatio=false}
let charts=[],MAP=null;
function kill(){charts.forEach(c=>c.destroy());charts=[];if(MAP){MAP.remove();MAP=null}stopCam();if(WTIMER){clearInterval(WTIMER);WTIMER=null}}
function chart(el,cfg){if(!HAS_CHART){const c=typeof el=="string"?$(el):el;if(c)c.parentNode.innerHTML='<div class="note">Charts could not load (Chart.js missing).</div>';return null}
 const c=new Chart(typeof el=="string"?$(el):el,cfg);charts.push(c);return c}
function drop(id){const o=charts.find(c=>c.canvas.id==id);if(o){o.destroy();charts=charts.filter(c=>c!==o)}}
function icons(){try{lucide.createIcons()}catch(e){}}
const SIM='',REAL='',MS='',MR='';
function head(icon,title,tag,sub){return `<div class="ph"><i data-lucide="${icon}" style="width:20px"></i><h1>${title}</h1>${tag||""}</div>${sub?`<div class="sub">${sub}</div>`:""}`}
const axes=yt=>({x:{grid:{display:false},ticks:{maxTicksLimit:12}},y:{beginAtZero:false,grid:{color:C.grid},title:{display:!!yt,text:yt}}});
function spark(el,data,dark){chart(el,{type:"line",data:{labels:data.map((_,i)=>i),datasets:[{data,borderColor:dark?C.lime:C.forest,borderWidth:1.8,pointRadius:0,tension:.35,fill:true,backgroundColor:dark?"rgba(205,228,106,.12)":"rgba(36,80,63,.08)"}]},
options:{plugins:{legend:{display:false},tooltip:{enabled:false}},scales:{x:{display:false},y:{display:false}}}})}
function card(t,v,s,cls){return `<div class="card ${cls||""}"><div class="ct">${t}</div><div class="stat">${v}</div><div class="note">${s||""}</div></div>`}
function twinChart(id,lab,ghost,actual,opt){opt=opt||{};const ds=[];
 if(opt.hi){ds.push({label:"hi",data:opt.hi,borderWidth:0,pointRadius:0,fill:false});ds.push({label:"Noise range (placebo 95%)",data:opt.lo,borderWidth:0,pointRadius:0,fill:"-1",backgroundColor:"rgba(151,165,156,.22)"})}
 ds.push({label:"Ghost twin (without the action)",data:ghost,borderColor:C.ghost,borderDash:[6,4],borderWidth:2,pointRadius:0});
 ds.push({label:"Live twin (what really happened)",data:actual,borderColor:C.forest,borderWidth:2.6,pointRadius:0,fill:opt.fill?"-1":false,backgroundColor:"rgba(205,228,106,.35)"});
 if(opt.cum)ds.push({label:"Saved so far",data:opt.cum,borderColor:C.amber,borderWidth:2,pointRadius:0,yAxisID:"y2"});
 return chart(id,{type:"line",data:{labels:lab,datasets:ds},options:{interaction:{mode:"index",intersect:false},plugins:{legend:{labels:{filter:i=>i.text!="hi",boxWidth:14}}},
 scales:{x:{grid:{display:false},ticks:{maxTicksLimit:10}},y:{grid:{color:C.grid},title:{display:true,text:opt.unit||""}},...(opt.cum?{y2:{position:"right",grid:{display:false},title:{display:true,text:opt.cumUnit||""}}}:{})}}})}
const EXPL={
dashboard:["The big green number is the energy GreenLedger has <b>proven</b> was saved — counted at the low end of its range, so it is never over-stated. The money chart shows what those savings are worth in rupees, month by month.","The cards show the whole campus over the last 12 months: energy, water, waste and the value of old electronics.","Money is shown per month, in thousands of rupees (₹K). Open <b>Support &amp; Info</b> for what every word means."],
departments:["Click a block on the campus photo (or a name below) to open that department.","Colours compare departments per m²: <b>green</b> = efficient, <b>amber</b> = average, <b>red</b> = high use.","“Saved” counts only changes that passed the ghost-twin test. Money is per month, in ₹ thousands (₹K)."],
twin:["An interactive map of the campus. Choose what to show — energy use, energy saved, money saved, water or waste — and every block re-colours.","<b>Green</b> = best on that measure, <b>amber</b> = middle, <b>red</b> = needs attention, <b>grey</b> = no saving proven yet. Click any block for its details."],
prove:["Pick an action on the left. The chart compares the building <b>with</b> the action (live twin) and <b>without</b> it (ghost twin).","The grey band is normal day-to-day noise, found by the placebo test: we pretend the action happened on dates when nothing was done. A saving counts only when it is clearly outside that band.","The money chart turns the gap into rupees: what was measured, and the lower amount GreenLedger counts."],
room:["The camera counts people; GreenLedger switches the fan, lights and AC from that count and the live weather.","<b>Automatic</b> = settings worked out from the live weather. <b>Manual</b> = your own numbers, or tap any device to switch it by hand.","Live room = what GreenLedger does. Ghost room = the same room where nobody switches anything off. The gap is the energy saved."],
ewaste:["Every device gets one instruction: keep using it, sell it second-hand, repair it, take out the reusable parts, or send it to a registered recycler.","“When to sell” shows how many working devices reach their best resale moment each month — a year before their end of life, while still worth the most."],
carbon:["CO₂ = electricity used × 0.71 kg per kWh (India's grid, Central Electricity Authority).","“Avoided” counts only savings that passed the ghost-twin test."],
recs:["One suggestion per department, picked from the measure where it is furthest from the campus median.","Reference-campus actions use real data: approve one, do it, and GreenLedger verifies whether it worked."],
reports:["Download the report and the data behind every number.","The box below says where the data comes from and where every setting comes from."],
guide:["Every line of the brief, ticked, with a “Show me” button that jumps to it in the app.","The four numbers at the top show how we checked that GreenLedger tells the truth."],
help:["Start with the 5 steps. Stuck on a word? It is explained in “Words used in this app”.","Settings, safety and where the data comes from are at the bottom."]};
function explain(k){return `<div class="explain"><b class="h">How to read this page</b><ul>${EXPL[k].map(t=>`<li>${t}</li>`).join("")}</ul></div>`}
const TWINS=`<div class="twins"><div><b><i class="ln"></i>Live twin</b>What really happened — the meter reading with the action in place.</div><div><b><i class="ln g"></i>Ghost twin</b>What the same building would have used <i>without</i> the action: an AI model fed the real weather and calendar.</div><div><b><i class="ln s"></i>Saving = ghost − live</b>Counted only when the gap is bigger than normal noise — and only its low end.</div></div>`;
const MET={energy:{label:"Energy use",key:"kwh_m2",fmt:v=>f0(v)+" kWh/m²",good:"low"},saved:{label:"Energy saved",key:"saved_mwh",fmt:v=>f1(v)+" MWh",good:"high"},
money:{label:"Money saved / month",key:"saved_inr",fmt:v=>inr(v),good:"high"},water:{label:"Water",key:"water",fmt:v=>f0(v)+" L/person/day",good:"low"},waste:{label:"Waste diverted",key:"diversion",fmt:v=>f0(v)+"%",good:"high"}};
function metColor(pins,mk){const M=MET[mk],vals=Object.values(pins).filter(p=>p.m).map(p=>p.m[M.key]).sort((a,b)=>a-b),q1=vals[Math.floor(vals.length/3)],q2=vals[Math.floor(2*vals.length/3)];
 return p=>{if(!p.m)return "#56645c";const v=p.m[M.key];if((mk=="saved"||mk=="money")&&v<=0)return "#8e9a92";const r=v<q1?0:v<q2?1:2;return [C.forest,C.amber,C.coral][M.good=="low"?r:2-r]}}
function campusMap(pins,mk,sel,compact){const col=metColor(pins,mk),M=MET[mk];
 return `<div class="twin-img ${compact?"compact":""}"><img src="static/campus.jpg" alt="Campus aerial view">${Object.entries(pins).map(([b,p])=>`<div class="tpin ${p.m?"":"lm"} ${p.depts[0]&&p.depts[0]==sel?"sel":""}" style="left:${p.x}%;top:${p.y}%" data-b="${esc(b)}" data-c="${esc(p.depts[0]||"")}" title="${esc(p.m?p.m.name+" — "+M.label+": "+M.fmt(p.m[M.key]):b)}"><div class="dot" style="background:${col(p)}"><i data-lucide="${p.m?"zap":"building"}"></i></div><div class="lb">${esc(b)}${p.m&&!compact?`<small>${M.fmt(p.m[M.key])}</small>`:""}</div></div>`).join("")}</div>
<div class="mleg"><span><i style="background:${C.forest}"></i>best</span><span><i style="background:${C.amber}"></i>middle</span><span><i style="background:${C.coral}"></i>needs attention</span>${mk=="saved"||mk=="money"?`<span><i style="background:#8e9a92"></i>no saving proven yet</span>`:""}<span><i style="background:#56645c"></i>no department data</span></div>`}
function savingsChart(el,S){const s=S.filter(x=>x.true_mwh>0||x.mwh>0).concat(S.filter(x=>!(x.true_mwh>0||x.mwh>0)));
 return chart(el,{type:"bar",data:{labels:s.map(x=>x.name),datasets:[{label:"Counted (verified, low end)",data:s.map(x=>x.mwh),backgroundColor:C.forest,borderRadius:4},{label:"True saving",data:s.map(x=>x.true_mwh),backgroundColor:C.sage,borderRadius:4}]},
 options:{indexAxis:"y",plugins:{legend:{position:"bottom",labels:{boxWidth:11}},tooltip:{callbacks:{label:c=>" "+c.dataset.label+": "+f1(c.raw)+" MWh"+(c.datasetIndex==0?" · "+inr(s[c.dataIndex].inr):"")}}},scales:{x:{grid:{color:C.grid},title:{display:true,text:"MWh saved"}},y:{grid:{display:false},ticks:{autoSkip:false,font:{size:11}}}}}})}
function moneyChart(el,M){const names=M.names||{};const ds=Object.entries(M.by_dept).map(([c,v],i)=>({type:"bar",label:names[c]||c,data:v,backgroundColor:DEPC[Object.keys(names).indexOf(c)%16],stack:"s",yAxisID:"y"}));
 ds.push({type:"line",label:"Running total (₹K)",data:M.cum_k,borderColor:C.forest,borderWidth:2.5,pointRadius:2,yAxisID:"y2",tension:.2});
 return chart(el,{data:{labels:M.months,datasets:ds},options:{interaction:{mode:"index",intersect:false},plugins:{legend:{position:"bottom",labels:{boxWidth:11}},tooltip:{callbacks:{label:c=>c.dataset.yAxisID=="y2"?" Total so far: ₹"+f0(c.raw)+"K":" "+c.dataset.label+": "+inr(c.raw)}}},
 scales:{x:{stacked:true,grid:{display:false}},y:{stacked:true,grid:{color:C.grid},ticks:{callback:v=>"₹"+Math.round(v/1000)+"K"},title:{display:true,text:"₹ saved per month"}},y2:{position:"right",grid:{display:false},beginAtZero:true,title:{display:true,text:"₹K, running total"}}}}})}
let SELDEPT=null;
const P={};
// ================= DASHBOARD =================
P.dashboard=async()=>{const d=await api("api/dashboard");const h=d.hero,k=d.kpis,T=d.trust;
page().innerHTML=`<div class="hero"><div><div class="eyebrow"><i data-lucide="leaf" style="width:13px"></i>Campus resource optimisation</div><h1>Dashboard</h1></div>${SIM}</div>${explain("dashboard")}
<div class="hero2"><div class="card dark"><div class="ct">Energy saved last month · ${h.month} · verified by the ghost twin</div><div class="big1">${f0(h.kwh_m)}<small>kWh</small></div>
<div class="trio"><div>${inr(h.inr_m)}<span>saved in ${h.month}</span></div><div>${f0(h.co2_m_kg)} kg<span>CO₂ avoided</span></div><div>${h.verified} of ${h.actions}<span>changes verified</span></div></div>
<div class="note" style="color:#cfe0c9;margin-top:12px">Since the first change: ${f1(h.mwh)} MWh · ${inr(h.inr)} saved at ₹${h.tariff}/kWh. Counted at the safe, low end only — we never over-count.</div></div>
<div class="card"><div class="ct">Can you trust it? ${MR}</div><div style="height:120px"><canvas id="tr"></canvas></div><div class="note">Months where nothing was done (Concordia University meters): a simple dashboard reported a “saving” ${T.naive_saving} of ${T.tests} times; GreenLedger ${T.gl_saving}.</div></div></div>
<div class="grid g4" style="margin-top:14px">
<div class="card kpi"><div class="row"><span class="lab">Energy (12 months)${MS}</span><i data-lucide="zap" style="width:17px"></i></div><div class="v">${f0(k.energy_mwh)}<small>MWh</small></div><div class="d">${f0(k.co2_t)} t CO₂</div><canvas id="s1"></canvas></div>
<div class="card kpi"><div class="row"><span class="lab">Water${MS}</span><i data-lucide="droplet" style="width:17px"></i></div><div class="v">${f0(k.water_kl_day)}<small>kL/day</small></div><div class="d">average, last 12 months</div><canvas id="s2"></canvas></div>
<div class="card kpi sage"><div class="row"><span class="lab">Waste diverted${MS}</span><i data-lucide="recycle" style="width:17px"></i></div><div class="v">${f0(k.diversion)}<small>%</small></div><div class="d">recycled or composted</div><canvas id="s3"></canvas></div>
<div class="card kpi"><div class="row"><span class="lab">E-waste value${MS}</span><i data-lucide="cpu" style="width:17px"></i></div><div class="v">${inr(k.ewaste_inr)}</div><div class="d">recoverable now: resale + parts</div><a class="btn sm" style="margin-top:14px" href="#/ewaste">See devices →</a></div></div>
<div class="split" style="margin-top:14px;grid-template-columns:minmax(0,1.5fr) minmax(0,1fr)"><div class="card"><div class="ct">Money saved — ₹ per month, by department (verified) ${MS}</div><div style="height:380px"><canvas id="mm"></canvas></div><div class="legend-note">Verified energy saved (low end) × ₹${h.tariff}/kWh. Total so far: <b>${inr(d.money.total_inr)}</b>. Each action counts from its own start date, so months with more verified actions save more.</div></div>
<div class="card"><div class="ct">Energy saved by department (verified) ${MS}</div><div style="height:380px"><canvas id="dsv"></canvas></div><div class="legend-note">Dark = counted by GreenLedger (safe, low end) · light = the true saving.</div></div></div>
<div style="margin-top:18px">${TWINS}</div>
<div class="split" style="grid-template-columns:minmax(0,1.5fr) minmax(0,1fr)"><div class="card"><div class="ct">Campus energy — live twin vs ghost twin (MWh / month) ${MS}</div><div style="height:380px"><canvas id="m1"></canvas></div><div class="legend-note">Shaded gap = energy saved by verified actions.</div></div>
<div class="card"><div class="ct">Departments — energy per m² (12 months) ${MS}</div><div style="height:380px"><canvas id="m2"></canvas></div></div></div>
<div class="grid g2" style="margin-top:14px"><div class="card"><div class="ct">Do this next ${MS}</div>${d.tips.map(t=>`<div style="border-left:3px solid var(--coral);padding:6px 10px;margin:8px 0"><b>${esc(t.name)}</b><div class="note">${esc(t.tip)}</div></div>`).join("")}<a class="btn sm" href="#/departments">All departments →</a></div>
<div class="card"><div class="ct">Live room right now <span class="mt r">Live camera</span></div><div class="stat">${d.room.people} people</div><div class="note">${d.room.on.length?d.room.on.join(", ")+" ON":"all devices off"} · ${f1(d.room.dev_min)} device-minutes switched off this session</div><a class="btn sm" style="margin-top:10px" href="#/room">Open camera →</a></div></div>`;
spark("#s1",k.energy_spark,false);spark("#s2",k.water_spark,false);spark("#s3",k.waste_spark,false);
chart("#tr",{type:"bar",data:{labels:["Simple dashboard","GreenLedger"],datasets:[{data:[T.naive_saving,T.gl_saving],backgroundColor:[C.amber,C.forest],borderRadius:6}]},options:{indexAxis:"y",plugins:{legend:{display:false}},scales:{x:{max:T.tests,grid:{color:C.grid}},y:{grid:{display:false}}}}});
twinChart("#m1",d.months,d.ghost,d.actual,{fill:true,unit:"MWh"});moneyChart("#mm",d.money);savingsChart("#dsv",d.dept_savings);
chart("#m2",{type:"bar",data:{labels:d.rank.map(r=>r.name),datasets:[{data:d.rank.map(r=>r.kwh_m2),backgroundColor:d.rank.map(r=>({"High use":C.coral,"Average":C.amber,"Efficient":C.forest}[r.eff])),borderRadius:5}]},
options:{indexAxis:"y",plugins:{legend:{display:false},tooltip:{callbacks:{label:c=>f0(c.raw)+" kWh/m² · "+d.rank[c.dataIndex].eff}}},scales:{x:{grid:{color:C.grid},title:{display:true,text:"kWh per m²"}},y:{grid:{display:false},ticks:{autoSkip:false,font:{size:11}}}}}});icons()};
// ================= DEPARTMENTS =================
P.departments=async()=>{const D=await api("api/departments");const T=D.table;let sel=SELDEPT&&T.find(r=>r.Code==SELDEPT)?SELDEPT:T[0].Code;SELDEPT=null;
page().innerHTML=head("building-2","Departments",SIM,"Each department's energy, water and waste — and whether its actions really worked.")+explain("departments")+
`<div class="split" style="grid-template-columns:minmax(0,1.3fr) minmax(0,1fr);margin-bottom:14px"><div class="card"><div class="ct">Pick a department on the campus · colour = energy per m²</div><div id="dmap"></div></div>
<div class="card"><div class="ct">Department-wise energy savings (verified)</div><div style="height:420px"><canvas id="dsv"></canvas></div><div class="legend-note">Dark = counted by GreenLedger (safe, low end) · light = the true saving. Hover for ₹.</div></div></div>
<div class="chips" id="dc">${T.map(r=>`<span class="pill ${r.Code==sel?"on":""}" data-c="${r.Code}">${esc(r.Department)}</span>`).join("")}</div>
<div id="dd"></div>
<div class="card" style="margin-top:14px"><div class="ct">All departments · last 12 months</div><div class="scroll"><table class="t"><tr><th>Department</th><th>Energy MWh</th><th>kWh/m²</th><th>kWh/person</th><th>Water L/person/day</th><th>Diverted %</th><th>CO₂ t</th><th>Bill / month</th><th>Status</th><th>Saved MWh</th></tr>
${T.map(r=>`<tr><td><b>${esc(r.Department)}</b></td><td>${f0(r["Energy MWh/yr"])}</td><td>${f0(r["kWh per m²"])}</td><td>${f0(r["kWh per person"])}</td><td>${f0(r["Water L/person/day"])}</td><td>${f0(r["Diversion %"])}</td><td>${f0(r["CO₂ t/yr"])}</td><td>${inr(r["Cost ₹/month"])}</td><td><span class="status st-${{"Efficient":"Optimal","Average":"Warning","High use":"Critical"}[r.Efficiency]}">${r.Efficiency}</span></td><td>${f1(r["Saved MWh (verified)"])}</td></tr>`).join("")}</table></div></div>
<div class="card" style="margin-top:14px"><div class="ct">Energy by department · MWh per month</div><div style="height:300px"><canvas id="ds"></canvas></div></div>`;
chart("#ds",{type:"bar",data:{labels:D.months,datasets:Object.entries(D.monthly).map(([c,v],i)=>({label:D.names[c],data:v,backgroundColor:DEPC[i%16]}))},options:{plugins:{legend:{position:"bottom",labels:{boxWidth:12}}},scales:{x:{stacked:true,grid:{display:false}},y:{stacked:true,grid:{color:C.grid},title:{display:true,text:"MWh"}}}}});
const drawMap=()=>{$("#dmap").innerHTML=campusMap(D.pins,"energy",sel,true);document.querySelectorAll("#dmap .tpin").forEach(e=>e.onclick=()=>{if(e.dataset.c)pickD(e.dataset.c)});icons()};
const pickD=c=>{sel=c;document.querySelectorAll("#dc .pill").forEach(q=>q.classList.toggle("on",q.dataset.c==c));drawMap();show(c);$("#dd").scrollIntoView({behavior:"smooth",block:"start"})};
savingsChart("#dsv",D.savings);drawMap();
const show=async c=>{const x=await api("api/department/"+c);const r=x.row;drop("dl");drop("dmo");
$("#dd").innerHTML=`<div class="grid g5">${card("Energy · 12 months",f0(r["Energy MWh/yr"])+" MWh",f0(r["kWh per m²"])+" kWh/m² · "+f0(r["kWh per person"])+" kWh/person","dark")}
${card("Electricity bill",inr(r["Cost ₹/month"]),"per month · "+f0(r["CO₂ t/yr"])+" t CO₂ a year")}${card("Water",f0(r["Water L/person/day"])+" L","per person per day")}
${card("Waste diverted",f0(r["Diversion %"])+"%",f1(r["Waste kg/person/day"])+" kg per person per day","sage")}${card("Saved last month",f0(r["Saved kWh last month"])+" kWh",r["Saved ₹ last month"]>0?inr(r["Saved ₹ last month"])+" · verified":(x.actions.length?"not proven yet":"no change logged yet"))}</div>
<div class="split" style="margin-top:14px;grid-template-columns:minmax(0,1.6fr) minmax(0,1fr)"><div class="card"><div class="ct">${esc(x.name)} · energy per week — live vs ghost twin</div><div style="height:280px"><canvas id="dl"></canvas></div></div>
<div class="card"><div class="ct">💡 Suggestion</div><div style="font-weight:600;line-height:1.5">${esc(r["Top suggestion"])}</div>
<div class="ct" style="margin-top:16px">Actions</div>${x.actions.map(a=>`<div style="margin:6px 0"><span class="vbadge ${a.verdict=="VERIFIED SAVING"?"vb-ok":"vb-no"}">${a.verdict}</span> <b>${esc(a.title)}</b><div class="note">${a.date} · estimated ${f1(a.est)}% (true ${f1(a.true)}%) · saved ${f1(a.counted_mwh)} MWh</div></div>`).join("")||'<div class="note">None yet.</div>'}
<div class="ct" style="margin-top:16px">E-waste</div><div class="note">${x.ewaste.devices} devices · ${x.ewaste.sell} to sell now · ${inr(x.ewaste.value)} recoverable</div></div></div>
<div class="card" style="margin-top:14px"><div class="ct">${esc(x.name)} · money saved per month (₹, verified)</div><div style="height:200px"><canvas id="dmo"></canvas></div><div class="legend-note">${x.money.inr.some(v=>v>0)?"Total so far: <b>"+inr(x.money.inr.reduce((a,b)=>a+b,0))+"</b> at ₹"+D.tariff+"/kWh.":"No verified saving yet — log an action and GreenLedger will measure it."}</div></div>`;
twinChart("#dl",x.dates,x.ghost,x.actual,{fill:true,unit:"MWh/week"});
chart("#dmo",{type:"bar",data:{labels:x.money.months,datasets:[{label:"₹ saved",data:x.money.inr,backgroundColor:C.forest,borderRadius:4}]},options:{plugins:{legend:{display:false},tooltip:{callbacks:{label:c=>" "+inr(c.raw)}}},scales:{x:{grid:{display:false}},y:{grid:{color:C.grid},beginAtZero:true,title:{display:true,text:"₹"}}}}})};
document.querySelectorAll("#dc .pill").forEach(p=>p.onclick=()=>{sel=p.dataset.c;document.querySelectorAll("#dc .pill").forEach(q=>q.classList.toggle("on",q==p));drawMap();show(p.dataset.c)});
await show(sel);icons()};
// ================= CAMPUS DIGITAL TWIN =================
P.twin=async()=>{const D=await api("api/departments");const pins=D.pins;let mk="energy",sel=null;
const deps=Object.values(pins).filter(p=>p.m),sum=k=>deps.reduce((a,p)=>a+p.m[k],0);
page().innerHTML=head("map","Campus Digital Twin",SIM,"Interactive campus dashboard — every block, every number, one click away.")+explain("twin")+
`<div class="grid g5" style="margin-bottom:14px">${card("Energy · 12 months",f0(deps.reduce((a,p)=>a+p.mwh,0))+" MWh",deps.length+" departments","dark")}${card("Electricity bill",inr(sum("cost_month")),"per month at ₹"+D.tariff+"/kWh")}
${card("Energy saved",f1(sum("saved_mwh"))+" MWh","verified, low end","sage")}${card("Money saved",inr(sum("saved_inr")),"last month, verified")}${card("CO₂",f0(sum("co2_t"))+" t","per year · India grid")}</div>
<div class="chips" id="met">${Object.entries(MET).map(([k,m])=>`<span class="pill ${k==mk?"on":""}" data-k="${k}">${m.label}</span>`).join("")}</div>
<div class="split" style="grid-template-columns:minmax(0,1.7fr) minmax(0,1fr)"><div class="card"><div id="cmap"></div></div>
<div id="tsel" class="card"><div class="ct">Selection</div><div class="note">Click any block on the campus.</div></div></div>
<div class="card" style="margin-top:14px"><div class="ct" id="rkt"></div><div style="height:300px"><canvas id="rk"></canvas></div><div class="legend-note">Click a bar to select that block.</div></div>
<details style="margin-top:16px"><summary>Reference campus with real data — Concordia University (Montréal)</summary><div class="card"><div class="note">Real meters and waste invoices from Concordia's open data. Aerial imagery © Esri.</div><div id="map" style="height:480px;margin-top:10px;border-radius:8px"></div></div></details>`;
const order=()=>Object.entries(pins).filter(([b,p])=>p.m).sort((a,b)=>MET[mk].good=="low"?b[1].m[MET[mk].key]-a[1].m[MET[mk].key]:a[1].m[MET[mk].key]-b[1].m[MET[mk].key]).reverse();
const draw=()=>{$("#cmap").innerHTML=campusMap(pins,mk,sel,false);document.querySelectorAll("#cmap .tpin").forEach(e=>e.onclick=()=>pick(e.dataset.b));icons();
 drop("rk");const o=order(),col=metColor(pins,mk);$("#rkt").textContent=MET[mk].label+" by block — "+(MET[mk].good=="low"?"lower is better":"higher is better");
 chart("#rk",{type:"bar",data:{labels:o.map(([b])=>b),datasets:[{data:o.map(([b,p])=>p.m[MET[mk].key]),backgroundColor:o.map(([b,p])=>col(p)),borderRadius:5}]},
 options:{indexAxis:"y",onClick:(e,el)=>{if(el.length)pick(o[el[0].index][0])},plugins:{legend:{display:false},tooltip:{callbacks:{label:c=>" "+o[c.dataIndex][1].m.name+": "+MET[mk].fmt(c.raw)}}},scales:{x:{grid:{color:C.grid},beginAtZero:true},y:{grid:{display:false}}}}})};
const pick=async b=>{const p=pins[b];sel=p.depts[0]||null;draw();
 if(!p.m){$("#tsel").innerHTML=`<div class="ct">${esc(b)}</div><div class="note">No meter data for this place yet. Add a sub-meter and it joins the dashboard.</div>`;return}
 const x=await api("api/department/"+p.depts[0]);const r=x.row;drop("tm");
 $("#tsel").innerHTML=`<div class="ct">${esc(b)}</div><div class="row" style="justify-content:space-between;align-items:center"><div style="font:700 17px Manrope,system-ui,sans-serif">${esc(x.name)}</div><span class="status st-${{"Efficient":"Optimal","Average":"Warning","High use":"Critical"}[r.Efficiency]}">${r.Efficiency}</span></div>
<div class="kv"><span>Energy</span><b>${f0(r["Energy MWh/yr"])} MWh/yr</b><span>per m²</span><b>${f0(r["kWh per m²"])} kWh</b><span>Bill</span><b>${inr(r["Cost ₹/month"])}/month</b><span>Saved last month</span><b>${f0(r["Saved kWh last month"])} kWh</b><span>Money saved</span><b>${inr(r["Saved ₹ last month"])}/month</b><span>Water</span><b>${f0(r["Water L/person/day"])} L/person/day</b><span>Waste diverted</span><b>${f0(r["Diversion %"])}%</b><span>E-waste to act on</span><b>${x.ewaste.sell} sell · ${inr(x.ewaste.value)}</b></div>
<div class="ct" style="margin-top:14px">Energy per week · live vs ghost twin</div><div style="height:170px"><canvas id="tm"></canvas></div>
<div class="note" style="margin-top:8px">💡 ${esc(r["Top suggestion"])}</div><a class="btn sm" style="margin-top:10px" href="#/departments" onclick="SELDEPT=this.dataset.c" data-c="${esc(p.depts[0])}">Open department →</a>`;
 twinChart("#tm",x.dates,x.ghost,x.actual,{fill:true,unit:"MWh"});const t=charts.find(c=>c.canvas.id=="tm");if(t){t.options.plugins.legend.display=false;t.update("none")}};
document.querySelectorAll("#met .pill").forEach(e=>e.onclick=()=>{mk=e.dataset.k;document.querySelectorAll("#met .pill").forEach(q=>q.classList.toggle("on",q==e));draw()});
draw();pick(Object.keys(pins).find(b=>pins[b].m&&pins[b].status=="High use")||Object.keys(pins)[0]);
$("details").addEventListener("toggle",async ev=>{if(!ev.target.open||MAP||typeof L=="undefined")return;const B=await api("api/buildings");
MAP=L.map("map");L.tileLayer("tiles/{z}/{y}/{x}",{maxZoom:19,attribution:"Imagery © Esri, Maxar, Earthstar Geographics"}).addTo(MAP);
const pts=[];B.forEach(b=>{const st=b.status.split(" ")[0];L.circleMarker([b.lat,b.lon],{radius:7,color:"#fff",weight:2,fillOpacity:1,fillColor:{Critical:C.coral,Warning:C.amber,Optimal:C.forest}[st]||"#56645c"}).addTo(MAP)
.bindPopup(`<b>${esc(b.code)} · ${esc(b.name)}</b><br>${b.meters.map(m=>esc(m.res+": "+m.name)).join("<br>")||"no meter"}${b.waste_t?`<br>waste ${f0(b.waste_t)} t`:""}`);if(b.campus=="SGW")pts.push([b.lat,b.lon])});MAP.fitBounds(pts,{padding:[30,30]})});icons()};
// ================= LIVE vs GHOST =================
P.prove=async()=>{const D=await api("api/departments");const A=D.actions;let cur=(A.find(a=>a.verdict=="VERIFIED SAVING")||A[0]).id;
page().innerHTML=head("git-compare-arrows","Live vs Ghost Twin",SIM,"Did each action really save energy — and how much money?")+explain("prove")+TWINS+
`<div class="split" style="grid-template-columns:320px minmax(0,1fr)"><div><div class="ct">Actions</div><div class="acts" id="al">${A.map(a=>`<div class="a ${a.id==cur?"on":""}" data-id="${a.id}"><div><b style="font-size:13px">${esc(a.title)}</b><div class="note">${esc(D.names[a.dept])} · ${a.date}</div></div><span class="vbadge ${a.verdict=="VERIFIED SAVING"?"vb-ok":"vb-no"}">${a.verdict=="VERIFIED SAVING"?"✓ "+f1(a.counted_mwh)+" MWh":"not proven"}</span></div>`).join("")}</div></div>
<div><div id="vd"></div><div class="card" style="margin-top:12px"><div style="height:340px"><canvas id="tw"></canvas></div></div>
<div class="card" style="margin-top:12px"><div class="ct">Money saved by this action — ₹, running total</div><div style="height:230px"><canvas id="mo"></canvas></div><div class="legend-note" id="mon"></div></div>
<div class="grid g2" style="margin-top:12px"><div class="card"><div class="ct">Placebo test · fake action dates in the year before</div><div style="height:220px"><canvas id="hist"></canvas></div></div><div id="truth"></div></div></div></div>
<details style="margin-top:18px"><summary>Proof on real data — Concordia University (ground-truth test)</summary><div class="card" id="real"></div></details>`;
const load=async id=>{const r=await api("api/dept_action/"+id);drop("tw");drop("hist");drop("mo");const M=r.money;
chart("#mo",{type:"line",data:{labels:M.dates,datasets:[{label:"Measured (ghost − live)",data:M.measured,borderColor:C.ghost,borderDash:[6,4],borderWidth:2,pointRadius:0},{label:"Counted by GreenLedger (low end)",data:M.counted,borderColor:C.forest,borderWidth:2.6,pointRadius:0,fill:"origin",backgroundColor:"rgba(205,228,106,.35)"}]},
options:{interaction:{mode:"index",intersect:false},plugins:{legend:{labels:{boxWidth:14}},tooltip:{callbacks:{label:c=>" "+c.dataset.label+": "+inr(c.raw)}}},scales:{x:{grid:{display:false},ticks:{maxTicksLimit:10}},y:{grid:{color:C.grid},beginAtZero:true,title:{display:true,text:"₹ saved so far"}}}}});
$("#mon").innerHTML=r.verdict=="VERIFIED SAVING"?`Counted so far: <b>${inr(M.counted[M.counted.length-1])}</b> of ${inr(M.measured[M.measured.length-1])} measured, at ₹${M.tariff}/kWh. The gap is the safety margin — GreenLedger only claims what it can prove.`:"Not proven, so nothing is counted — the dashed line is only what the meter difference would suggest.";const v=r.verdict=="VERIFIED SAVING";
$("#vd").innerHTML=`<div class="verdict ${v?"v-ok":"v-no"}"><div class="big">${v?"✅":"🤔"} ${r.verdict}</div><p><b>${esc(r.dept)} — ${esc(r.title)}</b> (${r.date}). The ghost twin says it used <b>${f1(r["saving_%"])}% less</b> than it would have (95% range ${f1(r["low_%"])} to ${f1(r["high_%"])}%, p = ${(+r.p).toFixed(3)}). ${v?`Saved so far: <b>${f1(r.counted_mwh)} MWh</b> counted (low end).`:"The change is within normal noise, so nothing is counted."}</p></div>`;
twinChart("#tw",[...r.pre_dates,...r.dates],[...r.pre_ghost,...r.ghost],[...r.pre_actual,...r.actual],{hi:[...Array(r.pre_dates.length).fill(null),...r.band_hi],lo:[...Array(r.pre_dates.length).fill(null),...r.band_lo],cum:[...Array(r.pre_dates.length).fill(null),...r.cum],unit:"kWh/day",cumUnit:"MWh saved"});
const fk=r.fakes,lo=Math.min(...fk,r["saving_%"])-2,hi=Math.max(...fk,r["saving_%"])+2,nb=18,w=(hi-lo)/nb,cnt=Array(nb).fill(0);fk.forEach(x=>cnt[Math.min(nb-1,Math.floor((x-lo)/w))]++);const ri=Math.min(nb-1,Math.floor((r["saving_%"]-lo)/w));
chart("#hist",{type:"bar",data:{labels:cnt.map((_,i)=>(lo+w*(i+.5)).toFixed(1)),datasets:[{label:"fake dates",data:cnt,backgroundColor:C.sage,grouped:false,borderRadius:3},{label:"real result",data:cnt.map((_,i)=>i==ri?Math.max(...cnt):null),backgroundColor:C.forest,barPercentage:.35,grouped:false}]},
options:{plugins:{legend:{display:false}},scales:{x:{title:{display:true,text:"apparent saving % (dark bar = the real action)"},grid:{display:false}},y:{grid:{color:C.grid}}}}});
$("#truth").innerHTML=`<div class="card sage" style="height:100%"><div class="ct">Accuracy check — the true effect is known</div><div class="kv"><span>True effect</span><b>${f1(r["true_%"])}%</b><span>GreenLedger estimate</span><b>${f1(r["saving_%"])}%</b><span>95% range</span><b>${f1(r["low_%"])} … ${f1(r["high_%"])}%</b><span>True energy saved</span><b>${f1(r.true_mwh)} MWh</b><span>Counted by GreenLedger</span><b>${f1(r.counted_mwh)} MWh</b></div>
<div class="note" style="margin-top:8px">${r["true_%"]>=r["low_%"]&&r["true_%"]<=r["high_%"]?"✔ The true value lies inside GreenLedger's range.":"✖ The true value is outside the range."}</div></div>`};
document.querySelectorAll("#al .a").forEach(e=>e.onclick=()=>{document.querySelectorAll("#al .a").forEach(x=>x.classList.remove("on"));e.classList.add("on");load(e.dataset.id)});await load(cur);
$("details").addEventListener("toggle",async ev=>{if(!ev.target.open||$("#real").dataset.done)return;$("#real").dataset.done=1;const [acts,tr]=await Promise.all([api("api/actions"),api("api/trust")]);const T=tr.summary;
const ver=acts.filter(a=>a.verdict=="VERIFIED SAVING").sort((a,b)=>(b.saving||0)-(a.saving||0));
$("#real").innerHTML=`<div class="grid g3">${card("Simple dashboard",T.naive_saving+" / "+T.tests,"normal months with a false “saving”")}${card("GreenLedger",T.gl_saving+" / "+T.tests,"false savings certified","dark")}${card("COVID closure","caught","flagged by the detector within days · not a saving","sage")}</div>
<details style="margin-top:10px"><summary>All ${tr.rows.length} tested months (nothing was done in any of them)</summary><div class="scroll" style="max-height:300px"><table class="t"><tr><th>Meter</th><th>Month</th><th>Simple dashboard says</th><th>GreenLedger 95% range</th><th>False saving?</th></tr>${tr.rows.filter(x=>x["naive_%"]!=null&&x["gl_%"]!=null).map(x=>`<tr><td>${esc(x.meter)}</td><td>${x.month}</td><td>${x["naive_%"]>0?f1(x["naive_%"])+"% saving":f1(-x["naive_%"])+"% increase"}</td><td>${f1(x.gl_low)} … ${f1(x.gl_high)}%</td><td>${x.gl_low>0?"<b>yes</b>":"no"}${x["naive_%"]>0?" (simple: yes)":""}</td></tr>`).join("")}</table></div></details>
<div class="field" style="margin-top:12px">Event<select id="ra">${acts.map(a=>`<option value="${a.id}" ${ver[0]&&a.id==ver[0].id?"selected":""}>${a.id} · ${esc(a.meter)} — ${a.verdict}</option>`).join("")}</select></div><div style="height:300px;margin-top:10px"><canvas id="rtw"></canvas></div><div class="note" id="rnote"></div>`;
const rl=async()=>{const r=await api("api/verify/"+$("#ra").value);drop("rtw");$("#rnote").textContent=r.explain;if(r["saving_%"]==null)return;
twinChart("#rtw",[...r.pre_dates,...r.dates],[...r.pre_ghost,...r.ghost],[...r.pre_actual,...r.actual],{hi:[...Array(r.pre_dates.length).fill(null),...r.band_hi],lo:[...Array(r.pre_dates.length).fill(null),...r.band_lo],unit:r.unit+"/day"})};$("#ra").onchange=rl;rl()});icons()};
// ================= LIVE ROOM =================
let STREAM=null,TIMER=null,BUSY=false,PHOTO=null,WTIMER=null;
function stopCam(){if(TIMER){clearInterval(TIMER);TIMER=null}PHOTO=null;if(STREAM){STREAM.getTracks().forEach(t=>t.stop());STREAM=null}}
P.room=async()=>{let st=await api("api/room");const D=st.defaults.devices;
const nm={Fan:"🌀 Fan",Lights:"💡 Lights",AC:"❄️ AC"};
page().innerHTML=head("video","Live Room Control","<span class='tag'>Live camera</span>","Camera counts people → fan, lights and AC switch by themselves, using the live weather. Empty room → everything off.")+explain("room")+
`<div class="split"><div><div class="grid g2"><div class="card"><div class="row" style="justify-content:space-between;flex-wrap:wrap;gap:6px"><div class="ct">Camera</div><div class="row" style="flex-wrap:wrap;gap:6px"><button class="btn sm pri" id="cs">Start camera</button><label class="btn sm">Test photo<input type="file" id="ph" accept="image/*" hidden></label><button class="btn sm" id="em" title="Demo: everyone leaves">Empty room</button><button class="btn sm" id="cx">Stop</button></div></div>
<video id="vid" class="cam" autoplay playsinline muted></video><canvas id="cap" hidden></canvas></div><div class="card"><div class="ct">What the AI sees</div><img id="ann" class="cam" alt=""></div></div>
<div class="grid g4" id="devs" style="margin-top:14px"></div><div id="dec" class="verdict v-wait" style="margin-top:12px"><b>Decision:</b> start the camera or upload a test photo.</div>
<div class="card" style="margin-top:14px"><div class="ct">Live twin vs ghost twin · this session</div><div style="height:240px"><canvas id="rc"></canvas></div></div>
<div class="card" style="margin-top:14px"><div class="ct">Switching log</div><div class="scroll" style="max-height:220px"><table class="t" id="rl"></table></div></div></div>
<div><div class="card dark"><div class="ct">Energy saved this session</div><div class="stat" id="rk" style="color:var(--lime)">0 Wh</div><div style="color:#cfe0c9" id="rk2"></div></div>
<div class="card" style="margin-top:12px"><div class="ct">Live weather</div><div class="row" style="gap:6px"><div class="field" style="flex:1"><input id="city" value="${esc((st.weather.city||"Bengaluru").split(",")[0])}" aria-label="City"></div><button class="btn sm" id="wb" title="Refresh now">↻</button></div><div id="wx" style="margin-top:10px"></div></div>
<div class="card" style="margin-top:12px"><div class="ct">Switching rules</div>
<div class="chips" id="mode" style="margin-bottom:8px"><span class="pill on" data-m="auto">Automatic (live weather)</span><span class="pill" data-m="manual">Manual</span></div>
<div class="note" id="mnote" style="margin-bottom:8px"></div>
<div class="grid g2" style="gap:8px"><div class="field">Fan ON above °C<input id="t_fan"></div><div class="field">AC ON above °C<input id="t_ac"></div><div class="field">AC set to °C<input id="t_set"></div><div class="field">Fan ON with at least … people<input id="t_crowd" value="${st.defaults.crowd}"></div></div>
<div class="field" style="margin-top:8px">Room temperature °C (blank = live outdoor)<input id="rt" placeholder="from a thermometer or the AC display"></div>
<label class="note" style="display:block;margin-top:8px"><input type="checkbox" id="auto" checked> Switch devices automatically</label><label class="note"><input type="checkbox" id="demo" checked> Demo: off 30 s after the room empties</label><br><label class="note"><input type="checkbox" id="priv" checked> Privacy: blur people in the preview</label>
<div class="note" style="margin-top:6px">🔒 Frames are analysed in memory and never saved — only the number of people is kept. No face recognition, no identities.</div></div>
<div class="card" style="margin-top:12px"><div class="ct">Devices in this room</div><div class="field" style="display:grid;grid-template-columns:78px minmax(0,1fr) minmax(0,1fr);gap:6px;align-items:center"><span></span><span>How many</span><span>Watts each (label)</span>${["Fan","Lights","AC"].map(d=>`<span style="font-size:13px;color:var(--fg)">${nm[d]}</span><input id="n_${d.toLowerCase()}" value="${D[d][0]}" aria-label="${d} count"><input id="w_${d.toLowerCase()}" value="${D[d][1]}" aria-label="${d} watts">`).join("")}</div>
<details style="margin-top:10px"><summary>More</summary><div class="field">Off after empty (min) — ASHRAE 90.1<input id="vac" value="20"></div><div class="field" style="margin-top:6px">Detection confidence <span id="cv">0.50</span><input type="range" id="conf" min="0.25" max="0.9" step="0.05" value="0.5"></div>
<div class="field" style="margin-top:6px">Outdoor 7-day mean °C (only if the weather can't load)<input id="od"></div></details>
<div class="row" style="margin-top:10px"><button class="btn sm" id="rs">↺ New session</button></div><div class="note" id="sw" style="margin-top:8px"></div></div></div></div>`;
let MODE="auto";
const vals=()=>({privacy:$("#priv").checked,auto:$("#auto").checked,demo:$("#demo").checked,vacancy:$("#vac").value,conf:$("#conf").value,room_temp:$("#rt").value,outdoor:$("#od").value,city:$("#city").value,
 mode:MODE,fan_on:$("#t_fan").value,ac_on:$("#t_ac").value,ac_set:$("#t_set").value,crowd:$("#t_crowd").value,
 n_fan:$("#n_fan").value,w_fan:$("#w_fan").value,n_lights:$("#n_lights").value,w_lights:$("#w_lights").value,n_ac:$("#n_ac").value,w_ac:$("#w_ac").value});
const fillAuto=r=>{const a=r.auto_thr;if(MODE!="auto")return;$("#t_fan").value=a.fan_on!=null?f1(a.fan_on):"";$("#t_ac").value=a.ac_on!=null?f1(a.ac_on):"";$("#t_set").value=a.ac_set!=null?f0(a.ac_set):"";$("#t_crowd").value=st.defaults.crowd};
const setMode=m=>{MODE=m;document.querySelectorAll("#mode .pill").forEach(p=>p.classList.toggle("on",p.dataset.m==m));["t_fan","t_ac","t_set","t_crowd"].forEach(i=>$("#"+i).disabled=m=="auto");
 $("#mnote").textContent=m=="auto"?`Set by itself from the live weather (ASHRAE 55 comfort model) and updated every ${st.weather.every_min} min. Crowded rooms get the fan because each person gives off about ${st.defaults.person_w} W of heat.`:"Your numbers are used. Blank = the automatic value.";fillAuto(st)};
document.querySelectorAll("#mode .pill").forEach(p=>p.onclick=()=>setMode(p.dataset.m));
$("#conf").oninput=()=>$("#cv").textContent=(+$("#conf").value).toFixed(2);
const upd=r=>{st=r;if(r.annotated){$("#ann").src=r.annotated;$("#ann").style.visibility="visible"}const w=r.weather,t=r.thr;
$("#wx").innerHTML=w.now!=null?`<div class="row" style="align-items:baseline;gap:8px"><div class="stat">${f1(w.now)} °C</div><div class="note">${esc(w.city)}</div></div><div class="note">feels like ${f1(w.feels)} °C · humidity ${f0(w.humidity)}% · 7-day mean ${f1(w.trm)} °C<br>updated ${w.fetched||"—"} · refreshes every ${w.every_min} min (Open-Meteo)</div>`:`<div class="note">Weather not loaded${w.error?": "+esc(w.error):""}. Check the city, or switch to Manual and type the temperatures.</div>`;
fillAuto(r);
$("#devs").innerHTML=card("People",r.people,r.people>=t.crowd?`crowded (≥ ${t.crowd}) → fan`:"in view now")+["Fan","Lights","AC"].map(d=>`<div class="card"><div class="ct">${nm[d]}</div><div class="dev"><div class="sw ${r.live[d]?"on":""}" onclick="sw('${d}',${!r.live[d]})" title="Switch by hand"></div><b>${r.live[d]?"ON":"OFF"}${d=="AC"&&r.live.AC&&r.setpoint?" · "+f0(r.setpoint)+" °C":""}</b></div><div class="note" style="margin-top:6px">${r.manual[d]?`<b>manual</b> · <a href="#" onclick="event.preventDefault();toAuto('${d}')">back to auto</a>`:"auto"} · ghost twin: ${r.ghost[d]?"ON":"OFF"}</div></div>`).join("");
const rules=t.fan_on!=null?`Rules (${esc(t.source)}): fan above ${f1(t.fan_on)} °C or with ${t.crowd}+ people · AC above ${f1(t.ac_on)} °C, set to ${f0(t.ac_set)} °C`:esc(t.source);
if(r.reason)$("#dec").innerHTML=`<b>Decision:</b> ${esc(r.reason)}<div class="note">${rules}<br>Temperature used: ${esc(r.temp_source||"—")} · ${esc(r.switch)}</div>`;
$("#rk").textContent=f1(r.kwh*1000)+" Wh";$("#rk2").textContent=`${f1(r.dev_min)} device-minutes off · ${f1(r.india_g)} g CO₂ avoided`;$("#sw").textContent=r.switch;
$("#rl").innerHTML=`<tr><th>Time</th><th>Device</th><th>Action</th><th>People</th><th>Why</th></tr>`+r.log.map(l=>`<tr><td>${l.time}</td><td>${l.device}</td><td><b>${l.action}</b></td><td>${l.people}</td><td class="note">${esc(l.reason)}</td></tr>`).join("");
const H=r.history,o=charts.find(c=>c.canvas.id=="rc");const ds=[{label:"Ghost twin (nobody switches off)",data:H.ghost_kw,borderColor:C.ghost,borderDash:[6,4],stepped:true,pointRadius:0},{label:"Live twin (GreenLedger)",data:H.live_kw,borderColor:C.forest,borderWidth:2.5,stepped:true,pointRadius:0,fill:"-1",backgroundColor:"rgba(205,228,106,.35)"},{label:"People",data:H.people,borderColor:C.water,stepped:true,pointRadius:0,yAxisID:"y2"}];
if(o){o.data.labels=H.t;o.data.datasets=ds;o.update("none")}else chart("#rc",{type:"line",data:{labels:H.t,datasets:ds},options:{animation:false,scales:{x:{grid:{display:false},ticks:{maxTicksLimit:8}},y:{beginAtZero:true,grid:{color:C.grid},title:{display:true,text:"kW"}},y2:{position:"right",beginAtZero:true,grid:{display:false},title:{display:true,text:"people"}}}}})};
window.sw=async(d,on)=>upd(await post("api/room/switch",{device:d,on}));
window.toAuto=async d=>upd(await post("api/room/switch",{device:d,auto:true}));
const send=async(u,still)=>{if(BUSY)return;BUSY=true;try{upd(await post("api/room/frame",{...vals(),image:u||null,still:!!still}))}catch(e){$("#dec").innerHTML="<b>Error:</b> "+esc(e.message)}finally{BUSY=false}};
const grab=()=>{const v=$("#vid");if(!v||!v.videoWidth)return;const c=$("#cap"),w=640,h=Math.round(640*v.videoHeight/v.videoWidth);c.width=w;c.height=h;c.getContext("2d").drawImage(v,0,0,w,h);send(c.toDataURL("image/jpeg",.7))};
$("#cs").onclick=async()=>{stopCam();$("#vid").removeAttribute("poster");try{STREAM=await navigator.mediaDevices.getUserMedia({video:{width:640},audio:false});$("#vid").srcObject=STREAM;TIMER=setInterval(grab,2000)}catch(e){$("#dec").innerHTML="<b>Camera not available:</b> "+esc(e.message)+" — use “Test photo”."}};
$("#ph").onchange=ev=>{const f=ev.target.files[0];if(!f)return;const fr=new FileReader();fr.onload=()=>{stopCam();PHOTO=fr.result;$("#vid").srcObject=null;$("#vid").poster=PHOTO;send(PHOTO,true);TIMER=setInterval(()=>send(PHOTO,true),2000)};fr.readAsDataURL(f);ev.target.value=""};
$("#em").onclick=()=>{stopCam();$("#ann").style.visibility="hidden";$("#vid").removeAttribute("poster");send(null,true);TIMER=setInterval(()=>send(null,true),2000)};
$("#cx").onclick=stopCam;$("#rs").onclick=async()=>{drop("rc");upd(await post("api/room/reset"))};
const loadW=async()=>upd(await api("api/weather?city="+encodeURIComponent($("#city").value)));
$("#wb").onclick=loadW;$("#city").onchange=loadW;
setMode("auto");upd(st);icons();
if(st.weather.now==null||st.weather.city==null)loadW();
WTIMER=setInterval(()=>{if(!TIMER)loadW()},st.weather.every_min*60000)};
// ================= E-WASTE =================
P.ewaste=async()=>{const draw=async dept=>{kill();const E=await api("api/ewaste_sim?dept="+(dept||""));const S=E.summary;
page().innerHTML=head("cpu","E-waste",SIM,"Every device: keep it, sell it second-hand (and when), repair it, or take out reusable parts before a registered recycler takes the rest.")+explain("ewaste")+
`<div class="chips"><span class="pill ${!dept?"on":""}" data-d="">All departments</span>${Object.entries(E.depts).map(([c,n])=>`<span class="pill ${dept==c?"on":""}" data-d="${c}">${esc(n)}</span>`).join("")}</div>
<div class="grid g4">${card("Value recoverable now",inr(S.recoverable_value),"resale + repairs + reusable parts","dark")}${card("Sell second-hand now",S.sell_now+" devices",inr(S.sell_value)+" · waiting a year loses "+inr(S.lost_per_year))}
${card("Reusable parts",S.harvest+" devices",inr(S.parts_value)+" in parts","sage")}${card("To registered recycler",f0(S.recycler_kg)+" kg",S.repair+" can be repaired · "+S.keep+" keep in use")}</div>
<div class="split" style="margin-top:14px;grid-template-columns:minmax(0,1.4fr) minmax(0,1fr)"><div class="card"><div class="ct">When to sell — devices reaching their best resale moment (next 12 months)</div><div style="height:260px"><canvas id="up"></canvas></div><div class="legend-note">Best moment = one year before end of life, while still working and worth the most.</div></div>
<div class="card"><div class="ct">What to do with each device</div><div style="height:260px"><canvas id="ac"></canvas></div></div></div>
<div class="split" style="margin-top:14px;grid-template-columns:minmax(0,1.6fr) minmax(0,1fr)"><div class="card"><div class="ct">Act now — sell first, then reuse parts, then repair</div><div class="scroll"><table class="t"><tr><th>Device</th><th>Dept</th><th>Age</th><th>Condition</th><th>Do this</th><th>Value</th><th>Reusable parts</th></tr>
${E.top.map(t=>`<tr><td><b>${esc(t.type)}</b><div class="note">${t.device_id}</div></td><td>${t.dept}</td><td>${f1(t.age_years)} y</td><td>${t.condition}</td><td><b>${t.action}</b></td><td>${inr(t.value_now)}</td><td class="note">${esc(t.reusable_parts)||"—"}</td></tr>`).join("")}</table></div></div>
<div class="card"><div class="ct">Most common reusable parts</div><div style="height:300px"><canvas id="pt"></canvas></div></div></div>
<details style="margin-top:16px"><summary>Use your own asset register instead</summary><div class="card note">Upload your real device list on the <a href="#/reports">Reports &amp; Data</a> page — routes follow India's E-Waste (Management) Rules, 2022.</div></details>`;
document.querySelectorAll(".chips .pill").forEach(p=>p.onclick=()=>draw(p.dataset.d));
chart("#up",{type:"bar",data:{labels:E.upcoming.months,datasets:[{label:"devices",data:E.upcoming.devices,backgroundColor:C.sage,borderRadius:4,yAxisID:"y"},{type:"line",label:"value ₹",data:E.upcoming.value,borderColor:C.forest,pointRadius:3,yAxisID:"y2"}]},
options:{scales:{x:{grid:{display:false}},y:{grid:{color:C.grid},title:{display:true,text:"devices"}},y2:{position:"right",grid:{display:false},title:{display:true,text:"resale value ₹"}}}}});
chart("#ac",{type:"doughnut",data:{labels:Object.keys(E.actions),datasets:[{data:Object.values(E.actions),backgroundColor:Object.keys(E.actions).map(k=>({"Keep in use":C.forest,"Sell second-hand now":C.lime,"Repair & keep using":C.sage,"Repair & resell":C.amber,"Harvest parts":C.water,"Harvest parts + recycle":C.coral})[k]||C.ghost)}]},options:{cutout:"62%",plugins:{legend:{position:"bottom",labels:{boxWidth:12}}}}});
chart("#pt",{type:"bar",data:{labels:Object.keys(E.parts),datasets:[{data:Object.values(E.parts),backgroundColor:C.forest,borderRadius:4}]},options:{indexAxis:"y",plugins:{legend:{display:false}},scales:{x:{grid:{color:C.grid},title:{display:true,text:"devices with this part reusable"}},y:{grid:{display:false}}}}});icons()};await draw("")};
// ================= CARBON =================
P.carbon=async()=>{const [D,c]=await Promise.all([api("api/departments"),api("api/carbon?gas=0")]);const T=D.table;const tot=T.reduce((a,r)=>a+r["CO₂ t/yr"],0),sav=D.actions.reduce((a,x)=>a+x.counted_mwh,0)*0.71;
page().innerHTML=head("cloud","Carbon Footprint",SIM,"Electricity on India's grid (CEA 0.71 kg CO₂/kWh). Avoided emissions count only verified savings.")+explain("carbon")+
`<div class="grid g3">${card("Campus emissions · 12 months",f0(tot)+" t CO₂","electricity, India grid","dark")}${card("Avoided by verified actions",f1(sav)+" t CO₂","counted at the low end","sage")}${card("Per person",f0(tot*1000/T.reduce((a,r)=>a+r.People,0))+" kg CO₂","per year")}</div>
<div class="card" style="margin-top:14px"><div class="ct">Emissions by department · t CO₂ per year</div><div style="height:300px"><canvas id="cd"></canvas></div></div>
<details style="margin-top:16px"><summary>Reference campus with real data — Concordia University (GHG Protocol scopes)</summary><div class="card"><label class="note"><input type="checkbox" id="gasu"> Count the gas meter as m³ — the unit Concordia's API documents, not yet confirmed by Facilities (off = gas shown only as a range)</label><div id="cref" style="margin-top:10px"></div></div></details>`;
chart("#cd",{type:"bar",data:{labels:T.map(r=>r.Department),datasets:[{label:"t CO₂/yr",data:T.map(r=>r["CO₂ t/yr"]),backgroundColor:T.map(r=>({"High use":C.coral,"Average":C.amber,"Efficient":C.forest}[r.Efficiency])),borderRadius:5}]},options:{plugins:{legend:{display:false}},scales:{x:{grid:{display:false}},y:{grid:{color:C.grid},title:{display:true,text:"t CO₂"}}}}});const rc=c=>{$("#cref").innerHTML=`<div class="grid g3">${card("Measured "+c.base_period,f0(Object.values(c.scopes).reduce((a,v)=>a+(v[c.periods.indexOf(c.base_period)]||0),0))+" t CO₂e",c.gas_on?"electricity + landfill + gas":"electricity + landfill")}${c.gas_on?card("Gas (counted as m³)",f0(c.gas_hi)+" t","included in the total on the left"):card("Gas (unit unconfirmed)",f0(c.gas_lo)+"–"+f0(c.gas_hi)+" t","the biggest source either way · not in the total")}${card("Same kWh on India's grid",f0(c.lines.filter(l=>l.period==c.base_period&&/electricity/.test(l.scope)).reduce((a,l)=>a+l.amount,0)*c.india_factor/1000)+" t","Québec is ~hydro")}</div>
<table class="t" style="margin-top:10px"><tr><th>Factor</th><th>Value</th><th>Source</th></tr>${c.factors.map(f=>`<tr><td>${esc(f.What)}</td><td>${esc(f.Factor)}</td><td class="note">${esc(f.Source)}</td></tr>`).join("")}</table>`};rc(c);$("#gasu").onchange=async e=>{const g=e.target.checked;const c2=await api("api/carbon?gas="+(g?1:0));c2.gas_on=g;rc(c2)};icons()};
// ================= RECOMMENDATIONS =================
P.recs=async()=>{const [D,R]=await Promise.all([api("api/departments"),api("api/recs")]);const today=new Date().toISOString().slice(0,10);
page().innerHTML=head("lightbulb","Recommendations","",`One clear suggestion per department, and data-sized actions for the reference campus that a person approves or rejects.`)+explain("recs")+
`<div class="ct">Departments ${SIM}</div><div class="grid g3">${D.table.map(r=>`<div class="card"><div class="row" style="justify-content:space-between"><b>${esc(r.Department)}</b><span class="status st-${{"Efficient":"Optimal","Average":"Warning","High use":"Critical"}[r.Efficiency]}">${r.Efficiency}</span></div><div class="note" style="margin-top:8px">💡 ${esc(r["Top suggestion"])}</div></div>`).join("")}</div>
<div class="ct" style="margin-top:22px">Reference campus · approve or reject ${REAL}</div><div id="rl"></div>`;
const ic={Electricity:"zap","Natural gas":"flame",Waste:"recycle",Water:"droplet"};
$("#rl").innerHTML=R.slice(0,12).map(r=>`<div class="rec"><div class="h" onclick="this.parentNode.classList.toggle('open')"><div class="ic"><i data-lucide="${ic[r.area]||"lightbulb"}"></i></div><div style="flex:1"><div class="t1">${r.priority} · ${esc(r.where)}</div><div class="t2">${esc(r.next)}</div><div class="t3">${esc(r.potential)}</div></div><span class="badge b-${r.status}">${r.status=="open"?"waiting":r.status}</span></div>
<div class="body"><div>${esc(r.finding)}</div>${r.status=="open"?`<div class="row" style="margin-top:10px">${r.meter?`<div class="field">Date done<input id="dt_${r.id}" value="${today}"></div>`:""}<div class="field" style="flex:1">Reason (to reject)<input id="rs_${r.id}"></div><button class="btn pri" onclick="dec('${r.id}','approved')">Approve</button><button class="btn" onclick="dec('${r.id}','rejected')">Reject</button></div><div class="msg" id="m_${r.id}"></div>`:""}</div></div>`).join("");
window.dec=async(id,d)=>{const r=await post("api/recs/"+id,{decision:d,reason:($("#rs_"+id)||{}).value,date:($("#dt_"+id)||{}).value});$("#m_"+id).className="msg"+(r.error?" err":"");$("#m_"+id).textContent=r.error||(r.action_id?`✔ Approved → ${r.action_id}: ${r.verdict}`:"✔ Saved");if(!r.error)setTimeout(P.recs,900)};icons()};
// ================= REPORTS & DATA =================
P.reports=async()=>{const R=[["Campus resource report","HTML","greenledger_report"],["Labelled dataset","ZIP · every reading labelled + data dictionary","greenledger_dataset"],["Carbon statement","CSV","carbon_report.csv"],["Verification ledger","CSV","verification.csv"],["Recommendations","CSV","recommendations.csv"]];
const s=await api("api/settings");
page().innerHTML=head("file-text","Reports & Data","",`Exports, data sources and every setting.`)+explain("reports")+
`<div class="grid g3">${R.map(r=>`<div class="card row" style="justify-content:space-between;align-items:center"><div><b>${r[0]}</b><div class="note">${r[1]}</div></div><button class="btn pri sm" onclick="exp('${r[2]}')">⬇ Export</button></div>`).join("")}</div><div class="msg" id="rm"></div>
<div class="card" style="margin-top:14px"><div class="ct">Data in this app</div><div class="kv" style="grid-template-columns:auto 1fr"><span>${SIM}</span><b style="text-align:left;font-weight:500">Departments, department actions and the e-waste register — generated to behave like a real Indian campus (Bengaluru climate, academic calendar). Used to test the method, because the true effect is known.</b>
<span>${REAL}</span><b style="text-align:left;font-weight:500">Concordia University open data (CC BY 4.0): 6 meters, 666 waste invoices — the trust test and ground-truth proof.</b>
<span><span class="tag">Live camera</span></span><b style="text-align:left;font-weight:500">Your webcam, counted by YOLOv8n; weather from Open-Meteo.</b></div></div>
<details><summary>Your own e-waste register (real devices)</summary><div class="card"><div class="note">Columns: ${["device_id","device_type","department","purchase_date","expected_life_years","condition","price_inr","weight_kg","embodied_kgco2e"].join(", ")}</div>
<div class="row" style="margin-top:8px"><a class="btn sm" href="files/ewaste_register_TEMPLATE.csv">Template CSV</a><label class="btn sm pri">Upload CSV<input type="file" id="ewf" accept=".csv" hidden></label></div><div id="ewr" class="note" style="margin-top:8px"></div></div></details>
<details><summary>Every setting and its source</summary><div class="card scroll"><table class="t">${s.parameters.map(p=>`<tr><td><b>${esc(p.What)}</b></td><td>${esc(p.Value)}</td><td class="note">${esc(p["Why / source"])}</td></tr>`).join("")}</table></div></details>
<details><summary>Model accuracy and bad data found (reference campus)</summary><div class="card scroll"><table class="t"><tr><th>Meter</th><th>Accuracy</th><th>Ghost twin model</th></tr>${s.accuracy.map(a=>`<tr><td>${esc(a.Meter)}</td><td>${esc(a.Accuracy)} (${f1(a["Typical daily error %"])}%)</td><td>${esc(a["Predictions by"])}</td></tr>`).join("")}</table>
<table class="t" style="margin-top:10px"><tr><th>Meter</th><th>From</th><th>To</th><th>Rule</th></tr>${s.faults.map(f=>`<tr><td>${esc(f.meter)}</td><td>${f.from}</td><td>${f.to}</td><td>${esc(f.rule)}</td></tr>`).join("")}</table></div></details>`;
window.exp=async key=>{$("#rm").textContent="Building…";try{const r=await post("api/report");const f=(r.files||[]).find(x=>x.startsWith(key));$("#rm").textContent=f?"✔ "+f:"Not available";if(f)location.href="files/"+f}catch(e){$("#rm").textContent="Could not build the export: "+e.message+" — re-run cell 12 and try again."}};
$("#ewf").onchange=async ev=>{const t=await ev.target.files[0].text();const L=t.trim().split(/\r?\n/),h=L[0].split(",");const rows=L.slice(1).map(l=>{const v=l.split(",");return Object.fromEntries(h.map((c,i)=>[c,v[i]??""]))});
const r=await post("api/ewaste",{rows});$("#ewr").textContent=r.plan?`✔ Plan made for ${r.plan.length} devices (also added to the carbon statement).`:"No valid rows."};icons()};
// ================= ROUTER =================
$("#nav").innerHTML=NAV.map(([k,l,i])=>`<a href="#/${k}" data-k="${k}"><i data-lucide="${i}"></i>${l}</a>`).join("");
// ================= JUDGE'S GUIDE =================
P.guide=async()=>{const G=await api("api/guide");const V=G.validation;
page().innerHTML=head("badge-check","Judge's Guide","",`${esc(G.team.problem)} · Team ${esc(G.team.name)}`)+explain("guide")+
`<div class="card dark" style="margin-bottom:14px"><div class="ct">GreenLedger in one sentence</div><div style="font:600 22px Manrope,system-ui,sans-serif;line-height:1.45;color:#fff">It shows every resource the campus uses, tells you what to change, and then <span style="color:var(--lime)">proves</span> whether the change really saved energy and money — counting only what it can prove.</div></div>
<div class="grid g4">${card("Found the true saving",V.sim_inside+" of "+V.sim_n,"changes where the real answer was known","sage")}${card("Never claimed too much",V.sim_no_overcount+" of "+V.sim_n,"counted saving ≤ real saving")}
${card("False alarms on real meters",V.trust_gl+" of "+V.trust_tests,"a normal dashboard: "+V.trust_naive+" of "+V.trust_tests)}${card("Real event detected",V.covid_verified+" of "+V.covid_n,"COVID-19 campus closure")}</div>
<div class="card" style="margin-top:14px"><div class="ct">What the brief asked → what we built</div><table class="t">${G.brief.map((b,i)=>`<tr><td style="width:36px"><span class="vbadge vb-ok">✓</span></td><td style="width:30%"><b>${esc(b.asks)}</b></td><td>${esc(b.how)}</td><td style="width:90px"><a class="btn sm" href="#/${b.where}">Show me →</a></td></tr>`).join("")}</table></div>
<div class="grid g2" style="margin-top:14px"><div class="card"><div class="ct">How we know it works</div><ol style="margin:6px 0;padding-left:20px;line-height:1.6;font-size:14px">${G.validation_done.map(x=>`<li style="margin:6px 0">${esc(x)}</li>`).join("")}</ol></div>
<div class="card"><div class="ct">Next step — a pilot on our campus</div><ol style="margin:6px 0;padding-left:20px;line-height:1.6;font-size:14px">${G.validation_plan.map(x=>`<li style="margin:6px 0">${esc(x)}</li>`).join("")}</ol>
<div class="ct" style="margin-top:14px">Who uses it</div><table class="t">${G.stakeholders.map(x=>`<tr><td><b>${esc(x.who)}</b></td><td class="note">${esc(x.role)}</td></tr>`).join("")}</table></div></div>
<div class="card" style="margin-top:14px"><div class="ct">Targets</div><table class="t"><tr><th>What we measure</th><th>Target</th><th>Today</th></tr>${G.success_criteria.map(x=>`<tr><td>${esc(x.metric)}</td><td><b>${esc(x.target)}</b></td><td>${esc(x.today)}</td></tr>`).join("")}</table></div>`;icons()};
// ================= SUPPORT & INFO =================
P.help=async()=>{const G=await api("api/guide");const T=G.team;
const GLOSS=[["Live twin","What really happened — the meter reading."],["Ghost twin","What the same building would have used without the change, on the same weather and calendar."],["Saving","Ghost twin minus live twin."],["Placebo test","We pretend the change happened on dates when nothing was done. If those ‘fake savings’ are as big as the real one, the saving is not proven."],["Verified saving","A saving bigger than normal ups and downs (95% sure)."],["Low end","The smallest saving that is still 95% sure — the number GreenLedger counts."],["kWh / MWh","Units of electricity. 1 MWh = 1,000 kWh."],["₹K","Thousands of rupees. ₹226K = ₹2,26,000."],["Always-on %","Share of energy used even when nobody is there (servers, freezers, corridors)."],["CO₂","Carbon dioxide from the power plants: 0.71 kg for every kWh on India’s grid."]];
const FAQ=[["Why does a saving say “not proven”?","The change was too small compared with normal day-to-day ups and downs. Nothing is counted until it is clear."],["Why is the counted saving smaller than the gap on the chart?","GreenLedger counts only the low, safe end of the range, so it never over-claims."],["How do I add a change we made?","Recommendations → Approve, with the date. It joins the action register and is checked automatically."],["Can I use our own bills or devices?","Yes — Reports & Data → upload your e-waste register, or load your meter/bill CSVs in the notebook."]];
const FIX=[["Charts are blank","Re-run the last two notebook cells, then refresh the page."],["Camera does not start","Allow camera access in the browser, or use “Test photo”."],["Weather does not load","Check the city name and press ↻, or switch the room rules to Manual."],["Public link does not open","Use the Colab link from the last cell, or mobile data instead of venue Wi-Fi."]];
page().innerHTML=head("life-buoy","Support & Info","",`Help, the meaning of every word, and the team behind GreenLedger.`)+explain("help")+
`<div class="split" style="grid-template-columns:minmax(0,1fr) minmax(0,1fr)"><div class="card dark"><div class="ct">Team</div><div style="font:800 34px Manrope,system-ui,sans-serif;color:var(--lime)">${esc(T.name)}</div>
<div style="margin-top:10px;display:flex;flex-direction:column;gap:6px">${T.members.map(m=>`<div style="font:600 17px 'DM Sans',system-ui,sans-serif;color:#fff">● ${esc(m)}</div>`).join("")}</div>
<div class="note" style="color:#cfe0c9;margin-top:12px">${esc(T.college)}<br>${esc(T.problem)} · Innovators Conclave 2026</div></div>
<div class="card"><div class="ct">How to use GreenLedger — 5 steps</div><ol style="margin:6px 0;padding-left:20px;line-height:1.6;font-size:14px">
<li><b>Dashboard</b> — how much was saved last month, and where to act next.</li><li><b>Departments</b> or <b>Campus Digital Twin</b> — click a block to see its energy, water, waste and one suggestion.</li>
<li><b>Recommendations</b> — approve a change; it is logged with its date.</li><li><b>Live vs Ghost Twin</b> — see whether each change really worked, in kWh and ₹.</li><li><b>Reports & Data</b> — download the report and the labelled data.</li></ol></div></div>
<div class="grid g2" style="margin-top:14px"><div class="card"><div class="ct">Words used in this app</div><table class="t">${GLOSS.map(g=>`<tr><td style="width:30%"><b>${g[0]}</b></td><td>${g[1]}</td></tr>`).join("")}</table></div>
<div class="card"><div class="ct">Frequently asked</div>${FAQ.map(q=>`<div style="margin:8px 0 12px"><b>${q[0]}</b><div class="note" style="font-size:13px;margin-top:3px">${q[1]}</div></div>`).join("")}
<div class="ct" style="margin-top:14px">If something does not work</div><table class="t">${FIX.map(f=>`<tr><td style="width:38%"><b>${f[0]}</b></td><td class="note">${f[1]}</td></tr>`).join("")}</table></div></div>
<div class="grid g2" style="margin-top:14px"><div class="card"><div class="ct">Settings and their sources</div><table class="t">${G.assumptions.map(a=>`<tr><td><b>${esc(a.what)}</b><div class="note">${esc(a.source)}</div></td><td>${esc(a.value)}</td></tr>`).join("")}</table></div>
<div class="card"><div class="ct">Safety and privacy</div><ul style="margin:6px 0;padding-left:18px;line-height:1.55;font-size:13px">${G.safety.map(x=>`<li style="margin:5px 0">${esc(x)}</li>`).join("")}</ul>
<div class="ct" style="margin-top:14px">About the data</div><div class="note" style="font-size:13px;line-height:1.55">${esc(G.about_data)}</div></div></div>`;icons()};
async function route(){const k=(location.hash.replace("#/","")||"dashboard");kill();document.querySelectorAll("#nav a").forEach(a=>a.classList.toggle("on",a.dataset.k==k));
const n=NAV.find(x=>x[0]==k);$("#ttl").textContent=n?n[1]:"Dashboard";page().innerHTML='<div class="note">Loading…</div>';
try{await (P[k]||P.dashboard)()}catch(e){page().innerHTML=`<div class="card"><b>Something went wrong:</b> ${esc(e.message)}</div>`;console.error(e)}window.scrollTo(0,0);icons()}
window.addEventListener("hashchange",route);icons();route();

</script>
</body>
</html>'''
print(f'Web page ready ({len(WEB_HTML):,} characters)')


## Cell 15 · The data API
FastAPI routes that serve the page and every number on it from the cells above (meters, ghost twins, verification, recommendations, waste, e-waste, carbon, live room).

In [ ]:
# ---- The web app's data API (FastAPI). Every number the pages show comes from the cells above. ----
import base64, io, math
from fastapi import FastAPI, Body
from fastapi.responses import HTMLResponse, JSONResponse, FileResponse, Response
from PIL import Image as _PILImage


def J(x):
    """Make anything JSON-safe (numpy, pandas, NaN → null)."""
    if isinstance(x, dict):
        return {str(k): J(v) for k, v in x.items()}
    if isinstance(x, (list, tuple, set)):
        return [J(v) for v in x]
    if isinstance(x, pd.DataFrame):
        return J(x.to_dict(orient="records"))
    if isinstance(x, pd.Series):
        return J(x.tolist())
    if isinstance(x, (pd.Timestamp, datetime.datetime, datetime.date)):
        return x.isoformat()[:19]
    if isinstance(x, pd.Period):
        return str(x)
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating, float)):
        return None if (x != x or x in (math.inf, -math.inf)) else float(x)
    if isinstance(x, np.bool_):
        return bool(x)
    if isinstance(x, np.ndarray):
        return J(x.tolist())
    return x


def ok(x):
    return JSONResponse(J(x))


def _mname(p):
    return METER_INFO[p][2]


ELEC_CAMPUS = CAMPUS_TOTAL["electricity"]
REAL_POINTS = [p for p in DAILY if not str(p).startswith("D_")]
_m_ok = [m for m in pd.period_range(DAYS[0], DAYS[-1], freq="M")
         if DAILY.loc[m.start_time:m.end_time.normalize(), ELEC_CAMPUS].notna().sum().min() >= 20]
LATEST = _m_ok[-1]
L0, L1 = LATEST.start_time, LATEST.end_time.normalize()
TYPICAL_MISS = TRUST.groupby("point")["gl_%"].apply(lambda s: float(np.sqrt((s ** 2).mean()))).to_dict()


def _monthly(p, kind="actual"):
    y = DAILY[p] if kind == "actual" else EXPECTED[p].reindex(DAYS).where(DAILY[p].notna())
    return y.resample("MS").sum(min_count=1)


def _campus_monthly(res, kind="actual"):
    return sum(_monthly(p, kind).fillna(0) for p in CAMPUS_TOTAL[res])


def meter_status(p, a=L0, b=L1):
    """vs the ghost twin in the period: Optimal (normal or below) · Warning (above normal) · Critical (alert in period)."""
    y = DAILY[p].loc[a:b].dropna()
    al = ALERT_TABLE[(ALERT_TABLE.point == p) & (pd.to_datetime(ALERT_TABLE["to"]) >= a) & (pd.to_datetime(ALERT_TABLE["from"]) <= b)]
    if p not in RELIABLE or not len(y):
        return "Unrated", None
    e = EXPECTED[p].reindex(y.index).sum()
    d = float((y.sum() - e) / e * 100)
    if len(al):
        return "Critical", d
    if d > TYPICAL_MISS.get(p, 0):
        return "Warning", d
    return "Optimal", d


def summary():
    em_a, em_g = _campus_monthly("electricity"), _campus_monthly("electricity", "ghost")
    wa = _monthly("Point_10")
    last12 = pd.date_range(L0 - pd.DateOffset(months=11), L0, freq="MS")
    e_now, g_now = float(em_a.get(L0, np.nan)), float(em_g.get(L0, np.nan))
    w_now = float(wa.get(L0, np.nan)); w_ly = float(wa.get(L0 - pd.DateOffset(years=1), np.nan))
    full = WASTE_YEARLY.dropna(subset=["Diversion %"])
    cb = CARBON[CARBON.period == _period(BASE_START)]
    base_t = float(cb["t CO2e"].sum())
    st = {p: meter_status(p) for p in REAL_POINTS}
    per_day = {p: float(DAILY[p].loc[L0:L1].mean()) for p in REAL_POINTS if METER_INFO[p][1] == "electricity"}
    top = max(per_day, key=per_day.get)
    best = max(OFFDAY, key=lambda q: OFFDAY[q]["cut"]) if OFFDAY else None
    W, kwh, m3, ce, cg = what_if(1, 1, 10, 1)
    _seen, _rows = set(), []
    for _, r in RECS[~RECS.priority.str.startswith("1")].iterrows():   # one per kind of action, biggest first
        if r["next step"] not in _seen:
            _seen.add(r["next step"]); _rows.append(r)
    opp = pd.DataFrame(_rows).head(3)
    S = TRUST_SUMMARY
    trust_pct = (S["tests"] - S["gl_saving"] - S["gl_increase"]) / max(S["tests"], 1) * 100
    ver = [(k, v) for k, v in VERIFIED.items() if v["verdict"] == "VERIFIED SAVING" and not k.startswith("D-")]
    gt = sum(v["counted"] for k, v in ver if "Ground-truth" in str(ACTIONS.set_index("action_id")["type"].get(k, "")))
    act = sum(v["counted"] for k, v in ver if "Ground-truth" not in str(ACTIONS.set_index("action_id")["type"].get(k, "")))
    _, dev_min, _H = room_savings()
    return {
        "campus": "Concordia University · Montréal", "month": LATEST.strftime("%B %Y"),
        "kpis": [
            {"key": "energy", "label": "Energy Consumption", "value": e_now / 1000, "unit": "MWh", "dark": True,
             "delta": (e_now - g_now) / g_now * 100 if g_now else None, "delta_label": "vs ghost twin",
             "spark": [float(em_a.get(d, np.nan)) for d in last12]},
            {"key": "water", "label": "Water Usage", "value": w_now, "unit": "m³",
             "delta": (w_now - w_ly) / w_ly * 100 if w_ly == w_ly and w_ly else None, "delta_label": "vs same month last year",
             "spark": [float(wa.get(d, np.nan)) for d in last12]},
            {"key": "waste", "label": "Waste Diverted", "value": float(full["Diversion %"].iloc[-1]) if len(full) else None, "unit": "%",
             "accent": True, "delta": float(full["Diversion %"].iloc[-1] - full["Diversion %"].iloc[0]) if len(full) > 1 else None,
             "delta_label": f"pts since {full.index[0]}" if len(full) else "",
             "spark": [float(v) for v in full["Diversion %"].tolist()]},
            {"key": "carbon", "label": "Carbon Emissions", "value": base_t, "unit": "t CO₂e",
             "delta": None, "delta_label": f"measured, {_period(BASE_START)} · gas pending unit",
             "spark": [float(v) for v in CARBON_BY_PERIOD.fillna(0).sum(axis=1).tolist()]},
        ],
        "insights": {
            "ring": trust_pct, "ring_label": "Trust score",
            "ring_text": f"{S['tests'] - S['gl_saving'] - S['gl_increase']} of {S['tests']} normal months correctly not counted",
            "top": {"name": _mname(top), "value": f"{per_day[top] / 1000:,.0f} MWh/day"},
            "best": {"name": _mname(best), "value": f"{OFFDAY[best]['cut']:.0f}% less on weekends"} if best else None,
            "potential": {"value": f"≈ {kwh / 365:,.0f} kWh/day", "text": "1 °C setpoints · 10% weekends · 1% always-on (projection)"},
            "opps": [{"title": r["next step"], "where": r["where"], "value": r["potential (not verified)"]} for _, r in opp.iterrows()],
        },
        "intel": {
            "energy": {"value": e_now / 1000, "delta": (e_now - g_now) / g_now * 100 if g_now else None,
                       "verified_mwh": act / 1000, "groundtruth_mwh": gt / 1000},
            "water": {"value": w_now, "ly": w_ly, "reliable": "Point_10" in RELIABLE,
                      "stuck_days": int((FAULTS[(FAULTS.point == "Point_10") & (FAULTS.rule == "not measuring")]).shape[0])},
            "carbon": {"base_t": base_t, "gas_lo": GAS_FT3_T, "gas_hi": GAS_DOC_T,
                       "india_t": float(cb[cb.scope.str.contains("electricity")].amount.sum()) * INDIA_GRID_KG_PER_KWH / 1000},
            "waste": {"tonnes": float(WASTE_BASE.sum()), "split": {STREAM_LABEL[k]: float(v) for k, v in WASTE_BASE.items()}},
            "room": {"people": ROOM["people"], "on": [d for d in DEVICES if ROOM["live"][d]], "dev_min": dev_min},
        },
        "status": {p: {"status": s, "delta": d, "name": _mname(p), "res": METER_INFO[p][1]} for p, (s, d) in st.items()},
        "alerts": len(ALERT_TABLE), "discoveries": len(DISCOVERY_TABLE),
        "recs_open": len(RECS) - (DECISIONS.rec_id.nunique() if len(DECISIONS) else 0),
        "trust": S, "headline": HEADLINE,
    }


def buildings():
    B = BUILDINGS.copy()
    B["code"] = B["Building"].astype(str)
    wt = WASTE.groupby("Building").tonnes.sum()
    wl = WASTE[WASTE.Waste_Type == "Landfill"].groupby("Building").tonnes.sum()
    out = []
    for _, r in B.iterrows():
        c = r.code
        meters = [p for p, v in METER_INFO.items() if (v[3] == "LOY campus" and r.Campus == "LOY") or c in v[3].split(",")]
        stats = [meter_status(p) for p in meters]
        order = {"Critical": 3, "Warning": 2, "Optimal": 1, "Unrated": 0}
        status = max([s for s, _ in stats], key=lambda s: order[s]) if stats else "No meter"
        wcode = "GN" if c == "GNA" else c
        out.append({"code": c, "name": r["Building Long Name"], "campus": r.Campus, "address": r.Address,
                    "lat": float(r.Latitude), "lon": float(r.Longitude), "status": status,
                    "meters": [{"point": p, "name": _mname(p), "res": METER_INFO[p][1]} for p in meters],
                    "waste_t": float(wt.get(wcode, 0)), "landfill_t": float(wl.get(wcode, 0))})
    return out


def meters():
    rows = []
    for p in REAL_POINTS:
        s, d = meter_status(p)
        acc = ACC.set_index("point").loc[p]
        rows.append({"point": p, "name": _mname(p), "res": METER_INFO[p][1], "unit": RESOURCE[METER_INFO[p][1]]["unit"],
                     "buildings": METER_INFO[p][3] or "not stated", "per_day": float(DAILY[p].loc[L0:L1].mean()),
                     "status": s, "delta": d, "accuracy": acc["Accuracy"], "cv": float(acc["Typical daily error %"]),
                     "model": acc["Predictions by"], "learned": acc["What the model learned"],
                     "alerts": int((ALERT_TABLE.point == p).sum())})
    return rows


def series(p):
    y, e = DAILY[p], EXPECTED[p].reindex(DAYS)
    return {"name": _mname(p), "unit": RESOURCE[METER_INFO[p][1]]["unit"], "dates": [d.strftime("%Y-%m-%d") for d in DAYS],
            "actual": y.tolist(), "ghost": e.tolist(), "reliable": p in RELIABLE,
            "baseline": [BASE_START.strftime("%Y-%m-%d"), BASE_END.strftime("%Y-%m-%d")],
            "alerts": ALERT_TABLE[ALERT_TABLE.point == p][["id", "from", "to", "what happened"]],
            "discoveries": DISCOVERY_TABLE[DISCOVERY_TABLE.point == p][["from", "to", "less than expected", "%"]],
            "events": [d.strftime("%Y-%m-%d") for d in sorted(set(ACTIONS[ACTIONS.point == p].date))]}


def hourly_profile(p):
    """Average weekday kW by hour: latest month vs the same month in the baseline year."""
    h = HOURLY[p]
    wk = ~h.index.normalize().isin(DAYS[OFF_DAY.values])
    cur = h[(h.index >= L0) & (h.index <= L1 + pd.Timedelta(hours=23)) & wk].groupby(lambda t: t.hour).mean()
    a1 = L0 - pd.DateOffset(years=1)
    base = h[(h.index >= a1) & (h.index < a1 + pd.DateOffset(months=1)) & wk].groupby(lambda t: t.hour).mean()
    return {"hours": list(range(24)), "now": [cur.get(i) for i in range(24)], "base": [base.get(i) for i in range(24)],
            "now_label": LATEST.strftime("%b %Y"), "base_label": a1.strftime("%b %Y")}


def verification(aid, window=42):
    a = ACTIONS.set_index("action_id").loc[aid]
    r = VERIFIED.get(aid)
    if not r or r.get("window") != int(window):
        r = verify(a.point, a.date, int(window), ACTIONS[ACTIONS.action_id != aid])
    out = {"action": aid, "window": int(window), "sim": aid.startswith("D-"), "title": a.title, "type": a.type, "meter": _mname(a.point), "unit": r["unit"], "verdict": r["verdict"],
           "explain": explain(r), "date": a.date.strftime("%Y-%m-%d")}
    if "saving_%" not in r:
        return out
    T0 = r["date"]
    pre = DAILY[a.point].loc[T0 - pd.Timedelta(days=56): T0 - pd.Timedelta(days=1)].dropna()
    pre = pre[TEMP.reindex(pre.index).notna()]
    gpre = expected(r["model"], pre.index)
    e, y = r["series_exp"], r["series_act"]
    half = r["high_%"] - r["saving_%"]
    out.update({k: r[k] for k in ["saving_%", "low_%", "high_%", "p", "naive_%", "expected", "actual", "counted", "counted_co2_kg", "typical_miss"]})
    out.update({"pre_dates": [d.strftime("%Y-%m-%d") for d in pre.index], "pre_actual": pre.tolist(), "pre_ghost": gpre.tolist(),
                "dates": [d.strftime("%Y-%m-%d") for d in e.index], "ghost": e.tolist(), "actual": y.reindex(e.index).tolist(),
                "band_hi": (e * (1 + half / 100)).tolist(), "band_lo": (e * (1 - half / 100)).tolist(),
                "cum": (e - y.reindex(e.index)).cumsum().tolist(), "fakes": list(r["fakes"]), "model": "gradient boosting" if r["kind"] == "ml" else "change-point"})
    return out


def register():
    rows = []
    for _, a in ACTIONS.iterrows():
        r = VERIFIED.get(a.action_id)
        rows.append({"id": a.action_id, "date": a.date.strftime("%Y-%m-%d"), "meter": _mname(a.point), "point": a.point,
                     "window": r.get("window", 42) if r else 42, "sim": a.action_id.startswith("D-"),
                     "title": a.title, "type": a.type, "verdict": r["verdict"] if r else "—",
                     "saving": r.get("saving_%") if r else None, "counted": r.get("counted") if r else None, "unit": r["unit"] if r else ""})
    return rows


def recs():
    dec = DECISIONS.drop_duplicates("rec_id", keep="last").set_index("rec_id") if len(DECISIONS) else pd.DataFrame()
    out = []
    for _, r in RECS.iterrows():
        d = dec.loc[r.id] if len(dec) and r.id in dec.index else None
        out.append({"id": r.id, "priority": r.priority[4:], "level": r.priority[0], "area": r.area, "where": r["where"],
                    "finding": r.finding, "potential": r["potential (not verified)"], "next": r["next step"],
                    "meter": isinstance(r["_point"], str),
                    "status": "open" if d is None else d["decision"], "by": "" if d is None else d["by"],
                    "reason": "" if d is None else d["reason"], "action_id": "" if d is None else d["action_id"]})
    return out


def waste_data():
    M = WASTE_MONTHLY.copy()
    return {"months": [d.strftime("%Y-%m") for d in M.index],
            "streams": {STREAM_LABEL.get(c, c): M[c].tolist() for c in M.columns},
            "colors": {STREAM_LABEL.get(k, k): v for k, v in STREAM_COLOR.items()},
            "by_building": WASTE_BY_BUILDING.reset_index().rename(columns=STREAM_LABEL),
            "yearly": WASTE_YEARLY.rename(columns=STREAM_LABEL).reset_index(),
            "cost": {STREAM_LABEL.get(k, k): float(v) for k, v in WASTE_COST_PER_T.items()},
            "no_weight": NO_WEIGHT, "landfill_base_t": LANDFILL_BASE_T, "landfill_base_co2e": LANDFILL_BASE_CO2E,
            "types": WASTE_TYPES}


def carbon_data(gas=False):
    C = carbon_by_period(gas)
    return {"periods": list(C.index), "scopes": {c: C[c].tolist() for c in C.columns}, "lines": CARBON,
            "factors": CARBON_FACTORS, "avoided": avoided_table(), "gas_lo": GAS_FT3_T, "gas_hi": GAS_DOC_T,
            "india_factor": INDIA_GRID_KG_PER_KWH, "base_period": _period(BASE_START)}


def settings():
    return {"parameters": PARAMETERS, "model_check": MODEL_CHECK.drop(columns="point"), "accuracy": ACC.drop(columns="point"),
            "faults": fault_periods(FAULTS[FAULTS.rule != "incomplete day"]),
            "incomplete_days": int((FAULTS.rule == "incomplete day").sum()), "meters": METERS.drop(columns=["point"]),
            "baseline": [BASE_START, BASE_END], "monitoring": [MON_START, DAYS[-1]],
            "sources": [{"name": "Concordia University open data (Facilities)", "url": "https://www.concordia.ca/web/open-data.html", "licence": "CC BY 4.0"},
                        {"name": "Concordia API documentation", "url": "https://github.com/opendataConcordiaU/documentation"},
                        {"name": "Open-Meteo (live weather)", "url": "https://open-meteo.com"},
                        {"name": "Esri World Imagery (aerial map tiles)", "url": "https://www.esri.com"}]}


# ---------------- room (camera) ----------------
def room_state(reason=""):
    kwh, dev_min, H = room_savings()
    return {"people": ROOM["people"], "raw": ROOM["raw"], "live": ROOM["live"], "ghost": ROOM["ghost"], "setpoint": ROOM["setpoint"],
            "reason": reason, "temp_source": ROOM["temp_source"],
            "weather": {**{k: WEATHER[k] for k in ("city", "now", "feels", "humidity", "trm", "error")},
                        "fetched": WEATHER["fetched"].strftime("%H:%M") if WEATHER["fetched"] else None, "every_min": WEATHER_REFRESH_MIN},
            "thr": ROOM["thr"] or thresholds(), "auto_thr": thresholds(), "manual": ROOM["manual"],
            "defaults": {"devices": DEVICE_DEFAULTS, "crowd": CROWD_DEFAULT, "person_w": PERSON_HEAT_W},
            "switch": "smart switch (webhook)" if DEVICE_WEBHOOK else "virtual switch — no smart plug connected",
            "kwh": kwh, "dev_min": dev_min, "india_g": kwh * INDIA_GRID_KG_PER_KWH * 1000,
            "history": {"t": [h["time"].strftime("%H:%M:%S") for h in ROOM["history"][-300:]],
                        "people": [h["people"] for h in ROOM["history"][-300:]],
                        "live_kw": [h["live_kw"] for h in ROOM["history"][-300:]], "ghost_kw": [h["ghost_kw"] for h in ROOM["history"][-300:]],
                        "live_on": [h["live_on"] for h in ROOM["history"][-300:]], "ghost_on": [h["ghost_on"] for h in ROOM["history"][-300:]]},
            "log": ROOM["log"][-40:][::-1]}


def _f(x):
    try:
        return float(x) if x not in (None, "", "null") else None
    except (TypeError, ValueError):
        return None


API = FastAPI(title="GreenLedger")


@API.get("/", response_class=HTMLResponse)
def _index():
    return HTMLResponse(WEB_HTML)


@API.get("/api/buildings")
def _buildings():
    return ok(buildings())


@API.get("/api/carbon")
def _carbon(gas: int = 0):
    return ok(carbon_data(bool(gas)))


@API.get("/api/actions")
def _actions():
    return ok(register())


@API.post("/api/actions")
def _add_action(body: dict = Body(...)):
    try:
        d = pd.Timestamp(str(body.get("date", "")).strip())
        p, title = body.get("point"), str(body.get("title", "")).strip()
        assert p in DAILY and title
    except Exception:
        return ok({"error": "Give a meter, a date (YYYY-MM-DD) and what was done."})
    aid = add_action(d, p, title, _f(body.get("claimed")))
    VERIFIED[aid] = verify(p, d, 42, ACTIONS[ACTIONS.action_id != aid])
    return ok({"id": aid, "verdict": VERIFIED[aid]["verdict"]})


@API.get("/api/verify/{aid}")
def _verify(aid: str, window: int = 42):
    return ok(verification(aid, window))


@API.get("/api/trust")
def _trust():
    T = TRUST.copy(); T["meter"] = T.point.map(_mname)
    T["month"] = T.month.dt.strftime("%b %Y")             # so a judge can audit exactly which months were tested
    return ok({"summary": TRUST_SUMMARY, "rows": T.drop(columns="point")})


@API.get("/api/recs")
def _recs():
    return ok(recs())


@API.post("/api/recs/{rid}")
def _decide(rid: str, body: dict = Body(...)):
    decision = body.get("decision")
    if decision == "rejected" and not str(body.get("reason", "")).strip():
        return ok({"error": "Give a reason for rejecting — it is kept in the log."})
    d = None
    if decision == "approved" and str(body.get("date", "")).strip():
        try:
            d = pd.Timestamp(str(body["date"]).strip())
        except Exception:
            return ok({"error": "Date must be YYYY-MM-DD"})
    aid = decide_rec(rid, decision, body.get("by", ""), body.get("reason", ""), d)
    if aid:
        VERIFIED[aid] = verify(ACTIONS.set_index("action_id").loc[aid, "point"], d, 42, ACTIONS[ACTIONS.action_id != aid])
    return ok({"action_id": aid, "verdict": VERIFIED[aid]["verdict"] if aid else None})


@API.get("/api/ewaste")
def _ewaste():
    return ok({"cols": EWASTE_COLS, "rows": (EWASTE[EWASTE_COLS] if EWASTE is not None else pd.DataFrame(columns=EWASTE_COLS)),
               "plan": EWASTE, "help": EWASTE_HELP})


@API.post("/api/ewaste")
def _ewaste_plan(body: dict = Body(...)):
    global EWASTE
    df = pd.DataFrame(body.get("rows", []), columns=EWASTE_COLS)
    df = df[df.device_id.astype(str).str.strip() != ""]
    if not len(df):
        EWASTE = None
        refresh_ewaste_carbon()
        return ok({"plan": None})
    EWASTE = ewaste_plan(df)
    EWASTE.to_csv(os.path.join(OUT_DIR, "ewaste_plan.csv"), index=False)
    refresh_ewaste_carbon()
    return ok({"plan": EWASTE})


@API.get("/api/settings")
def _settings():
    return ok(settings())


@API.get("/api/room")
def _room():
    return ok(room_state())


@API.post("/api/room/frame")
def _room_frame(body: dict = Body(...)):
    frame = None
    if body.get("image"):                                  # no image = "room is empty" demo tick
        frame = np.asarray(_PILImage.open(io.BytesIO(base64.b64decode(body["image"].split(",")[-1]))).convert("RGB"))
    kw = {}
    for d, key in (("Fan", "fan"), ("Lights", "lights"), ("AC", "ac")):    # how many × watts each → kW
        n_, w_ = _f(body.get(f"n_{key}")), _f(body.get(f"w_{key}"))
        kw[d] = (n_ if n_ is not None else DEVICE_DEFAULTS[d][0]) * (w_ if w_ is not None else DEVICE_DEFAULTS[d][1]) / 1000
    vac = DEMO_VACANCY_MIN if body.get("demo", True) else (_f(body.get("vacancy")) or VACANCY_MIN_DEFAULT)
    weather_if_stale(body.get("city"))
    manual = {"mode": body.get("mode", "auto"), "fan_on": _f(body.get("fan_on")), "ac_on": _f(body.get("ac_on")),
              "ac_set": _f(body.get("ac_set")), "crowd": _f(body.get("crowd"))}
    ann, n, raw, reason = process_frame(frame, _f(body.get("room_temp")), bool(body.get("auto", True)), vac,
                                        _f(body.get("outdoor")), _f(body.get("conf")) or CONF_DEFAULT, kw, manual,
                                        bool(body.get("still")), body.get("privacy", True) is not False)
    st = room_state(reason)
    if ann is not None:
        buf = io.BytesIO(); _PILImage.fromarray(ann).save(buf, format="JPEG", quality=70)
        st["annotated"] = "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()
    return ok(st)


@API.post("/api/room/switch")
def _room_switch(body: dict = Body(...)):
    """Hand switch: the device goes to MANUAL (GreenLedger stops switching it) until body.auto = true."""
    dev = body.get("device")
    if dev not in DEVICES:
        return ok(room_state())
    with ROOM["lock"]:
        if body.get("auto"):
            ROOM["manual"][dev] = False
            return ok(room_state(f"{dev} is back on automatic control."))
        on = bool(body.get("on"))
        ROOM["live"][dev] = on
        ROOM["manual"][dev] = True
        if on:
            ROOM["ghost"][dev] = True
        _log(datetime.datetime.now(), dev, "ON" if on else "OFF", "switched by hand (manual)", send_command(dev, on))
    return ok(room_state(f"Manual: {dev} {'ON' if on else 'OFF'} — GreenLedger will not switch it until you press “auto”."))


@API.post("/api/room/reset")
def _room_reset():
    """Reset IN PLACE under the room's own lock, so a camera frame being processed at that moment can't write to a stale room."""
    with ROOM["lock"]:
        fresh = _new_room()
        fresh["lock"] = ROOM["lock"]
        ROOM.clear()
        ROOM.update(fresh)
    return ok(room_state("New session started."))


@API.get("/api/weather")
def _weather(city: str = "Bengaluru"):
    fetch_weather(city)
    return ok(room_state("Weather updated." if WEATHER["trm"] is not None else f"Weather not reachable: {WEATHER['error']}"))


@API.get("/api/guide")
def _guide():
    return ok({**JUDGE_GUIDE, "validation": VALIDATION, "sim_check": SIM_CHECK, "trust": TRUST_SUMMARY, "team": TEAM,
               "month": LAST_MONTH.strftime("%B %Y"), "tariff": TARIFF_INR_PER_KWH})


@API.post("/api/report")
def _report():
    h, z = build_report()
    c = os.path.join(OUT_DIR, "carbon_report.csv"); CARBON.to_csv(c, index=False)
    v = os.path.join(OUT_DIR, "verification.csv"); verification_table().to_csv(v, index=False)
    r = os.path.join(OUT_DIR, "recommendations.csv"); RECS.drop(columns=["_value", "_point"]).to_csv(r, index=False)
    files = [h, z, c, v, r] + ([os.path.join(OUT_DIR, "ewaste_plan.csv")] if EWASTE is not None else [])
    return ok({"files": [os.path.basename(f) for f in files]})


@API.get("/files/{name}")
def _file(name: str):
    path = os.path.join(OUT_DIR, os.path.basename(name))
    if not os.path.exists(path):
        return Response(status_code=404)
    return FileResponse(path, filename=os.path.basename(path))


print(f"API ready · {len(API.routes)} routes · latest full month: {LATEST.strftime('%B %Y')}")


# ================= additions: local libraries, map tiles, simulated departments & e-waste =================
STATIC_DIR = os.path.join(OUT_DIR, "static")
os.makedirs(STATIC_DIR, exist_ok=True)
LIBS = {  # saved once by the notebook, then served by GreenLedger itself → charts work even if the browser blocks CDNs
    "chart.umd.js": ["https://cdn.jsdelivr.net/npm/chart.js@4.4.1/dist/chart.umd.js",
                     "https://unpkg.com/chart.js@4.4.1/dist/chart.umd.js",
                     "https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"],
    "leaflet.js": ["https://unpkg.com/leaflet@1.9.4/dist/leaflet.js", "https://cdn.jsdelivr.net/npm/leaflet@1.9.4/dist/leaflet.js",
                   "https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.js"],
    "leaflet.css": ["https://unpkg.com/leaflet@1.9.4/dist/leaflet.css", "https://cdn.jsdelivr.net/npm/leaflet@1.9.4/dist/leaflet.css",
                    "https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.css"],
    "lucide.min.js": ["https://unpkg.com/lucide@0.460.0/dist/umd/lucide.min.js",
                      "https://cdn.jsdelivr.net/npm/lucide@0.460.0/dist/umd/lucide.min.js"],
}
for _name, _urls in LIBS.items():
    _dst = os.path.join(STATIC_DIR, _name)
    if os.path.exists(_dst) and os.path.getsize(_dst) > 1000:
        continue
    for _u in _urls:
        try:
            urllib.request.urlretrieve(_u, _dst)
            if os.path.getsize(_dst) > 1000:
                break
        except Exception:
            pass
_have = {n: os.path.exists(os.path.join(STATIC_DIR, n)) and os.path.getsize(os.path.join(STATIC_DIR, n)) > 1000 for n in LIBS}
print("Chart/map libraries saved locally:" if all(_have.values()) else "Some libraries could not be saved (no internet?) — the browser will load them from the CDN instead:", _have)


@API.get("/static/{name}")
def _static(name: str):
    if name.startswith("campus"):
        return Response(CAMPUS_IMG_BYTES, media_type=CAMPUS_IMG_TYPE, headers={"Cache-Control": "max-age=3600"})
    path = os.path.join(STATIC_DIR, os.path.basename(name))
    if not os.path.exists(path) or os.path.getsize(path) < 1000:
        if name in LIBS:                       # notebook could not download it → let the browser try the CDN
            from fastapi.responses import RedirectResponse
            return RedirectResponse(LIBS[name][0])
        return Response(status_code=404)
    mt = "text/css" if name.endswith(".css") else "application/javascript"
    return FileResponse(path, media_type=mt, headers={"Cache-Control": "max-age=86400"})


_TILES = {}


@API.get("/tiles/{z}/{y}/{x}")
def _tile(z: int, y: int, x: int):
    """Aerial tiles fetched by the notebook (Esri World Imagery) — works even if the browser cannot reach Esri."""
    k = (z, y, x)
    if k not in _TILES:
        try:
            req = urllib.request.Request(f"https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                                         headers={"User-Agent": "GreenLedger/1.0"})
            _TILES[k] = urllib.request.urlopen(req, timeout=10).read()
            if len(_TILES) > 3000:
                _TILES.pop(next(iter(_TILES)))
        except Exception:
            return Response(status_code=404)
    return Response(_TILES[k], media_type="image/jpeg", headers={"Cache-Control": "max-age=86400"})


def _dept_ghost_series(c):
    """Department total without its verified actions (ghost) vs actual."""
    a = SIM_E[c].copy(); g = a.copy()
    for r in DEPT_RESULTS.values():
        if r["dept"] == c and r["verdict"] == "VERIFIED SAVING":
            g.loc[r["ghost"].index] = r["ghost"].values
    return a, g


def _counted_daily(r):
    """Daily kWh saved by one verified action, scaled to the LOW end of its range (same rule as the headline total)."""
    m = (r["ghost"] - r["actual"]).dropna()
    return m * (r["low_%"] / r["saving_%"]) if r["verdict"] == "VERIFIED SAVING" and r["saving_%"] > 0 else m * 0


def money_saved():
    """₹ saved per month, per department (verified, low end) and the running total."""
    ver = [r for r in DEPT_RESULTS.values() if r["verdict"] == "VERIFIED SAVING"]
    if not ver:
        return {"months": [], "by_dept": {}, "cum_k": [], "names": {}}
    daily = pd.DataFrame({r["id"]: _counted_daily(r) for r in ver}).fillna(0)
    dept_of = {r["id"]: r["dept"] for r in ver}
    by = daily.T.groupby(dept_of).sum().T.resample("MS").sum() * TARIFF_INR_PER_KWH
    tot = by.sum(axis=1)
    return {"months": [d.strftime("%b %y") for d in by.index], "by_dept": {c: by[c].round(0).tolist() for c in by.columns},
            "cum_k": (tot.cumsum() / 1000).round(1).tolist(), "names": DEPTS.name.to_dict(), "total_inr": float(tot.sum())}


def dept_savings():
    """Department-wise verified energy and money saved (low end), next to the true simulated saving."""
    out = []
    for c in DEPTS.index:
        rs = [r for r in DEPT_RESULTS.values() if r["dept"] == c]
        kwh = sum(r["counted_kwh"] for r in rs)
        out.append({"code": c, "name": DEPTS.loc[c, "name"], "mwh": kwh / 1000, "inr": kwh * TARIFF_INR_PER_KWH,
                    "true_mwh": sum(r["true_kwh"] for r in rs) / 1000, "actions": len(rs),
                    "verified": sum(r["verdict"] == "VERIFIED SAVING" for r in rs)})
    return sorted(out, key=lambda x: -x["mwh"])


def dashboard():
    tot_a = sum(_dept_ghost_series(c)[0] for c in DEPTS.index)
    tot_g = sum(_dept_ghost_series(c)[1] for c in DEPTS.index)
    ma, mg = tot_a.resample("MS").sum(), tot_g.resample("MS").sum()
    L = SIM_DAYS[-365:]
    ver = [r for r in DEPT_RESULTS.values() if r["verdict"] == "VERIFIED SAVING"]
    w = SIM_WASTE.loc[L].sum()
    div = float((w * DEPTS.diversion).sum() / w.sum() * 100)
    T = DEPT_TABLE.sort_values("kWh per m²", ascending=False)
    tips = sorted([(DEPT_TABLE.loc[c, "kWh per m²"], c, DEPT_TABLE.loc[c, "Top suggestion"]) for c in DEPTS.index], reverse=True)[:3]
    kwh_room, dev_min, _ = room_savings()
    return {
        "hero": {"mwh": DEPT_SAVED_KWH / 1000, "inr": DEPT_SAVED_KWH * TARIFF_INR_PER_KWH, "co2_t": DEPT_SAVED_KWH * INDIA_KG_PER_KWH / 1000,
                 "verified": len(ver), "actions": len(DEPT_RESULTS), "true_mwh": DEPT_TRUE_KWH / 1000, "tariff": TARIFF_INR_PER_KWH,
                 "room_wh": kwh_room * 1000, "month": LAST_MONTH.strftime("%B %Y"),
                 "kwh_m": float(DEPT_TABLE["Saved kWh last month"].sum()), "inr_m": float(DEPT_TABLE["Saved ₹ last month"].sum()),
                 "co2_m_kg": float(DEPT_TABLE["Saved kWh last month"].sum() * INDIA_KG_PER_KWH)},
        "kpis": {"energy_mwh": float(SIM_E.loc[L].sum().sum() / 1000), "water_kl_day": float(SIM_W.loc[L].sum().sum() / 365 / 1000),
                 "diversion": div, "ewaste_inr": EW_SUMMARY["recoverable_value"],
                 "energy_spark": [float(v) for v in ma.iloc[-12:] / 1000],
                 "water_spark": [float(v) for v in SIM_W.sum(axis=1).resample("MS").sum().iloc[-12:] / 1000],
                 "waste_spark": [float(v) for v in SIM_WASTE.sum(axis=1).resample("MS").sum().iloc[-12:]],
                 "co2_t": float(SIM_E.loc[L].sum().sum() * INDIA_KG_PER_KWH / 1000)},
        "months": [d.strftime("%b %y") for d in ma.index[-24:]], "actual": (ma.iloc[-24:] / 1000).tolist(), "ghost": (mg.iloc[-24:] / 1000).tolist(),
        "rank": [{"code": c, "name": DEPTS.loc[c, "name"], "kwh_m2": float(T.loc[c, "kWh per m²"]), "eff": T.loc[c, "Efficiency"]} for c in T.index],
        "tips": [{"code": c, "name": DEPTS.loc[c, "name"], "tip": t} for _, c, t in tips],
        "trust": TRUST_SUMMARY, "room": {"people": ROOM["people"], "on": [d for d in DEVICES if ROOM["live"][d]], "dev_min": dev_min},
        "money": money_saved(), "dept_savings": dept_savings(),
    }


def departments():
    T = DEPT_TABLE.rename_axis("Code").reset_index()
    m = SIM_E.resample("MS").sum() / 1000
    return {"table": T, "months": [d.strftime("%b %y") for d in m.index], "monthly": {c: m[c].tolist() for c in m.columns},
            "names": DEPTS.name.to_dict(),
            "actions": [{"id": r["id"], "dept": r["dept"], "title": r["title"], "date": r["date"].strftime("%d %b %Y"), "verdict": r["verdict"],
                         "est": r["saving_%"], "lo": r["low_%"], "hi": r["high_%"], "p": r["p"], "true": r["true_%"],
                         "counted_mwh": r["counted_kwh"] / 1000, "true_mwh": r["true_kwh"] / 1000} for r in DEPT_RESULTS.values()],
            "savings": dept_savings(), "tariff": TARIFF_INR_PER_KWH,
            "pins": {b: {"x": x, "y": y, "depts": d,
                         "status": (max([DEPT_TABLE.loc[c, "Efficiency"] for c in d], key=["Efficient", "Average", "High use"].index) if d else "No data"),
                         "mwh": float(sum(DEPT_TABLE.loc[c, "Energy MWh/yr"] for c in d)),
                         "m": ({"kwh_m2": float(DEPT_TABLE.loc[d[0], "kWh per m²"]), "saved_mwh": float(DEPT_TABLE.loc[d[0], "Saved MWh (verified)"]),
                                "saved_inr": float(DEPT_TABLE.loc[d[0], "Saved ₹ last month"]),
                                "water": float(DEPT_TABLE.loc[d[0], "Water L/person/day"]), "diversion": float(DEPT_TABLE.loc[d[0], "Diversion %"]),
                                "cost_month": float(DEPT_TABLE.loc[d[0], "Cost ₹/month"]), "co2_t": float(DEPT_TABLE.loc[d[0], "CO₂ t/yr"]),
                                "name": DEPTS.loc[d[0], "name"]} if d else None)} for b, (x, y, d) in CAMPUS_PINS.items()}}


def department(c):
    a, g = _dept_ghost_series(c)
    L = SIM_DAYS[-364:]                                  # last 52 whole weeks, shown as weekly totals (easier to read)
    _full = a.loc[L].resample("W-SUN").count() == 7                    # whole weeks only (no half-week dips at the ends)
    aw, gw = a.loc[L].resample("W-SUN").sum()[_full], g.loc[L].resample("W-SUN").sum()[_full]
    row = DEPT_TABLE.loc[c].to_dict()
    ew = EW_SIM[EW_SIM.dept == c]
    ms = money_saved()
    return {"code": c, "name": DEPTS.loc[c, "name"], "building": DEPTS.loc[c, "building"], "row": row,
            "money": {"months": ms["months"], "inr": ms["by_dept"].get(c, [0] * len(ms["months"]))},
            "dates": [d.strftime("%d %b %y") for d in aw.index], "actual": (aw / 1000).tolist(), "ghost": (gw / 1000).tolist(),
            "temp": SIM_TEMP.loc[L].resample("W-SUN").mean()[_full].tolist(),
            "actions": [x for x in departments()["actions"] if x["dept"] == c],
            "ewaste": {"devices": len(ew), "sell": int((ew.action == "Sell second-hand now").sum()),
                       "value": float(ew.loc[ew.action != "Keep in use", "value_now"].sum())}}


def dept_action(aid):
    r = DEPT_RESULTS[aid]
    pre, pg = r["pre"], r["pre_ghost"]
    # chart shows the first 180 days AFTER the action (r["ghost"] starts on the action date), whatever that date is;
    # totals and the money curve below use the whole period to date
    g, a = r["ghost"].iloc[:180], r["actual"].iloc[:180]
    half = r["high_%"] - r["saving_%"]
    return {"id": aid, "title": r["title"], "dept": DEPTS.loc[r["dept"], "name"], "verdict": r["verdict"], "date": r["date"].strftime("%Y-%m-%d"),
            "saving_%": r["saving_%"], "low_%": r["low_%"], "high_%": r["high_%"], "p": r["p"], "true_%": r["true_%"],
            "counted_mwh": r["counted_kwh"] / 1000, "measured_mwh": r["measured_kwh"] / 1000, "true_mwh": r["true_kwh"] / 1000,
            "pre_dates": [d.strftime("%Y-%m-%d") for d in pre.index], "pre_actual": pre.tolist(), "pre_ghost": pg.tolist(),
            "dates": [d.strftime("%Y-%m-%d") for d in g.index], "ghost": g.tolist(), "actual": a.tolist(),
            "band_hi": (g * (1 + half / 100)).tolist(), "band_lo": (g * (1 - half / 100)).tolist(),
            "cum": ((g - a).cumsum() / 1000).tolist(), "fakes": r["fakes"].tolist(),
            "money": _money_curve(r)}


def _money_curve(r):
    """Running ₹ saved by one action: what was measured (ghost − live) and what GreenLedger counts (low end)."""
    m = (r["ghost"] - r["actual"]).dropna()
    meas = (m.cumsum() * TARIFF_INR_PER_KWH).resample("W-SUN").last()
    cnt = (_counted_daily(r).cumsum() * TARIFF_INR_PER_KWH).resample("W-SUN").last()
    return {"dates": [d.strftime("%d %b %y") for d in meas.index], "measured": meas.round(0).tolist(), "counted": cnt.round(0).tolist(),
            "tariff": TARIFF_INR_PER_KWH}


def ewaste_sim(dept=""):
    E = EW_SIM if not dept else EW_SIM[EW_SIM.dept == dept]
    _order = {"Sell second-hand now": 0, "Harvest parts": 1, "Repair & resell": 2, "Repair & keep using": 3, "Harvest parts + recycle": 4}
    act = (E[E.action != "Keep in use"].assign(_o=lambda x: x.action.map(_order))
           .sort_values(["_o", "value_now"], ascending=[True, False]).groupby("action", sort=False).head(5))   # a few of each, sell first
    up = E[(E.action == "Keep in use") & (E["when"] <= TODAY + pd.DateOffset(months=12))]
    upm = up.groupby(up["when"].dt.to_period("M")).agg(devices=("device_id", "count"), value=("value_then", "sum"))
    months = pd.period_range(TODAY, periods=12, freq="M")
    return {"summary": EW_SUMMARY if not dept else {
                "devices": len(E), "sell_now": int((E.action == "Sell second-hand now").sum()),
                "sell_value": float(E.loc[E.action == "Sell second-hand now", "value_now"].sum()),
                "repair": int(E.action.str.startswith("Repair").sum()), "repair_value": float(E.loc[E.action.str.startswith("Repair"), "value_now"].sum()),
                "harvest": int(E.action.str.startswith("Harvest").sum()), "parts_value": float(E.loc[E.action.str.startswith("Harvest"), "value_now"].sum()),
                "recycler_kg": float(E.to_recycler_kg.sum()), "keep": int((E.action == "Keep in use").sum()),
                "lost_per_year": float(E.lost_per_year_waiting.sum()),
                "recoverable_value": float(E.loc[E.action != "Keep in use", "value_now"].sum())},
            "upcoming": {"months": [m.strftime("%b %y") for m in months], "devices": [int(upm["devices"].get(m, 0)) for m in months],
                         "value": [float(upm["value"].get(m, 0)) for m in months]},
            "actions": E.action.value_counts().to_dict(),
            "by_dept": EW_BY_DEPT.reset_index().to_dict(orient="records"),
            "parts": (E[E.reusable_parts != ""].assign(part=E.reusable_parts.str.split(", ")).explode("part").groupby("part").size()
                      .sort_values(ascending=False).head(10).to_dict()),
            "top": act.head(25)[["device_id", "type", "dept", "age_years", "condition", "action", "value_now", "reusable_parts"]],
            "depts": DEPTS.name.to_dict()}


@API.get("/api/dashboard")
def _dash():
    return ok(dashboard())


@API.get("/api/departments")
def _depts():
    return ok(departments())


@API.get("/api/department/{c}")
def _dept(c: str):
    return ok(department(c))


@API.get("/api/dept_action/{aid}")
def _dact(aid: str):
    return ok(dept_action(aid))


@API.get("/api/ewaste_sim")
def _ews(dept: str = ""):
    return ok(ewaste_sim(dept))


print("Simulated department + e-waste routes ready.")


## Cell 16 · Open GreenLedger
Prints a **Colab link** (works in your browser, camera included) and a **public link** through a free Cloudflare tunnel (for judges' phones). Keep this cell's runtime alive during the demo.

**Demo-day checklist**
1. Run the whole notebook on a fresh Colab runtime the day before and time it.
2. Open the **public link on a phone using mobile data** (venue Wi-Fi can block Cloudflare tunnels). If it is blocked, present from the Colab link on your laptop.
3. **Screen-record the Live Room working** (camera → fan/AC on → empty room → all off), 30 seconds, as a backup.
4. Keep this cell running during the demo.

In [ ]:
# ---- Open GreenLedger ----
# Starts the web app and gives you links:
#   • Colab link — opens in your browser (camera works; only you can open it)
#   • Public link — via a free Cloudflare tunnel, for judges' phones (needs internet at the venue)
import threading, subprocess, re, time, socket
import uvicorn

PORT = int(os.environ.get("GL_PORT", "7860"))


def _free(port):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) != 0


if _free(PORT):
    _srv = uvicorn.Server(uvicorn.Config(API, host="0.0.0.0", port=PORT, log_level="warning"))
    threading.Thread(target=_srv.run, daemon=True).start()
    time.sleep(2)
print(f"GreenLedger is running on port {PORT}.")

if os.environ.get("GL_NO_LAUNCH") != "1":
    try:                                                        # 1) Colab's own secure link
        from google.colab.output import eval_js
        print("▶ Open (Colab link):", eval_js(f"google.colab.kernel.proxyPort({PORT})"))
    except Exception:
        print(f"▶ Open: http://localhost:{PORT}")
    try:                                                        # 2) public link for other devices
        CF = os.path.join(OUT_DIR, "cloudflared")
        if not os.path.exists(CF):
            urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
            os.chmod(CF, 0o755)
        _cf = subprocess.Popen([CF, "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        _t0 = time.time(); _url = None
        while time.time() - _t0 < 40 and not _url:
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", _cf.stdout.readline())
            _url = m.group(0) if m else None
        print("▶ Public link (phones, judges):", _url or "not available — use the Colab link")
    except Exception as e:
        print("Public link not available:", e)
